In [2]:

import requests

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct"

try:
    # Check that Ollama is reachable
    health = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )
    health.raise_for_status()

    available_models = [
        model["name"]
        for model in health.json().get("models", [])
    ]

    print("Ollama connection successful!")
    print("Available models:", available_models)

    # Generate a test answer
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": MODEL_NAME,
            "prompt": "What is the capital of France? Answer in one sentence.",
            "stream": False,
            "options": {
                "temperature": 0,
                "seed": 42,
                "num_predict": 60
            }
        },
        timeout=180
    )

    response.raise_for_status()
    result = response.json()

    print("\nGenerated answer:")
    print(result["response"])

except requests.exceptions.RequestException as e:
    print("Ollama request failed:", e)


Ollama connection successful!
Available models: ['qwen3:4b-instruct']

Generated answer:
The capital of France is Paris.


In [4]:

from pathlib import Path
import pandas as pd

# Find the project root by searching the current directory and its parents.
current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Project root not found from {current}"
    )

print("Project root:", PROJECT_ROOT)

generated_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_generated.csv"
)

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

if not generated_path.exists():
    raise FileNotFoundError(
        f"Generated pilot CSV not found: {generated_path}"
    )

print("Generated pilot file found!")
print("Review file exists:", review_path.exists())


Project root: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination
Generated pilot file found!
Review file exists: False


In [ ]:

from pathlib import Path

print("Current notebook directory:")
print(Path.cwd())

filename = "unified_train_all_features.parquet"

search_roots = [
    Path.cwd(),
    Path.home() / "Desktop/Projects/Predictive-model/Predictive_hallucination",
    Path.home() / "Desktop/Projects",
]

matches = []

for root in search_roots:
    if root.exists():
        # First check the expected location directly
        direct_path = root / "data" / "processed" / filename

        if direct_path.is_file() and direct_path not in matches:
            matches.append(direct_path)

        # Search recursively only if needed
        if not matches:
            for path in root.rglob(filename):
                if path.is_file() and path not in matches:
                    matches.append(path)

    if matches:
        break

if matches:
    DATA_PATH = matches[0]
    PROJECT_ROOT = DATA_PATH.parents[2]

    print("\nFile found!")
    print("Data path:", DATA_PATH)
    print("Project root:", PROJECT_ROOT)
else:
    print("\nFile not found in the expected locations.")
    print("Check whether the project folder is on your Desktop.")


Current notebook directory:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/notebooks

File found!
Data path: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/data/processed/unified_train_all_features.parquet
Project root: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination


In [ ]:

import pandas as pd

df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDataset distribution:")
print(df["source_dataset"].value_counts())


Dataset loaded successfully!
Shape: (27326, 29)
Columns: ['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category', 'split', 'has_context', 'question_length', 'answer_length', 'context_length', 'entity_count', 'person_count', 'organization_count', 'location_count', 'date_count', 'question_answer_similarity', 'question_context_similarity', 'answer_context_similarity', 'retrieval_corpus', 'retrieval_top1_similarity', 'retrieval_top3_mean_similarity', 'retrieval_coverage', 'answer_length_normalized', 'semantic_uncertainty', 'retrieval_uncertainty', 'feature_disagreement']

Dataset distribution:
source_dataset
halueval      14000
fever          9160
truthfulqa     4166
Name: count, dtype: int64


In [ ]:
output_path = PROJECT_ROOT / "experiments" / "pregeneration_pilot_questions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:

import pandas as pd

# Keep only questions and their supplied contexts from HaluEval.
# Do not include the original answers or labels in the pilot.
halueval = df.loc[
    df["source_dataset"].eq("halueval"),
    ["question", "context"]
].copy()

halueval = halueval.dropna(subset=["question"])
halueval["question"] = halueval["question"].astype(str).str.strip()
halueval["context"] = halueval["context"].fillna("").astype(str).str.strip()

# Normalize question text to identify paired-answer duplicates.
halueval["_question_key"] = (
    halueval["question"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Rows available:", len(halueval))
print("Unique questions:", halueval["_question_key"].nunique())

# Verify that repeated questions have consistent contexts.
context_variants = halueval.groupby(
    "_question_key"
)["context"].nunique()

inconsistent = context_variants[context_variants > 1]

print("Questions with inconsistent contexts:", len(inconsistent))

if len(inconsistent):
    print("\nExamples requiring inspection:")
    print(inconsistent.head(10))
    raise ValueError(
        "Some repeated questions have different contexts. "
        "Inspect these before creating the pilot."
    )

# One row per unique question.
questions = (
    halueval.drop_duplicates(subset="_question_key")
    [["question", "context"]]
    .reset_index(drop=True)
)

# Select a reproducible pilot of up to 50 questions.
pilot = questions.sample(
    n=min(50, len(questions)),
    random_state=42
).reset_index(drop=True)

pilot.insert(0, "pilot_id", range(1, len(pilot) + 1))

# Initial features available before answer generation.
pilot["question_char_count"] = pilot["question"].str.len()
pilot["context_char_count"] = pilot["context"].str.len()
pilot["context_sentence_count"] = pilot["context"].str.count(
    r"[.!?]+"
)

# Save under the actual project root.
output_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_pilot_questions.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
pilot.to_csv(output_path, index=False)

print("\nPilot dataset created successfully!")
print("Rows:", len(pilot))
print("Empty contexts:", int(pilot["context"].str.strip().eq("").sum()))
print("Saved to:", output_path)

display(pilot.head(5))


Rows available: 14000
Unique questions: 7000
Questions with inconsistent contexts: 0

Pilot dataset created successfully!
Rows: 50
Empty contexts: 0
Saved to: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_questions.csv


,pilot_id,question,context,question_char_count,context_char_count,context_sentence_count
0,1,Billings Logan International Airport and Palm ...,Billings Logan International Airport (IATA: BI...,104,346,2
1,2,Which US naval facility and airport is located...,The region also oversees installation support ...,109,444,4
2,3,What former United States Army soldier and rel...,The U.S. government also sent Twitter a subpoe...,93,497,4
3,4,When was the company that publishes the journa...,Magnetic Resonance Imaging is a peer-reviewed ...,88,275,2
4,5,"South Korean actress Kim Go-eun, appeared in 2...","Kim Go-eun (born July 2, 1991) is a South Kore...",97,372,3


In [ ]:

import requests
import pandas as pd
from pathlib import Path

MODEL_NAME = "qwen3:4b-instruct"
OLLAMA_URL = "http://localhost:11434"

output_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_pilot_generated.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

# Resume an existing run, or start from the pilot questions.
if output_path.exists():
    generated = pd.read_csv(output_path)
    print("Resuming existing generation file.")
else:
    generated = pilot.copy()
    generated["generated_answer"] = ""
    generated["generation_status"] = "pending"
    generated["generation_error"] = ""

# Make sure the required columns exist.
for col in [
    "generated_answer",
    "generation_status",
    "generation_error"
]:
    if col not in generated.columns:
        generated[col] = ""

# Test the Ollama connection before beginning.
health = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10)
health.raise_for_status()

available_models = [
    item["name"] for item in health.json().get("models", [])
]

if MODEL_NAME not in available_models:
    raise RuntimeError(
        f"{MODEL_NAME} was not found. Available models: {available_models}"
    )

for idx, row in generated.iterrows():

    # Skip answers already generated successfully.
    if row["generation_status"] == "success":
        continue

    question = str(row["question"])
    context = str(row.get("context", "") or "")

    prompt = f"""
You are answering a factual question for a research experiment.

Question:
{question}

Available context:
{context}

Instructions:
- Give a concise, direct answer to the question.
- Use the supplied context as your primary source when relevant.
- If the context is incomplete, give your best factual answer.
- Do not invent names, dates, numbers, or supporting details.
- Do not provide a long explanation or reveal your reasoning.
- Return only the answer.
"""

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "seed": 42,
                    "num_predict": 160
                }
            },
            timeout=180
        )

        response.raise_for_status()
        answer = response.json().get("response", "").strip()

        if not answer:
            raise ValueError("The model returned an empty answer.")

        generated.at[idx, "generated_answer"] = answer
        generated.at[idx, "generation_status"] = "success"
        generated.at[idx, "generation_error"] = ""

        print(f"[{idx + 1}/{len(generated)}] Generated successfully")

    except Exception as exc:
        generated.at[idx, "generation_status"] = "failed"
        generated.at[idx, "generation_error"] = str(exc)

        print(f"[{idx + 1}/{len(generated)}] Failed: {exc}")

    # Save after every question to support recovery.
    generated.to_csv(output_path, index=False)

print("\nGeneration run complete.")
print("Successful:", (generated["generation_status"] == "success").sum())
print("Failed:", (generated["generation_status"] == "failed").sum())
print("Saved to:", output_path)

display(
    generated[
        ["pilot_id", "question", "generated_answer", "generation_status"]
    ].head(10)
)


[1/50] Generated successfully
[2/50] Generated successfully
[3/50] Generated successfully
[4/50] Generated successfully
[5/50] Generated successfully
[6/50] Generated successfully
[7/50] Generated successfully
[8/50] Generated successfully
[9/50] Generated successfully
[10/50] Generated successfully
[11/50] Generated successfully
[12/50] Generated successfully
[13/50] Generated successfully
[14/50] Generated successfully
[15/50] Generated successfully
[16/50] Generated successfully
[17/50] Generated successfully
[18/50] Generated successfully
[19/50] Generated successfully
[20/50] Generated successfully
[21/50] Generated successfully
[22/50] Generated successfully
[23/50] Generated successfully
[24/50] Generated successfully
[25/50] Generated successfully
[26/50] Generated successfully
[27/50] Generated successfully
[28/50] Generated successfully
[29/50] Generated successfully
[30/50] Generated successfully
[31/50] Generated successfully
[32/50] Generated successfully
[33/50] Generated

,pilot_id,question,generated_answer,generation_status
0,1,Billings Logan International Airport and Palm ...,Yes,success
1,2,Which US naval facility and airport is located...,"The Pacific Missile Range Facility, Barking Sands",success
2,3,What former United States Army soldier and rel...,Chelsea Elizabeth Manning,success
3,4,When was the company that publishes the journa...,1880,success
4,5,"South Korean actress Kim Go-eun, appeared in 2...",Park Heung-sik,success
5,6,The Canterbury cap is descended from an ancien...,Ancient Greece and surrounding regions,success
6,7,"What American actress portrays Barbara ""Barb"" ...",Jeanne Tripplehorn,success
7,8,"Which band was formed first, Monaco or Rise Ag...",Monaco was formed first.,success
8,9,How many seats are there at the stadium hostin...,"45,000",success
9,10,X.com was an online bank that later became the...,PayPal,success


In [5]:

import pandas as pd

generated_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_pilot_generated.csv"
)

review_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(generated_path)

# Separate correctness from whether the supplied context supports an answer.
review["answer_verdict"] = ""
review["context_support"] = ""
review["supporting_evidence"] = ""
review["review_notes"] = ""

preliminary = {
    1: (
        "needs_external_verification",
        "not_established_by_context",
        "Context names Montana and California but does not explicitly state that both airports are in the US."
    ),
    2: (
        "correct_from_context",
        "supported",
        "Context explicitly identifies the Pacific Missile Range Facility, Barking Sands as a US naval facility and airport."
    ),
    3: (
        "correct_from_context",
        "supported",
        "Context identifies Chelsea Manning as a former US Army soldier and a WikiLeaks-related person."
    ),
    4: (
        "needs_external_verification",
        "ambiguous",
        "The passage gives 1880 as the establishment year of a publishing company, but the antecedent is grammatically ambiguous."
    ),
    5: (
        "correct_from_context",
        "supported",
        "Context explicitly says Park Heung-sik co-wrote and directed the film."
    ),
    6: (
        "correct_from_context",
        "supported",
        "Context explicitly says the pileus was worn in Ancient Greece and surrounding regions."
    ),
    7: (
        "correct_from_context",
        "supported",
        "Context directly identifies Jeanne Tripplehorn as the actress portraying Barb."
    ),
    8: (
        "correct_from_context",
        "supported",
        "Context says Monaco formed in 1995 and Rise Against in 1999."
    ),
    9: (
        "correct_from_context",
        "supported",
        "Context explicitly gives the stadium capacity as 45,000 seats."
    ),
    10: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The question asks for a feature. The context gives online money transfers; PayPal is the company, not the feature."
    )
}

for pilot_id, (verdict, support, notes) in preliminary.items():
    mask = review["pilot_id"].eq(pilot_id)

    if mask.any():
        review.loc[mask, "answer_verdict"] = verdict
        review.loc[mask, "context_support"] = support
        review.loc[mask, "review_notes"] = notes

review.to_csv(review_path, index=False)

print("Updated review sheet saved to:")
print(review_path)

print("\nPreliminary judgments recorded:", len(preliminary))
print(
    "Still awaiting review:",
    int(review["answer_verdict"].eq("").sum())
)

display(
    review[
        [
            "pilot_id",
            "generated_answer",
            "answer_verdict",
            "context_support",
            "review_notes"
        ]
    ].head(10)
)

Updated review sheet saved to:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_review.csv

Preliminary judgments recorded: 10
Still awaiting review: 40


,pilot_id,generated_answer,answer_verdict,context_support,review_notes
0,1,Yes,needs_external_verification,not_established_by_context,Context names Montana and California but does ...
1,2,"The Pacific Missile Range Facility, Barking Sands",correct_from_context,supported,Context explicitly identifies the Pacific Miss...
2,3,Chelsea Elizabeth Manning,correct_from_context,supported,Context identifies Chelsea Manning as a former...
3,4,1880,needs_external_verification,ambiguous,The passage gives 1880 as the establishment ye...
4,5,Park Heung-sik,correct_from_context,supported,Context explicitly says Park Heung-sik co-wrot...
5,6,Ancient Greece and surrounding regions,correct_from_context,supported,Context explicitly says the pileus was worn in...
6,7,Jeanne Tripplehorn,correct_from_context,supported,Context directly identifies Jeanne Tripplehorn...
7,8,Monaco was formed first.,correct_from_context,supported,Context says Monaco formed in 1995 and Rise Ag...
8,9,"45,000",correct_from_context,supported,Context explicitly gives the stadium capacity ...
9,10,PayPal,incorrect_or_nonresponsive,not_supported_as_an_answer,The question asks for a feature. The context g...


In [6]:

import pandas as pd

from pathlib import Path

# Restore paths if the notebook kernel was restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)
review = review.sort_values("pilot_id").reset_index(drop=True)

# Display pilot IDs 11–20.
batch = review[review["pilot_id"].between(11, 20)]

for _, row in batch.iterrows():
    print("\n" + "=" * 80)
    print(f"PILOT ID: {row['pilot_id']}")
    print(f"\nQUESTION:\n{row['question']}")
    print(f"\nCONTEXT:\n{row['context']}")
    print(f"\nGENERATED ANSWER:\n{row['generated_answer']}")
    print(f"\nGENERATION STATUS: {row.get('generation_status', 'unknown')}")



PILOT ID: 11

QUESTION:
Which Lightweight division mixed martial artist founded the American Combat Association?

CONTEXT:
The American Combat Association is a small mixed martial arts company founded by Olympic wrestler, world Abu Dhabi champion and UFC fighter Kamal Shalorus and professional mixed martial arts fighter, Broadcaster and American professional wrestler Matthew "The Granimal" Granahan.Kamal Shalorus is an Iranian professional mixed martial artist currently competing in the Lightweight division of ONE FC.

GENERATED ANSWER:
Kamal Shalorus

GENERATION STATUS: success

PILOT ID: 12

QUESTION:
What moving starring Garvin Cross was set in the Bronx area of New York City, but filmed i ad around Vancouver, Canada?

CONTEXT:
Cross is best known for his role as Angelo in the 1995 martial arts film "Rumble in the Bronx", which starred Jackie Chan. The film is set in the Bronx area of New York City but was filmed in and around Vancouver, Canada.

GENERATED ANSWER:
Rumble in the Bro

In [11]:

# Fix annotation columns that pandas incorrectly inferred as float64

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes"
]

for col in annotation_columns:
    if col not in review.columns:
        review[col] = ""
    else:
        review[col] = review[col].fillna("").astype(object)

print("Annotation column types fixed!")

print(review[annotation_columns].dtypes)


Annotation column types fixed!
answer_verdict         object
context_support        object
supporting_evidence    object
review_notes           object
dtype: object


In [ ]:

from pathlib import Path
import pandas as pd

# Restore paths if the notebook kernel was restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

# Ensure annotation columns exist.
for col in [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes"
]:
    if col not in review.columns:
        review[col] = ""

assessments = {
    12: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context identifies Rumble in the Bronx as the film "
            "set in the Bronx and filmed around Vancouver."
        ),
        "review_notes": "Answer directly supported by context."
    },
    13: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context describes the Plott Hound as large and the "
            "Entlebucher Mountain Dog as medium-sized."
        ),
        "review_notes": "Comparison supported by the supplied descriptions."
    },
    14: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context identifies Jack Reacher as based on Lee Child's "
            "novel One Shot and links Rosamund Pike to Jane Bennet."
        ),
        "review_notes": "Answer directly supported by context."
    },
    15: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context identifies the University of Nebraska–Lincoln "
            "as the institution represented by the Cornhuskers."
        ),
        "review_notes": "Answer supported by context."
    },
    16: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context describes Cowboy Junkies as an alternative "
            "country/blues/folk rock band."
        ),
        "review_notes": "Answer directly supported by context."
    },
    17: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context identifies Chris Hadfield as a former ISS "
            "Commander Colonel and gives his birth date as 29 August 1959."
        ),
        "review_notes": "Answer directly supported by context."
    },
    18: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context states that Ambush was based on the serial story "
            "Ambush by Luke Short."
        ),
        "review_notes": "Answer directly supported by context."
    },
    19: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "supporting_evidence": (
            "The context describes Call Me Lucky as a documentary about "
            "satirist Barry Crimmins."
        ),
        "review_notes": "Answer directly supported by context."
    },
    20: {
        "answer_verdict": "needs_external_verification",
        "context_support": "not_established_by_context",
        "supporting_evidence": (
            "The context mentions Jhené Aiko and B2K music videos, "
            "but does not establish the Big Sean mixtape detail."
        ),
        "review_notes": (
            "Partially supported; verify the complete answer before "
            "assigning a definitive verdict."
        )
    }
}



TypeError: Invalid value 'The context identifies Rumble in the Bronx as the film set in the Bronx and filmed around Vancouver.' for dtype 'float64'

In [12]:
for pilot_id, values in assessments.items():
    mask = review["pilot_id"].eq(pilot_id)

    if not mask.any():
        raise ValueError(f"Pilot ID {pilot_id} is missing from the review CSV.")

    for column, value in values.items():
        review.loc[mask, column] = value

review.to_csv(review_path, index=False)

print("Saved assessments for pilot IDs 12–20.")
print("Review file:", review_path)

display(
    review.loc[
        review["pilot_id"].between(11, 20),
        [
            "pilot_id",
            "generated_answer",
            "answer_verdict",
            "context_support"
        ]
    ]
)


Saved assessments for pilot IDs 12–20.
Review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_review.csv


,pilot_id,generated_answer,answer_verdict,context_support
10,11,Kamal Shalorus,,
11,12,Rumble in the Bronx,correct_from_context,supported
12,13,Plott Hound is bigger than Entlebucher Mountai...,correct_from_context,supported
13,14,"The novel is ""One Shot"" by Lee Child.",correct_from_context,supported
14,15,University of Nebraska–Lincoln,correct_from_context,supported
15,16,alternative country/blues/folk rock,correct_from_context,supported
16,17,Chris Hadfield,correct_from_context,supported
17,18,Luke Short,correct_from_context,supported
18,19,Call Me Lucky,correct_from_context,supported
19,20,Jhené Aiko,needs_external_verification,not_established_by_context


In [13]:
display(
    review.loc[
        review["pilot_id"].between(11, 20),
        [
            "pilot_id",
            "answer_verdict",
            "context_support"
        ]
    ]
)

,pilot_id,answer_verdict,context_support
10,11,,
11,12,correct_from_context,supported
12,13,correct_from_context,supported
13,14,correct_from_context,supported
14,15,correct_from_context,supported
15,16,correct_from_context,supported
16,17,correct_from_context,supported
17,18,correct_from_context,supported
18,19,correct_from_context,supported
19,20,needs_external_verification,not_established_by_context


In [14]:

# Restore the project path if necessary
from pathlib import Path
import pandas as pd

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

# Ensure annotation columns accept text
for col in [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes"
]:
    if col not in review.columns:
        review[col] = ""
    else:
        review[col] = review[col].fillna("").astype(object)

# Save the preliminary assessment for ID 11
mask = review["pilot_id"].eq(11)

review.loc[mask, "answer_verdict"] = "correct_from_context"
review.loc[mask, "context_support"] = "supported"
review.loc[mask, "supporting_evidence"] = (
    "The context identifies Kamal Shalorus as a founder of the "
    "American Combat Association and as competing in the Lightweight division."
)
review.loc[mask, "review_notes"] = (
    "Answer directly supported by the supplied context."
)

review.to_csv(review_path, index=False)

display(
    review.loc[
        review["pilot_id"].between(11, 20),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


,pilot_id,answer_verdict,context_support
10,11,correct_from_context,supported
11,12,correct_from_context,supported
12,13,correct_from_context,supported
13,14,correct_from_context,supported
14,15,correct_from_context,supported
15,16,correct_from_context,supported
16,17,correct_from_context,supported
17,18,correct_from_context,supported
18,19,correct_from_context,supported
19,20,needs_external_verification,not_established_by_context


In [15]:

batch = review.loc[review["pilot_id"].between(21, 30)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:300])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:250])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 21
QUESTION: Which English writer argued that women outhg to have an education in her book from 1792?
CONTEXT: A Vindication of the Rights of Woman: with Strictures on Political and Moral Subjects (1792), written by the 18th-century British proto-feminist Mary Wollstonecraft, is one of the earliest works of feminist philosophy. She argues that women ought to have an education commensurate with their position in society, claiming that women are essential to the nation because they educate its children and because they could be "companions" to their husbands, rather than mere wives.Mary Wollstonecraft ( ; 
GENERATED ANSWER: Mary Wollstonecraft
STATUS: success

PILOT ID: 22
QUESTION: What American real estate developer owns the Encore hotel and casino in Las Vegas?
CONTEXT: Encore Las Vegas (also called Encore at Wynn Las Vegas; often just called Encore) is a luxury resort, casino and hotel located on the Las Vegas Strip in Paradise, Nevada. The resort is connected to its siste

In [16]:

from pathlib import Path
import pandas as pd

# Find the project root after a possible kernel restart.
current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

# Ensure annotation fields can store text.
for col in [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes"
]:
    if col not in review.columns:
        review[col] = ""
    else:
        review[col] = review[col].fillna("").astype(object)

assessments = {
    21: (
        "correct_from_context",
        "supported",
        "The context identifies Mary Wollstonecraft as the author and "
        "states that she argued women ought to receive an education."
    ),
    22: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The context names Wynn Resorts as the owner and Steve Wynn as "
        "the developer heading the company; the generated answer gives "
        "the company name rather than the requested person."
    ),
    23: (
        "correct_from_context",
        "supported",
        "The context identifies David Koechner as a star of No Clue "
        "and names his role on The Office."
    ),
    24: (
        "correct_from_context",
        "supported",
        "The context identifies Sarah J. Maas as the author of Throne of Glass."
    ),
    25: (
        "correct_from_context",
        "supported",
        "The context states that the Otter Tail River is 192 miles long."
    ),
    26: (
        "correct_from_context",
        "supported",
        "The context identifies both people as musicians or singers."
    ),
    27: (
        "correct_from_context",
        "supported",
        "The context identifies the New England town as the local-government "
        "unit described in the question."
    ),
    28: (
        "correct_from_context",
        "supported",
        "The context identifies Sogakope as the capital of South Tongu district."
    ),
    29: (
        "correct_from_context",
        "supported",
        "The context explicitly states that Samsung Scholarship was "
        "established in 2002."
    ),
    30: (
        "needs_external_verification",
        "ambiguous",
        "The context appears to combine the 1950 film-unit employee John "
        "O'Shea with an Irish footballer born in 1981. The conflict requires "
        "review rather than an automatic hallucination label."
    )
}

for pilot_id, (verdict, support, notes) in assessments.items():
    mask = review["pilot_id"].eq(pilot_id)

    if not mask.any():
        raise ValueError(f"Pilot ID {pilot_id} not found.")

    review.loc[mask, "answer_verdict"] = verdict
    review.loc[mask, "context_support"] = support
    review.loc[mask, "supporting_evidence"] = notes
    review.loc[mask, "review_notes"] = "Preliminary context-based assessment."

review.to_csv(review_path, index=False)

print("Saved assessments for pilot IDs 21–30.")
print("Review file:", review_path)

display(
    review.loc[
        review["pilot_id"].between(21, 30),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for pilot IDs 21–30.
Review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_review.csv


,pilot_id,answer_verdict,context_support
20,21,correct_from_context,supported
21,22,incorrect_or_nonresponsive,not_supported_as_an_answer
22,23,correct_from_context,supported
23,24,correct_from_context,supported
24,25,correct_from_context,supported
25,26,correct_from_context,supported
26,27,correct_from_context,supported
27,28,correct_from_context,supported
28,29,correct_from_context,supported
29,30,needs_external_verification,ambiguous


In [17]:

batch = review.loc[review["pilot_id"].between(31, 40)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:300])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 31
QUESTION: How large is the civil parish and townland in which Stephen Brown was born?
CONTEXT: Stephen James Meridith Brown (born 24 September 1881 in Holywood, County Down, Ireland, died 8 May 1962, in Kilcrony, Wicklow, Ireland) was an Irish Catholic Jesuit priest, writer, bibliographer and librarian. It is a civil parish and townland of 755 acres lying on the shore of Belfast Lough, between Belfast and Bangor.
GENERATED ANSWER: 755 acres
STATUS: success

PILOT ID: 32
QUESTION: The Office of the Federal Register is an office of the United States government within which independent agency of the United States government charged with preserving and documenting government and historical records?
CONTEXT: The Office of the Federal Register is an office of the United States government within the National Archives and Records Administration.The National Archives and Records Administration (NARA) is an independent agency of the United States government charged with preserving 

In [18]:

from pathlib import Path
import pandas as pd

# Locate the project root
current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

# Ensure annotation columns can store text.
annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes"
]

for col in annotation_columns:
    if col not in review.columns:
        review[col] = ""
    else:
        review[col] = review[col].fillna("").astype(object)

assessments = {
    31: (
        "correct_from_context",
        "supported",
        "The context describes the relevant civil parish and townland as 755 acres."
    ),
    32: (
        "correct_from_context",
        "supported",
        "The context identifies the National Archives and Records Administration as the agency."
    ),
    33: (
        "needs_external_verification",
        "not_established_by_context",
        "The context reports an Academy Award won by Neil Jordan but does not establish "
        "the comparison with Nadine Labaki."
    ),
    34: (
        "correct_from_context",
        "supported",
        "The context identifies Ariel as a playable character in minigames."
    ),
    35: (
        "correct_from_context",
        "supported",
        "The context explicitly gives the 2010 population as 9,212."
    ),
    36: (
        "correct_from_context",
        "supported",
        "The context states that Dom Capers was the inaugural head coach of two different NFL expansion teams."
    ),
    37: (
        "correct_from_context",
        "supported",
        "The context identifies Only You as the 1992 comedy starring Helen Hunt and says Betty Thomas directed it."
    ),
    38: (
        "correct_from_context",
        "supported",
        "The context explicitly identifies Doug TenNapel as the creator of Catscratch."
    ),
    39: (
        "correct_from_context",
        "supported",
        "The context states that Mortal Kombat was originally developed in 1992."
    ),
    40: (
        "correct_from_context",
        "supported",
        "The context identifies Stirling Castle as the location of the baptism entertainment."
    )
}

for pilot_id, (verdict, support, notes) in assessments.items():
    mask = review["pilot_id"].eq(pilot_id)

    if not mask.any():
        raise ValueError(f"Pilot ID {pilot_id} not found.")

    review.loc[mask, "answer_verdict"] = verdict
    review.loc[mask, "context_support"] = support
    review.loc[mask, "supporting_evidence"] = notes
    review.loc[mask, "review_notes"] = "Preliminary context-based assessment."

review.to_csv(review_path, index=False)

print("Saved assessments for pilot IDs 31–40.")
print("Review file:", review_path)

display(
    review.loc[
        review["pilot_id"].between(31, 40),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for pilot IDs 31–40.
Review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_review.csv


,pilot_id,answer_verdict,context_support
30,31,correct_from_context,supported
31,32,correct_from_context,supported
32,33,needs_external_verification,not_established_by_context
33,34,correct_from_context,supported
34,35,correct_from_context,supported
35,36,correct_from_context,supported
36,37,correct_from_context,supported
37,38,correct_from_context,supported
38,39,correct_from_context,supported
39,40,correct_from_context,supported


In [19]:

batch = review.loc[review["pilot_id"].between(41, 50)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:300])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 41
QUESTION: Which opera had more acts: Madama Butterfly (original) or Rigoletto?
CONTEXT: Madama Butterfly (] ; "Madam Butterfly") is an opera in three acts (originally two) by Giacomo Puccini, with an Italian libretto by Luigi Illica and Giuseppe Giacosa.Rigoletto (] ) is an opera in three acts by Giuseppe Verdi.
GENERATED ANSWER: Both Madama Butterfly (original) and Rigoletto have three acts.
STATUS: success

PILOT ID: 42
QUESTION: Which of the two European Space Agency astronauts, Jean-François Clervoy and Thomas Reiter, has participated in missions with NASA?
CONTEXT: Jean-François André Clervoy (born 19 November 1958) is a French engineer and a CNES and ESA astronaut. He is a veteran of three NASA Space Shuttle missions.Thomas Arthur Reiter (born 23 May 1958 in Frankfurt, West Germany) is a retired European astronaut and is a Brigadier General in the German Air Force currently working as ESA Interagency Coordinator and Advisor to the Director General at the European Sp

In [20]:

from pathlib import Path
import pandas as pd

# Locate project root
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

# Ensure annotation columns can store text
annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes"
]

for col in annotation_columns:
    if col not in review.columns:
        review[col] = ""
    else:
        review[col] = review[col].fillna("").astype(object)

assessments = {
    41: (
        "incorrect_or_nonresponsive",
        "contradicted",
        "The context says Madama Butterfly was originally in two acts, "
        "while Rigoletto has three acts. The generated answer says both "
        "have three acts."
    ),
    42: (
        "correct_from_context",
        "supported",
        "The context explicitly says Jean-François Clervoy is a veteran "
        "of three NASA Space Shuttle missions."
    ),
    43: (
        "incorrect_or_nonresponsive",
        "contradicted",
        "The supplied context says Baker Mayfield won the starting job "
        "over incumbent Trevor Knight, who it identifies as an Atlanta "
        "Falcons quarterback. The generated response contradicts the "
        "supplied context."
    ),
    44: (
        "correct_from_context",
        "supported",
        "The context states that the actress was second runner-up in "
        "Miss Hong Kong 1990."
    ),
    45: (
        "correct_from_context",
        "supported",
        "The context states that the British heavyweight title was held "
        "twice between 2000 and 2010."
    ),
    46: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The question asks for the full name of the Roman general and "
        "statesman. The context gives Lucius Cornelius Sulla Felix; "
        "the generated answer gives Sulla and Otho."
    ),
    47: (
        "correct_from_context",
        "supported",
        "The context gives Amber Joy Mariano's birth date as August 11, 1978."
    ),
    48: (
        "correct_from_context",
        "supported",
        "The context identifies both the Hedley Mascot Mine and the Ruth "
        "Mine as gold mines."
    ),
    49: (
        "correct_from_context",
        "supported",
        "The context says Zoo is based on a novel by James Patterson and "
        "Michael Ledwidge."
    ),
    50: (
        "correct_from_context",
        "supported",
        "The context identifies Big Data as the electronic music project "
        "whose song Dangerous was certified gold by the RIAA in May 2015."
    )
}

for pilot_id, (verdict, support, notes) in assessments.items():
    mask = review["pilot_id"].eq(pilot_id)

    if not mask.any():
        raise ValueError(f"Pilot ID {pilot_id} not found.")

    review.loc[mask, "answer_verdict"] = verdict
    review.loc[mask, "context_support"] = support
    review.loc[mask, "supporting_evidence"] = notes
    review.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )

review.to_csv(review_path, index=False)

print("Saved preliminary assessments for IDs 41–50.")
print("File:", review_path)

display(
    review.loc[
        review["pilot_id"].between(41, 50),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved preliminary assessments for IDs 41–50.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_review.csv


,pilot_id,answer_verdict,context_support
40,41,incorrect_or_nonresponsive,contradicted
41,42,correct_from_context,supported
42,43,incorrect_or_nonresponsive,contradicted
43,44,correct_from_context,supported
44,45,correct_from_context,supported
45,46,incorrect_or_nonresponsive,not_supported_as_an_answer
46,47,correct_from_context,supported
47,48,correct_from_context,supported
48,49,correct_from_context,supported
49,50,correct_from_context,supported


In [21]:

review = pd.read_csv(review_path)

print("Total pilot rows:", len(review))
print("Unique pilot IDs:", review["pilot_id"].nunique())

print("\nAnswer verdict distribution:")
print(
    review["answer_verdict"]
    .fillna("")
    .replace("", "not_reviewed")
    .value_counts()
    .to_string()
)

print("\nContext support distribution:")
print(
    review["context_support"]
    .fillna("")
    .replace("", "not_reviewed")
    .value_counts()
    .to_string()
)

print("\nRows without a verdict:")
print(
    review.loc[
        review["answer_verdict"].fillna("").eq(""),
        "pilot_id"
    ].tolist()
)

print("\nRows requiring external verification:")
print(
    review.loc[
        review["answer_verdict"].eq("needs_external_verification"),
        ["pilot_id", "question", "generated_answer"]
    ].to_string(index=False)
)


Total pilot rows: 50
Unique pilot IDs: 50

Answer verdict distribution:
answer_verdict
correct_from_context           40
needs_external_verification     5
incorrect_or_nonresponsive      5

Context support distribution:
context_support
supported                     40
not_established_by_context     3
not_supported_as_an_answer     3
ambiguous                      2
contradicted                   2

Rows without a verdict:
[]

Rows requiring external verification:
 pilot_id                                                                                                                               question                                                                                                                                                                                                                                                     generated_answer
        1                               Billings Logan International Airport and Palm Springs International Airport, both are 

In [22]:

# Resolve the five uncertain cases after external verification.
# Keep context support separate from factual correctness.

from pathlib import Path
import pandas as pd

current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

for col in [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "verification_sources",
    "verification_notes"
]:
    if col not in review.columns:
        review[col] = ""
    else:
        review[col] = review[col].fillna("").astype(object)

updates = {
    1: {
        "verdict": "verified_correct_externally",
        "sources": (
            "https://www.flybillings.com/ ; "
            "https://flypsp.com/starthere/"
        ),
        "notes": (
            "Official airport websites confirm locations in Montana "
            "and California, respectively. Answer Yes is correct; "
            "the supplied context does not explicitly state the country."
        )
    },
    4: {
        "verdict": "verified_correct_externally",
        "sources": (
            "https://www.elsevier.com/en-gb/connect/discover-the-elsevier-experience ; "
            "https://www.uk.elsevierhealth.com/magnetic-resonance-imaging-0730725x.html"
        ),
        "notes": (
            "Elsevier states that the company was founded in 1880, and "
            "the journal page identifies Elsevier as publisher. The "
            "original context remains grammatically ambiguous."
        )
    },
    20: {
        "verdict": "verified_correct_externally",
        "sources": (
            "https://www.hot97.com/news/b2k-reunites-at-jhene-aiko-show/ ; "
            "https://www.xxlmag.com/jhene-aiko-talks-big-sean-j-cole-features-and-her-five-dream-collaborations/"
        ),
        "notes": (
            "External sources support Jhené Aiko's appearances in B2K "
            "music videos and her feature on Big Sean's Detroit mixtape. "
            "The supplied context establishes only the B2K part."
        )
    },
    30: {
        "verdict": "source_inconsistent",
        "sources": (
            "https://teara.govt.nz/en/biographies/6o2/oshea-john-dempsey ; "
            "https://www.nzonscreen.com/film-in-nz/"
        ),
        "notes": (
            "The source conflates the Irish footballer with filmmaker "
            "John O'Shea, who joined Pacific Films in 1950. Preserve "
            "this as a source/question inconsistency, not an automatic "
            "hallucination label."
        )
    },
    33: {
        "verdict": "verified_correct_externally",
        "sources": (
            "https://www.oscars.org/oscars/ceremonies/1993 ; "
            "https://www.oscars.org/oscars/ceremonies/2019"
        ),
        "notes": (
            "Official Academy records support Neil Jordan's Original "
            "Screenplay win for The Crying Game; Capernaum was nominated "
            "in 2019. Neil Jordan appears to be the correct answer, "
            "though the supplied context does not establish the full comparison."
        )
    }
}

for pilot_id, item in updates.items():
    mask = review["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot_id={pilot_id}"
        )

    review.loc[mask, "answer_verdict"] = item["verdict"]
    review.loc[mask, "verification_sources"] = item["sources"]
    review.loc[mask, "verification_notes"] = item["notes"]

# Preserve context_support and existing evidence notes.
review.to_csv(review_path, index=False)

print("Updated review file:", review_path)

print("\nFinal answer-verdict distribution:")
print(review["answer_verdict"].value_counts(dropna=False))

print("\nContext-support distribution (unchanged):")
print(review["context_support"].value_counts(dropna=False))

print("\nResolved rows:")
display(
    review.loc[
        review["pilot_id"].isin([1, 4, 20, 30, 33]),
        [
            "pilot_id",
            "answer_verdict",
            "context_support",
            "verification_notes"
        ]
    ]
)


Updated review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_review.csv

Final answer-verdict distribution:
answer_verdict
correct_from_context           40
incorrect_or_nonresponsive      5
verified_correct_externally     4
source_inconsistent             1
Name: count, dtype: int64

Context-support distribution (unchanged):
context_support
supported                     40
not_established_by_context     3
not_supported_as_an_answer     3
ambiguous                      2
contradicted                   2
Name: count, dtype: int64

Resolved rows:


,pilot_id,answer_verdict,context_support,verification_notes
0,1,verified_correct_externally,not_established_by_context,Official airport websites confirm locations in...
3,4,verified_correct_externally,ambiguous,Elsevier states that the company was founded i...
19,20,verified_correct_externally,not_established_by_context,External sources support Jhené Aiko's appearan...
29,30,source_inconsistent,ambiguous,The source conflates the Irish footballer with...
32,33,verified_correct_externally,not_established_by_context,Official Academy records support Neil Jordan's...


In [23]:

from pathlib import Path
import pandas as pd

# Restore project paths if needed.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

review = pd.read_csv(review_path)

# Record the type of answer error separately.
review["answer_error_type"] = "none"

# Correct answers remain the negative class.
correct_mask = review["answer_verdict"].isin([
    "correct_from_context",
    "verified_correct_externally"
])

failure_mask = review["answer_verdict"].eq(
    "incorrect_or_nonresponsive"
)

source_issue_mask = review["answer_verdict"].eq(
    "source_inconsistent"
)

# Split the five answer failures into interpretable categories.
review.loc[
    review["pilot_id"].isin([41, 43]),
    "answer_error_type"
] = "contradicted_by_supplied_context"

review.loc[
    review["pilot_id"].isin([10, 22, 46]),
    "answer_error_type"
] = "nonresponsive_or_answer_mismatch"

review.loc[
    source_issue_mask,
    "answer_error_type"
] = "source_inconsistency"

# Build a provisional binary answer-failure label.
# 0 = answer judged correct
# 1 = incorrect or non-responsive answer
# Missing = source inconsistent, unreviewed, or otherwise unresolved
review["answer_failure_label"] = float("nan")

review.loc[correct_mask, "answer_failure_label"] = 0
review.loc[failure_mask, "answer_failure_label"] = 1

# Keep the original review file intact except for the added audit fields.
review.to_csv(review_path, index=False)

# Save a separate labeled pilot for subsequent analysis.
output_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_labeled.csv"
)

review.to_csv(output_path, index=False)

print("Labeled pilot saved:", output_path)

print("\nAnswer-failure label distribution:")
print(
    review["answer_failure_label"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nAnswer-error categories:")
print(review["answer_error_type"].value_counts())

print("\nRows excluded from binary labeling:")
print(
    review.loc[
        review["answer_failure_label"].isna(),
        ["pilot_id", "answer_verdict", "answer_error_type"]
    ].to_string(index=False)
)


Labeled pilot saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_pilot_labeled.csv

Answer-failure label distribution:
answer_failure_label
0.0    44
1.0     5
NaN     1
Name: count, dtype: int64

Answer-error categories:
answer_error_type
none                                44
nonresponsive_or_answer_mismatch     3
contradicted_by_supplied_context     2
source_inconsistency                 1
Name: count, dtype: int64

Rows excluded from binary labeling:
 pilot_id      answer_verdict    answer_error_type
       30 source_inconsistent source_inconsistency


In [24]:

from pathlib import Path
import pandas as pd

# Find the project root, even if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

data_path = (
    PROJECT_ROOT / "data" / "processed"
    / "unified_train_all_features.parquet"
)

pilot_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_pilot_generated.csv"
)

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_pilot_review.csv"
)

output_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_questions.csv"
)

# Load the existing source data and previously generated pilot.
df = pd.read_parquet(data_path)
pilot = pd.read_csv(pilot_path)
review = pd.read_csv(review_path)

def normalize_question(series):
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Use HaluEval questions and contexts only.
source = df.loc[
    df["source_dataset"].eq("halueval"),
    ["question", "context"]
].copy()

source = source.dropna(subset=["question"])
source["question"] = source["question"].astype(str).str.strip()
source["context"] = source["context"].fillna("").astype(str).str.strip()
source["_question_key"] = normalize_question(source["question"])

# Confirm duplicate question records have consistent contexts.
context_counts = source.groupby("_question_key")["context"].nunique()

if (context_counts > 1).any():
    raise ValueError(
        "Some duplicate questions have different contexts. "
        "Inspect those questions before sampling."
    )

# Keep exactly one row per question.
questions = (
    source.drop_duplicates("_question_key")
    .reset_index(drop=True)
)

# Exclude every question from the original 50-question pilot.
pilot_keys = set(normalize_question(pilot["question"]))

remaining = questions.loc[
    ~questions["_question_key"].isin(pilot_keys)
].copy()

print("Unique source questions:", len(questions))
print("Questions in original pilot:", len(pilot_keys))
print("New questions available:", len(remaining))

# Select 250 additional questions, or fewer if unavailable.
batch_size = min(250, len(remaining))

batch2 = remaining.sample(
    n=batch_size,
    random_state=2026
).copy()

batch2 = batch2[["question", "context"]].reset_index(drop=True)
batch2.insert(0, "pilot_id", range(51, 51 + len(batch2)))

# Features available before answer generation.
batch2["question_char_count"] = batch2["question"].str.len()
batch2["context_char_count"] = batch2["context"].str.len()
batch2["context_sentence_count"] = batch2["context"].str.count(
    r"[.!?]+"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
batch2.to_csv(output_path, index=False)

print("\nNew batch created!")
print("Rows:", len(batch2))
print("Empty contexts:", int(batch2["context"].str.strip().eq("").sum()))
print("Saved to:", output_path)

display(batch2.head(5))


Unique source questions: 7000
Questions in original pilot: 50
New questions available: 6950

New batch created!
Rows: 250
Empty contexts: 0
Saved to: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_questions.csv


,pilot_id,question,context,question_char_count,context_char_count,context_sentence_count
0,51,The 1932 Maccabiah Games were in commemoration...,The games were in commemoration of the 1800th ...,123,311,2
1,52,Are Castle of Frankenstein and Modern Drummer ...,"Castle of Frankenstein was an American horror,...",61,443,3
2,53,"homas Edward Lawrence, was a British archaeolo...",The breadth and variety of his activities and ...,294,320,4
3,54,Which ride at the theme park located between B...,Giant Barn Swing is a Screamin' Swing style am...,169,362,3
4,55,"What organization did the author of ""Physics f...",Physics for Future Presidents: The Science Beh...,82,413,3


In [25]:

from pathlib import Path
import pandas as pd
import requests

# Restore project root if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

input_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_questions.csv"
)

output_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_generated.csv"
)

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct"

# Load the 250 questions.
batch2 = pd.read_csv(input_path)

# Resume an earlier generation run if one exists.
if output_path.exists():
    generated = pd.read_csv(output_path)

    if generated["pilot_id"].tolist() != batch2["pilot_id"].tolist():
        raise ValueError(
            "The existing output does not match batch 2. "
            "Check the file before continuing."
        )

    print("Resuming existing batch 2 generation.")
else:
    generated = batch2.copy()
    generated["generated_answer"] = ""
    generated["generation_status"] = "pending"
    generated["generation_error"] = ""

# Ensure status fields are available.
for col in [
    "generated_answer",
    "generation_status",
    "generation_error"
]:
    if col not in generated.columns:
        generated[col] = ""

# Confirm Ollama is running and the requested model is available.
health = requests.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10
)
health.raise_for_status()

available_models = [
    model["name"]
    for model in health.json().get("models", [])
]

if MODEL_NAME not in available_models:
    raise RuntimeError(
        f"Model {MODEL_NAME} is unavailable. "
        f"Available models: {available_models}"
    )

print("Ollama connected.")
print("Model:", MODEL_NAME)
print("Questions:", len(generated))

for i, idx in enumerate(generated.index, start=1):

    # Skip answers already generated successfully.
    status = generated.at[idx, "generation_status"]

    if isinstance(status, str) and status == "success":
        continue

    question = str(generated.at[idx, "question"])
    context_value = generated.at[idx, "context"]
    context = "" if pd.isna(context_value) else str(context_value)

    # Keep the generation instructions consistent with batch 1.
    prompt = f"""
You are answering a factual question for a research experiment.

Question:
{question}

Available context:
{context}

Instructions:
- Give a concise, direct answer to the question.
- Use the supplied context as your primary source when relevant.
- If the context is incomplete, give your best factual answer.
- Do not invent names, dates, numbers, or supporting details.
- Do not provide a long explanation or reveal your reasoning.
- Return only the answer.
"""

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "seed": 42,
                    "num_predict": 160
                }
            },
            timeout=180
        )

        response.raise_for_status()
        answer = response.json().get("response", "").strip()

        if not answer:
            raise ValueError("The model returned an empty answer.")

        generated.at[idx, "generated_answer"] = answer
        generated.at[idx, "generation_status"] = "success"
        generated.at[idx, "generation_error"] = ""

    except Exception as exc:
        generated.at[idx, "generation_status"] = "failed"
        generated.at[idx, "generation_error"] = str(exc)

    # Save immediately so progress isn't lost.
    generated.to_csv(output_path, index=False)

    if i % 10 == 0 or i == len(generated):
        succeeded = (
            generated["generation_status"] == "success"
        ).sum()
        failed = (
            generated["generation_status"] == "failed"
        ).sum()

        print(
            f"Progress: {i}/{len(generated)} | "
            f"Successful: {succeeded} | Failed: {failed}"
        )

print("\nGeneration run finished.")
print(
    "Successful:",
    (generated["generation_status"] == "success").sum()
)
print(
    "Failed:",
    (generated["generation_status"] == "failed").sum()
)
print("Saved to:", output_path)

display(
    generated[
        [
            "pilot_id",
            "question",
            "generated_answer",
            "generation_status"
        ]
    ].head(10)
)


Ollama connected.
Model: qwen3:4b-instruct
Questions: 250
Progress: 10/250 | Successful: 10 | Failed: 0
Progress: 20/250 | Successful: 20 | Failed: 0
Progress: 30/250 | Successful: 30 | Failed: 0
Progress: 40/250 | Successful: 40 | Failed: 0
Progress: 50/250 | Successful: 50 | Failed: 0
Progress: 60/250 | Successful: 60 | Failed: 0
Progress: 70/250 | Successful: 70 | Failed: 0
Progress: 80/250 | Successful: 80 | Failed: 0
Progress: 90/250 | Successful: 90 | Failed: 0
Progress: 100/250 | Successful: 100 | Failed: 0
Progress: 110/250 | Successful: 110 | Failed: 0
Progress: 120/250 | Successful: 120 | Failed: 0
Progress: 130/250 | Successful: 130 | Failed: 0
Progress: 140/250 | Successful: 140 | Failed: 0
Progress: 150/250 | Successful: 150 | Failed: 0
Progress: 160/250 | Successful: 160 | Failed: 0
Progress: 170/250 | Successful: 170 | Failed: 0
Progress: 180/250 | Successful: 180 | Failed: 0
Progress: 190/250 | Successful: 190 | Failed: 0
Progress: 200/250 | Successful: 200 | Failed: 0


,pilot_id,question,generated_answer,generation_status
0,51,The 1932 Maccabiah Games were in commemoration...,The Second Revolt of Judea,success
1,52,Are Castle of Frankenstein and Modern Drummer ...,"Yes, both Castle of Frankenstein and Modern Dr...",success
2,53,"homas Edward Lawrence, was a British archaeolo...",1962,success
3,54,Which ride at the theme park located between B...,Giant Barn Swing,success
4,55,"What organization did the author of ""Physics f...",Berkeley Earth,success
5,56,The Enemy Expatriation Act (HR 3166 and S. 169...,New Zealand,success
6,57,A Norwegian composer who received numerous awa...,Grotten,success
7,58,Who is the current world record holder in high...,Javier Sotomayor Sanabria is the current world...,success
8,59,"Who wrote the 2007 British television film ""Co...",Mark Haddon,success
9,60,Three Nationwide Plaza is a 408 ft(124m) postm...,Ohio,success


In [26]:

from pathlib import Path
import pandas as pd

# Locate the project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

generated_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_generated.csv"
)

batch2_generated = pd.read_csv(generated_path)

batch = batch2_generated[
    batch2_generated["pilot_id"].between(51, 60)
]

for _, row in batch.iterrows():
    print("\n" + "=" * 80)
    print(f"PILOT ID: {row['pilot_id']}")
    print(f"\nQUESTION:\n{row['question']}")
    print(f"\nCONTEXT:\n{row['context']}")
    print(f"\nGENERATED ANSWER:\n{row['generated_answer']}")
    print(f"\nSTATUS: {row['generation_status']}")



PILOT ID: 51

QUESTION:
The 1932 Maccabiah Games were in commemoration of the 1800th anniversary of a revolt that historians also refer to as what?

CONTEXT:
The games were in commemoration of the 1800th anniversary of the Bar Kokhba revolt, a major rebellion by the Jews of Judaea Province against the Roman Empire. Some historians also refer to it as the Second Revolt of Judea, not counting the Kitos War (115–117 CE), which had only marginally been fought in Judea.

GENERATED ANSWER:
The Second Revolt of Judea

STATUS: success

PILOT ID: 52

QUESTION:
Are Castle of Frankenstein and Modern Drummer both magazines?

CONTEXT:
Castle of Frankenstein was an American horror, science fiction and fantasy film magazine, published between 1962 and 1975 by Calvin Thomas Beck's Gothic Castle Publishing Company, distributed by Kable News.Modern Drummer is a monthly publication targeting the interests of drummers and percussionists. The magazine features interviews, equipment reviews, and columns of

In [27]:

from pathlib import Path
import pandas as pd

# Locate the project root.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

experiments = PROJECT_ROOT / "experiments"

generated_path = experiments / "pregeneration_batch2_generated.csv"
review_path = experiments / "pregeneration_batch2_review.csv"

# Load existing annotations if this batch has already been reviewed.
if review_path.exists():
    review2 = pd.read_csv(review_path)
    print("Loaded existing batch 2 review file.")
else:
    review2 = pd.read_csv(generated_path)

    for col in [
        "answer_verdict",
        "context_support",
        "supporting_evidence",
        "review_notes",
        "answer_error_type"
    ]:
        review2[col] = ""

    print("Created a new batch 2 review file.")

# Ensure annotation columns can store strings.
annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    51: (
        "correct_from_context",
        "supported",
        "The context identifies the Bar Kokhba revolt as also being called "
        "the Second Revolt of Judea.",
        "none"
    ),
    52: (
        "correct_from_context",
        "supported",
        "The context identifies both Castle of Frankenstein and Modern "
        "Drummer as magazines.",
        "none"
    ),
    53: (
        "correct_from_context",
        "supported",
        "The context identifies Lawrence of Arabia as a 1962 film.",
        "none"
    ),
    54: (
        "correct_from_context",
        "supported",
        "The context states that Giant Barn Swing opened on April 14, 2007.",
        "none"
    ),
    55: (
        "correct_from_context",
        "supported",
        "The context explicitly says Richard A. Muller and his daughter "
        "founded Berkeley Earth in 2010.",
        "none"
    ),
    56: (
        "correct_from_context",
        "supported",
        "The context identifies New Zealand as Scott Brown's ambassadorial post.",
        "none"
    ),
    57: (
        "correct_from_context",
        "supported",
        "The context says Arne Nordheim lived in the honorary residence Grotten.",
        "none"
    ),
    58: (
        "correct_from_context",
        "supported",
        "The context identifies Javier Sotomayor as the high jumper and "
        "world record holder, and describes him as Lorenzo's uncle.",
        "none"
    ),
    59: (
        "correct_from_context",
        "supported",
        "The context identifies Mark Haddon as the writer of Coming Down "
        "the Mountain and the 2003 novel.",
        "none"
    ),
    60: (
        "incorrect_or_nonresponsive",
        "supported",
        "The context gives the state as Ohio (OH). The generated answer "
        "gives the full state name instead of the requested acronym.",
        "format_or_answer_mismatch"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot_id={pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved batch 2 review file:", review_path)
print("\nAssessments for IDs 51–60:")

display(
    review2.loc[
        review2["pilot_id"].between(51, 60),
        [
            "pilot_id",
            "answer_verdict",
            "context_support",
            "answer_error_type"
        ]
    ]
)


Created a new batch 2 review file.
Saved batch 2 review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv

Assessments for IDs 51–60:


,pilot_id,answer_verdict,context_support,answer_error_type
0,51,correct_from_context,supported,none
1,52,correct_from_context,supported,none
2,53,correct_from_context,supported,none
3,54,correct_from_context,supported,none
4,55,correct_from_context,supported,none
5,56,correct_from_context,supported,none
6,57,correct_from_context,supported,none
7,58,correct_from_context,supported,none
8,59,correct_from_context,supported,none
9,60,incorrect_or_nonresponsive,supported,format_or_answer_mismatch


In [28]:

batch = review2.loc[review2["pilot_id"].between(61, 70)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 61
QUESTION: Who wrote and directed the film in which Adam Beach played Slipknot?
CONTEXT: He starred in the Canadian 2012-2014 series "Arctic Air", and played Slipknot in the 2016 film "Suicide Squad". The film is written and directed by David Ayer and stars an ensemble cast featuring Will Smith, Jared Leto, Margot Robbie, Joel Kinnaman, Viola Davis, Jai Courtney, Jay Hernandez, Adewale Akinnuoye-Agbaje, Ike Barinholtz, Scott Eastwood, and Cara Delevingne.
GENERATED ANSWER: David Ayer
STATUS: success

PILOT ID: 62
QUESTION: David Luchins served as an aide to a Vice President who was passed away in which year ?
CONTEXT: Luchins served as an aide to then-Vice President Hubert Humphrey and for 20 years on the Senate staff of New York Senator Daniel Patrick Moynihan.Hubert Horatio Humphrey Jr. (May 27, 1911January 13, 1978) was an American politician who served as the 38th Vice President of the United States from 1965 to 1969.
GENERATED ANSWER: 1978
STATUS: success

PILOT ID: 6

In [29]:

from pathlib import Path
import pandas as pd

# Locate the project root.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

experiments = PROJECT_ROOT / "experiments"
generated_path = experiments / "pregeneration_batch2_generated.csv"
review_path = experiments / "pregeneration_batch2_review.csv"

if review_path.exists():
    review2 = pd.read_csv(review_path)
else:
    review2 = pd.read_csv(generated_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    61: "The context explicitly identifies David Ayer as the writer and director of Suicide Squad.",
    62: "The context gives Hubert Humphrey's death date as January 13, 1978.",
    63: "The context gives James Earl Chaney's birth date as May 30, 1943.",
    64: "Both schools are described as selective schools; Carre's is explicitly a selective secondary school.",
    65: "The context says Kelvin Sampson was head coach at Montana Tech from 1981 to 1985.",
    66: "The context places the Hagen Site near Glendive and gives the population as 8,966.",
    67: "The context says Manuel Perez worked on Fritz the Cat and identifies it as a comic strip created by Robert Crumb.",
    68: "The context states that the first Dresden Files novel, Storm Front, was published in 2000.",
    69: "The context states that the company went public in 1992.",
    70: "The context explicitly identifies Shaolin and Wu Tang as a film directed by Gordon Liu."
}

for pilot_id, evidence in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for pilot_id={pilot_id}")

    review2.loc[mask, "answer_verdict"] = "correct_from_context"
    review2.loc[mask, "context_support"] = "supported"
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = "none"

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 61–70.")
print("Review file:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(61, 70),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 61–70.
Review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
10,61,correct_from_context,supported
11,62,correct_from_context,supported
12,63,correct_from_context,supported
13,64,correct_from_context,supported
14,65,correct_from_context,supported
15,66,correct_from_context,supported
16,67,correct_from_context,supported
17,68,correct_from_context,supported
18,69,correct_from_context,supported
19,70,correct_from_context,supported


In [30]:

batch = review2.loc[review2["pilot_id"].between(71, 80)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 71
QUESTION: Which singer formed a band first, Chris Martin or Jimmy Gestapo?
CONTEXT: Born in Whitestone, near Exeter in Devon, Martin went to University College London where he formed a rock band with Jonny Buckland in 1996 called Pectoralz, which was eventually renamed Coldplay in 1998.James Drescher (born December 10, 1960), better known as Jimmy G or Jimmy Gestapo and also known as Jimmy Spliff, is the lead singer for New York based hardcore punk band Murphy's Law.
GENERATED ANSWER: Chris Martin formed a band first.
STATUS: success

PILOT ID: 72
QUESTION: Yellow is a Bangladesh-based fashion brand and clothing retailer owned by what, the largest of the Yellow stores are located in Bashundhara City Shopping Complex?
CONTEXT: Yellow is a Bangladesh-based fashion brand and clothing retailer owned by Beximco.Bashundhara City (Bengali: বসুন্ধরা সিটি ) is a shopping mall in Dhaka, and the second largest shopping mall in Bangladesh.
GENERATED ANSWER: Beximco
STATUS: success

P

In [31]:

from pathlib import Path
import pandas as pd

# Locate the project root.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

if not review_path.exists():
    raise FileNotFoundError(
        f"Batch 2 review file not found: {review_path}"
    )

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    71: (
        "needs_external_verification",
        "not_established_by_context",
        "The context says Chris Martin formed a band in 1996 but gives no "
        "formation date for Murphy's Law, so the comparison cannot be "
        "established from the supplied context.",
        "insufficient_comparison_evidence"
    ),
    72: (
        "correct_from_context",
        "supported",
        "The context explicitly states that Yellow is owned by Beximco.",
        "none"
    ),
    73: (
        "correct_from_context",
        "supported",
        "The context describes Ares as a magazine and Right On! as a teen magazine.",
        "none"
    ),
    74: (
        "correct_from_context",
        "supported",
        "The context identifies Porgy and Bess as an opera by George Gershwin.",
        "none"
    ),
    75: (
        "correct_from_context",
        "supported",
        "The context states that John List committed the murders in his home in Westfield, New Jersey.",
        "none"
    ),
    76: (
        "correct_from_context",
        "supported",
        "The context identifies Chicken Run as a 2000 stop-motion film.",
        "none"
    ),
    77: (
        "correct_from_context",
        "supported",
        "The context identifies John Buchan and D. H. Lawrence as writers.",
        "none"
    ),
    78: (
        "correct_from_context",
        "supported",
        "The context translates Quebec Nordiques as Quebec City Northmen or Northerners.",
        "none"
    ),
    79: (
        "needs_external_verification",
        "not_established_by_context",
        "The context provides incomplete and ambiguous release information "
        "for The Operation M.D. and does not give a comparable Framing Hanley release date.",
        "insufficient_comparison_evidence"
    ),
    80: (
        "correct_from_context",
        "supported",
        "The context gives Kenji Mizoguchi's birth year as 1898 and Alain Tanner's as 1929.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 71–80.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(71, 80),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 71–80.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
20,71,needs_external_verification,not_established_by_context
21,72,correct_from_context,supported
22,73,correct_from_context,supported
23,74,correct_from_context,supported
24,75,correct_from_context,supported
25,76,correct_from_context,supported
26,77,correct_from_context,supported
27,78,correct_from_context,supported
28,79,needs_external_verification,not_established_by_context
29,80,correct_from_context,supported


In [32]:

batch = review2.loc[review2["pilot_id"].between(81, 90)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 81
QUESTION: Martin Beck grew up 20 blocks from a neighborhood within what city of New York?
CONTEXT: Beck's "social conscience grew up with him about 20 blocks from the famous suburban Love Canal eco-disaster in Niagara Falls, New York."Love Canal is a neighborhood within Niagara Falls, New York.
GENERATED ANSWER: Niagara Falls
STATUS: success

PILOT ID: 82
QUESTION: are Richard Yates and Ernest J. Gaines both Americans ?
CONTEXT: Richard Yates (February 3, 1926 – November 7, 1992) was an American fiction writer, identified with the mid-century "Age of Anxiety".Ernest James Gaines (born January 15, 1933) is an African-American author whose works have been taught in college classrooms and translated into many languages, including French, Spanish, German, Russian and Chinese.
GENERATED ANSWER: Yes, both Richard Yates and Ernest J. Gaines are Americans.
STATUS: success

PILOT ID: 83
QUESTION: Codorus Navigation and Holyoke Canal System, are based in the United States?
CONTEXT:

In [33]:

from pathlib import Path
import pandas as pd

current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    81: (
        "correct_from_context", "supported",
        "The context identifies Love Canal as a neighborhood within Niagara Falls, New York.",
        "none"
    ),
    82: (
        "correct_from_context", "supported",
        "Richard Yates is described as an American writer and Ernest J. Gaines as an African-American author.",
        "none"
    ),
    83: (
        "correct_from_context", "supported",
        "The context places Codorus Navigation in Pennsylvania and the Holyoke Canal System in Massachusetts.",
        "none"
    ),
    84: (
        "incorrect_or_nonresponsive", "contradicted",
        "The supplied context explicitly identifies Sara Kingdom as a Doctor Who character played by Jean Marsh. The generated response contradicts the supplied context rather than answering the question.",
        "contradicted_by_supplied_context"
    ),
    85: (
        "correct_from_context", "supported",
        "The context states that Michael Strahan spent his NFL career with the New York Giants.",
        "none"
    ),
    86: (
        "correct_from_context", "supported",
        "The context explicitly identifies Batman as the superhero of whom Batkid was a sidekick.",
        "none"
    ),
    87: (
        "correct_from_context", "supported",
        "The context says Sense8 is a play on the word sensate.",
        "none"
    ),
    88: (
        "correct_from_context", "supported",
        "The context states that Cherrie Ireland was on holiday in Melbourne when she was paralysed by polio.",
        "none"
    ),
    89: (
        "correct_from_context", "supported",
        "The context identifies tamarind as a leguminous tree and identifies Imli as the tamarind flavour.",
        "none"
    ),
    90: (
        "incorrect_or_nonresponsive", "contradicted",
        "The context explicitly states that Debra Lehrmann was appointed by Governor Rick Perry. The generated response says the answer is not stated.",
        "nonresponsive_or_answer_mismatch"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for pilot ID {pilot_id}")

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 81–90.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(81, 90),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 81–90.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
30,81,correct_from_context,supported
31,82,correct_from_context,supported
32,83,correct_from_context,supported
33,84,incorrect_or_nonresponsive,contradicted
34,85,correct_from_context,supported
35,86,correct_from_context,supported
36,87,correct_from_context,supported
37,88,correct_from_context,supported
38,89,correct_from_context,supported
39,90,incorrect_or_nonresponsive,contradicted


In [34]:

batch = review2.loc[review2["pilot_id"].between(91, 100)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 91
QUESTION: Who were the creators of the long-running British soap opera in which Alison Pargeter plays the role of Sarah Cairns?
CONTEXT: Alison Pargeter (born 31 May 1972) is an English actress who played the roles of stalker Sarah Cairns in the BBC soap opera "EastEnders", also Mary Slessor in an 11-part television series of" Mary Slessor", and the Nag's Head barmaid called Val in the BBC "Only Fools and Horses" prequel "Rock & Chips".EastEnders is a long-running British soap opera created by Julia Smith and Tony Holland which has been broadcast on BBC One since 1985.
GENERATED ANSWER: Julia Smith and Tony Holland
STATUS: success

PILOT ID: 92
QUESTION: For how many seasons did the series in which Courtney Thorne-Smith played Lyndsey McElroy run ?
CONTEXT: She is best known for her roles as Alison Parker on "Melrose Place", Georgia Thomas on "Ally McBeal", Cheryl in "According to Jim", and her recurring role on "Two and a Half Men" as Lyndsey McElroy.Two and a Half Men i

In [35]:

from pathlib import Path
import pandas as pd

# Locate project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    91: (
        "correct_from_context", "supported",
        "The context identifies Julia Smith and Tony Holland as creators of EastEnders.",
        "none"
    ),
    92: (
        "incorrect_or_nonresponsive", "contradicted",
        "The context states that Two and a Half Men ran for twelve seasons, not three.",
        "contradicted_by_supplied_context"
    ),
    93: (
        "correct_from_context", "supported",
        "The context identifies the Phoenix Suns as a team based in Phoenix, Arizona.",
        "none"
    ),
    94: (
        "correct_from_context", "supported",
        "The context identifies Otis Redding as an American singer and songwriter and names him as a co-writer of Mr. Pitiful.",
        "none"
    ),
    95: (
        "needs_external_verification", "not_established_by_context",
        "The context identifies Needtobreathe as being from South Carolina but does not establish where We Are Scientists formed.",
        "insufficient_comparison_evidence"
    ),
    96: (
        "correct_from_context", "supported",
        "The context gives Kim Ji-won's birth date as October 19, 1992.",
        "none"
    ),
    97: (
        "correct_from_context", "supported",
        "The context names the Canada-U.S. Free Trade Agreement as an economic reform introduced during Brian Mulroney's tenure.",
        "none"
    ),
    98: (
        "correct_from_context", "supported",
        "The context identifies Christer Fuglesang as a Swedish physicist and ESA astronaut.",
        "none"
    ),
    99: (
        "correct_from_context", "supported",
        "The context locates the 2013 NRA 500 at Texas Motor Speedway in Fort Worth, Texas.",
        "none"
    ),
    100: (
        "correct_from_context", "supported",
        "The context explicitly identifies the Sydney Sixers WBBL team as a women's cricket team based in Sydney.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for pilot ID {pilot_id}")

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 91–100.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(91, 100),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 91–100.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
40,91,correct_from_context,supported
41,92,incorrect_or_nonresponsive,contradicted
42,93,correct_from_context,supported
43,94,correct_from_context,supported
44,95,needs_external_verification,not_established_by_context
45,96,correct_from_context,supported
46,97,correct_from_context,supported
47,98,correct_from_context,supported
48,99,correct_from_context,supported
49,100,correct_from_context,supported


In [36]:

batch = review2.loc[review2["pilot_id"].between(101, 110)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 101
QUESTION: Hank Azaria joined what cartoon as a voice actor 10 years before acting in Mystery Men?
CONTEXT: Mystery Men is a 1999 American superhero comedy film directed by Kinka Usher and written by Neil Cuthbert and Bob Burden, loosely based on Burden's "Flaming Carrot Comics", and starring Hank Azaria, Claire Forlani, Janeane Garofalo, Eddie Izzard, Greg Kinnear, William H. Macy, Kel Mitchell, Lena Olin, Paul Reubens, Geoffrey Rush, Ben Stiller, Wes Studi, and Tom Waits. He is known for starring in the animated television sitcom "The Simpsons" (1989–present), voicing Moe Szyslak, Apu Nahasapeemapetilon, Chief Wiggum, Comic Book Guy, Carl Carlson a
GENERATED ANSWER: The Simpsons
STATUS: success

PILOT ID: 102
QUESTION: Puddleglum the Marshwiggle is a character from a novel published in which year ?
CONTEXT: Lewis's gardener at The Kilns, Fred Paxford, is said to have inspired the character of Puddleglum the Marshwiggle in "The Silver Chair".The Silver Chair is a high fa

In [37]:

from pathlib import Path
import pandas as pd

# Locate the project root if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    101: (
        "correct_from_context",
        "supported",
        "The context identifies The Simpsons as the animated television sitcom "
        "in which Hank Azaria voices multiple characters.",
        "none"
    ),
    102: (
        "correct_from_context",
        "supported",
        "The context states that The Silver Chair was published in 1953.",
        "none"
    ),
    103: (
        "correct_from_context",
        "supported",
        "The context identifies the Air Force Test Center as the operator "
        "of the Arnold Engineering Development Complex flight-test facility.",
        "none"
    ),
    104: (
        "needs_external_verification",
        "not_established_by_context",
        "The context associates Bob Mould with alternative rock bands but "
        "does not explicitly establish the leadership comparison.",
        "insufficient_comparison_evidence"
    ),
    105: (
        "correct_from_context",
        "supported",
        "The context states that Andrzej Fonfara held the IBO light "
        "heavyweight title from 2012 to 2013.",
        "none"
    ),
    106: (
        "correct_from_context",
        "supported",
        "The context identifies Judge Judy Sheindlin as the retired "
        "Manhattan family court judge who presides over Judge Judy.",
        "none"
    ),
    107: (
        "needs_external_verification",
        "ambiguous",
        "The question conflates a government financial-inclusion programme "
        "with a politician, so the intended comparison is unclear.",
        "ambiguous_question"
    ),
    108: (
        "needs_external_verification",
        "ambiguous",
        "The context mentions Earl Slick's collaboration with Robert Smith "
        "and his work with Siouxsie and the Banshees, but does not clearly "
        "identify another guitarist who worked with both.",
        "ambiguous_question_or_context"
    ),
    109: (
        "correct_from_context",
        "supported",
        "The context identifies The Scarlet Letter as Lil' Mo's album and "
        "says she debuted as Missy Elliott's protégé.",
        "none"
    ),
    110: (
        "correct_from_context",
        "supported",
        "The context identifies The Best Little Whorehouse in Texas as a "
        "1982 musical comedy film starring Dolly Parton and Jim Nabors.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 101–110.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(101, 110),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 101–110.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
50,101,correct_from_context,supported
51,102,correct_from_context,supported
52,103,correct_from_context,supported
53,104,needs_external_verification,not_established_by_context
54,105,correct_from_context,supported
55,106,correct_from_context,supported
56,107,needs_external_verification,ambiguous
57,108,needs_external_verification,ambiguous
58,109,correct_from_context,supported
59,110,correct_from_context,supported


In [38]:

batch = review2.loc[review2["pilot_id"].between(111, 120)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 111
QUESTION: What genre of films are both Outfoxed and Transparent?
CONTEXT: Outfoxed: Rupert Murdoch's War on Journalism is a 2004 documentary film by filmmaker Robert Greenwald about Fox News Channel and its owner, Rupert Murdoch, promotion of right-wing views.Transparent is a 2005 documentary film written, directed, and produced by Jules Rosskam.
GENERATED ANSWER: documentary
STATUS: success

PILOT ID: 112
QUESTION: Bowling has been part of the pan american games since the games held in which city ?
CONTEXT: Bowling — primarily in the form of the tenpin sport, as regulated by the World Bowling organization (the Pan American Games' tenpin events are governed by World Bowling's "PABCON" American Zone division), has been a sport at the Pan American Games since the 1991 Pan American Games.The 11th Pan American Games were held in Havana, Cuba from August 2 to August 18, 1991.
GENERATED ANSWER: Havana
STATUS: success

PILOT ID: 113
QUESTION: What is the  large marine bivalve m

In [39]:

from pathlib import Path
import pandas as pd

current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    111: (
        "correct_from_context", "supported",
        "The context identifies both Outfoxed and Transparent as documentary films.",
        "none"
    ),
    112: (
        "correct_from_context", "supported",
        "The context states that bowling became a Pan American Games sport in 1991 and that the 1991 Games were held in Havana.",
        "none"
    ),
    113: (
        "correct_from_context", "supported",
        "The context explicitly identifies Panopea generosa as the Pacific geoduck.",
        "none"
    ),
    114: (
        "correct_from_context", "supported",
        "The context places Summertown in South Australia and refers to Australia.",
        "none"
    ),
    115: (
        "needs_external_verification", "not_established_by_context",
        "The context lists Khalid Latif among the suspended cricketers but does not establish the right-handed opening-batsman and captain details in the question.",
        "insufficient_biographical_evidence"
    ),
    116: (
        "correct_from_context", "supported",
        "The context identifies Walt Disney Productions as the producer of both films.",
        "none"
    ),
    117: (
        "correct_from_context", "supported",
        "The context identifies Emir Kusturica as Serbian and links him to Arizona Dream.",
        "none"
    ),
    118: (
        "correct_from_context", "supported",
        "The context explicitly identifies the Nobel Peace Prize Concert as an event held at Oslo Spektrum to honor laureates.",
        "none"
    ),
    119: (
        "correct_from_context", "supported",
        "The context describes Syed Ali Raza Usama as a film director and Dionciel Armstrong as a director.",
        "none"
    ),
    120: (
        "correct_from_context", "supported",
        "The context states that RMS Olympic gained the nickname Old Reliable.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 111–120.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(111, 120),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 111–120.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
60,111,correct_from_context,supported
61,112,correct_from_context,supported
62,113,correct_from_context,supported
63,114,correct_from_context,supported
64,115,needs_external_verification,not_established_by_context
65,116,correct_from_context,supported
66,117,correct_from_context,supported
67,118,correct_from_context,supported
68,119,correct_from_context,supported
69,120,correct_from_context,supported


In [40]:

batch = review2.loc[review2["pilot_id"].between(121, 130)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 121
QUESTION: Which movie starring Verna Bloom is produced by Ivan Reitman and Matty Simmons and is an American Comedy Film?
CONTEXT: She has had roles in more than 30 films and television episodes since the 1960s, including playing Mary, mother of Jesus, in "The Last Temptation of Christ" in 1988 and Marion Wormer in "Animal House" in 1978.National Lampoon's Animal House is a 1978 American comedy film from Universal Pictures. It was produced by Ivan Reitman and Matty Simmons, directed by John Landis, and stars John Belushi, Tim Matheson, John Vernon, Verna Bloom, Thomas Hulce, Stephen Furst, and Donald Sutherland.
GENERATED ANSWER: Animal House
STATUS: success

PILOT ID: 122
QUESTION: Todd Crag is a hill and a satellite peak of a hill located in the central part of where?
CONTEXT: Todd Crag is a hill and a satellite peak of Loughrigg Fell at 224 m (735 ft) it is not classed as a fell but is a vantage point near AmblesideLoughrigg Fell is a hill in the central part of the En

In [41]:

from pathlib import Path
import pandas as pd

# Locate project root in case the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    121: (
        "correct_from_context", "supported",
        "The context identifies National Lampoon's Animal House as a 1978 "
        "American comedy film produced by Ivan Reitman and Matty Simmons "
        "and starring Verna Bloom.",
        "none"
    ),
    122: (
        "correct_from_context", "supported",
        "The context identifies Loughrigg Fell as being in the central "
        "part of the English Lake District.",
        "none"
    ),
    123: (
        "correct_from_context", "supported",
        "The context identifies B. D. Wong as a cast member and says he "
        "won a Tony Award for M. Butterfly.",
        "none"
    ),
    124: (
        "correct_from_context", "supported",
        "The context identifies Cherry Hill as a historic house in Albany, "
        "New York, United States.",
        "none"
    ),
    125: (
        "needs_external_verification", "ambiguous",
        "The context lists multiple Barry Keoghan films before saying "
        "'It was selected' for the Palme d'Or; the antecedent is ambiguous.",
        "ambiguous_source_reference"
    ),
    126: (
        "correct_from_context", "supported",
        "The context identifies both Craig Serling and Jeff Celentano "
        "as American.",
        "none"
    ),
    127: (
        "correct_from_context", "supported",
        "The context explicitly says Paul Auster's books have been "
        "translated into more than forty languages.",
        "none"
    ),
    128: (
        "correct_from_context", "supported",
        "The context gives Edward III's reign as January 1327 to June 1377, "
        "approximately 50 years.",
        "none"
    ),
    129: (
        "correct_from_context", "supported",
        "The context explicitly identifies Salten as the traditional region.",
        "none"
    ),
    130: (
        "incorrect_or_nonresponsive", "contradicted",
        "The supplied context describes Cyclamen as native to Europe, "
        "the Mediterranean, Iran and Somalia, while the Fargesia description "
        "lists regions in Asia. This points toward Cyclamen spanning more "
        "continents.",
        "contradicted_by_supplied_context"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 121–130.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(121, 130),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 121–130.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
70,121,correct_from_context,supported
71,122,correct_from_context,supported
72,123,correct_from_context,supported
73,124,correct_from_context,supported
74,125,needs_external_verification,ambiguous
75,126,correct_from_context,supported
76,127,correct_from_context,supported
77,128,correct_from_context,supported
78,129,correct_from_context,supported
79,130,incorrect_or_nonresponsive,contradicted


In [42]:

batch = review2.loc[review2["pilot_id"].between(131, 140)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 131
QUESTION: In what city was the Bill Cosby album For Adults Only recorded?
CONTEXT: It was recorded at the Westgate Las Vegas Resort & Casino, then known as the International Hotel.The Westgate Las Vegas Resort & Casino is a hotel and casino in Winchester, Nevada.
GENERATED ANSWER: The context is incorrect. The Westgate Las Vegas Resort & Casino is located in Las Vegas, Nevada, not Winchester, Nevada. Therefore, the Bill Cosby album "For Adults Only" was recorded in Las Vegas, Nevada.
STATUS: success

PILOT ID: 132
QUESTION: What movement is David Yerushalmi behind that would prohibit the implementation of Islanic law in courts in any civil jurisdiction?
CONTEXT: David Yerushalmi (born 1956) is an American lawyer and political activist who is the driving counsel behind the anti-sharia movement in the United States.A ban on sharia law is legislation that would prohibit the application or implementation of Islamic law (sharia) in courts in any civil (non-religious) jurisdic

In [43]:

from pathlib import Path
import pandas as pd

# Locate project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    131: (
        "needs_external_verification",
        "ambiguous",
        "The context identifies the recording venue as the Westgate Las Vegas Resort "
        "and describes that venue as being in Winchester, Nevada. The location "
        "wording needs clarification.",
        "source_location_ambiguity"
    ),
    132: (
        "correct_from_context",
        "supported",
        "The context describes David Yerushalmi as driving counsel for the anti-sharia movement.",
        "none"
    ),
    133: (
        "needs_external_verification",
        "not_established_by_context",
        "The displayed context is truncated before the relevant ancestry statement is complete. "
        "Inspect the full context before finalizing the verdict.",
        "truncated_context"
    ),
    134: (
        "needs_external_verification",
        "not_established_by_context",
        "The displayed context is truncated before the relevant player-of-the-year details "
        "are complete. Inspect the full context before finalizing the verdict.",
        "truncated_context"
    ),
    135: (
        "correct_from_context",
        "supported",
        "The context identifies Nathan Summers, also called Cable, as the Marvel Comics character.",
        "extraneous_formatting"
    ),
    136: (
        "correct_from_context",
        "supported",
        "The context identifies Granby, Colorado, as the location of the events involving Marvin Heemeyer.",
        "none"
    ),
    137: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The answer repeats the park's name, while the question asks for its location. "
        "The context places it in Polk County, Tennessee.",
        "nonresponsive_or_answer_mismatch"
    ),
    138: (
        "correct_from_context",
        "supported",
        "The context describes David Gordon Green as a filmmaker and Larry Hagman as an actor; "
        "the generated answer correctly distinguishes their occupations.",
        "none"
    ),
    139: (
        "correct_from_context",
        "supported",
        "The context states that Norma McCorvey became an activist in the pro-life movement.",
        "none"
    ),
    140: (
        "correct_from_context",
        "supported",
        "The context identifies Anthony Burgess as an English writer and composer.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment; unresolved cases require further review."
        if verdict == "needs_external_verification"
        else "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 131–140.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(131, 140),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 131–140.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
80,131,needs_external_verification,ambiguous
81,132,correct_from_context,supported
82,133,needs_external_verification,not_established_by_context
83,134,needs_external_verification,not_established_by_context
84,135,correct_from_context,supported
85,136,correct_from_context,supported
86,137,incorrect_or_nonresponsive,not_supported_as_an_answer
87,138,correct_from_context,supported
88,139,correct_from_context,supported
89,140,correct_from_context,supported


In [44]:

batch = review2.loc[review2["pilot_id"].between(141, 150)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 141
QUESTION: What is the name of this 1998 British-American crime comedy film, whose soundtrack included Lujon?
CONTEXT: "Lujon" (also known as "Slow Hot Wind") is a musical piece by Henry Mancini. It was included in the soundtracks for the films "The Big Lebowski", "Sexy Beast", "W.E.", and "Two Lovers".The Big Lebowski is a 1998 British-American crime comedy film written, produced, and directed by Joel and Ethan Coen.
GENERATED ANSWER: The Big Lebowski
STATUS: success

PILOT ID: 142
QUESTION: Martha Ellen Reeves got her masters at a university located in which US state ?
CONTEXT: Reeves studied at the University of Montana (B.A. 1973), Truman State University (M.A. 1976) in English literature and Keele University, (Ph.D. 1998), for industrial relations and human resources management. The university is named after U.S. President Harry Truman, the only president born in Missouri.
GENERATED ANSWER: Missouri
STATUS: success

PILOT ID: 143
QUESTION: Alex Joseph "AJ" Pritchard 

In [45]:

from pathlib import Path
import pandas as pd

current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    141: (
        "correct_from_context", "supported",
        "The context identifies The Big Lebowski as a 1998 British-American "
        "crime comedy film and says Lujon was included in its soundtrack.",
        "none"
    ),
    142: (
        "needs_external_verification", "not_established_by_context",
        "The context identifies Truman State University as the institution "
        "where Reeves earned her master's degree, but does not explicitly "
        "state the university's location.",
        "insufficient_location_evidence"
    ),
    143: (
        "correct_from_context", "supported",
        "The context identifies Chloe Hewitt as Pritchard's partner and "
        "describes her as a dancer and choreographer born in 1995.",
        "none"
    ),
    144: (
        "correct_from_context", "supported",
        "The context explicitly states that Hani Ramadan is originally from Egypt.",
        "none"
    ),
    145: (
        "incorrect_or_nonresponsive", "contradicted",
        "The context says Louis José Lucien Dolhem was born on 26 April 1944 "
        "and identifies Didier Pironi as his half-brother. The question asks "
        "for the birth date of the driver who had José Dolhem as a half-brother, "
        "so the generated answer appears to give the wrong person's birth date.",
        "confused_related_people"
    ),
    146: (
        "correct_from_context", "supported",
        "The context identifies the 1988 Victorian Football League season "
        "as the 92nd season of the elite competition.",
        "none"
    ),
    147: (
        "correct_from_context", "supported",
        "The context identifies Curzon Ashton Ladies Football Club as a women's club.",
        "none"
    ),
    148: (
        "correct_from_context", "supported",
        "The context states that Donald John Pinkava is Professor Emeritus "
        "at Arizona State University.",
        "none"
    ),
    149: (
        "needs_external_verification", "ambiguous",
        "The context connects Ashley Lloyd with the stage musical Whistle Down "
        "the Wind, but does not clearly establish the film-cast relationship "
        "described in the question.",
        "ambiguous_film_vs_musical"
    ),
    150: (
        "correct_from_context", "supported",
        "The context identifies Jacques Tourneur and Alex Cox as film directors.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 141–150.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(141, 150),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 141–150.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
90,141,correct_from_context,supported
91,142,needs_external_verification,not_established_by_context
92,143,correct_from_context,supported
93,144,correct_from_context,supported
94,145,incorrect_or_nonresponsive,contradicted
95,146,correct_from_context,supported
96,147,correct_from_context,supported
97,148,correct_from_context,supported
98,149,needs_external_verification,ambiguous
99,150,correct_from_context,supported


In [46]:

batch = review2.loc[review2["pilot_id"].between(151, 160)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 151
QUESTION: Are Ben Okri and Nicholas Evans both novelists?
CONTEXT: Ben Okri OBE FRSL (born 15 March 1959) is a Nigerian poet and novelist.Nicholas Evans (born 26 July 1950) is an English journalist, screenwriter television and film producer and novelist.
GENERATED ANSWER: Yes, both Ben Okri and Nicholas Evans are novelists.
STATUS: success

PILOT ID: 152
QUESTION: The papal election of 1159 followed the death of which pope, who ruled from 1154 until 1159?
CONTEXT: The Papal election of 4–7 September 1159 followed the death of Pope Adrian IV.Pope Adrian IV (Latin: "Adrianus IV" ; born Nicholas Breakspear;  1100 1 September 1159) was Pope from 4 December 1154 to his death in 1159.
GENERATED ANSWER: Pope Adrian IV
STATUS: success

PILOT ID: 153
QUESTION: What was the nationality of the mixed martial artist who defeated Trent Jenkins in the first ever fight in UFC history?
CONTEXT: Jenkins also was in the first ever fight in UFC history losing to Jason DeLucia.Jason DeLucia 

In [47]:

from pathlib import Path
import pandas as pd

# Locate the project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    151: (
        "Both Ben Okri and Nicholas Evans are explicitly described as novelists."
    ),
    152: (
        "The context explicitly identifies Pope Adrian IV as the pope who died in 1159."
    ),
    153: (
        "The context identifies Jason DeLucia as the fighter who defeated Trent Jenkins "
        "and identifies DeLucia as American."
    ),
    154: (
        "The context identifies Robin Beck as the singer whose European tour Jorge Salan "
        "worked on and gives her birthplace as Brooklyn, New York."
    ),
    155: (
        "The context states that the series was cancelled after only 12 episodes had aired."
    ),
    156: (
        "The context states that Richard Lindzen was the Alfred P. Sloan Professor of "
        "Meteorology at the Massachusetts Institute of Technology during the period "
        "when he proposed the hypothesis."
    ),
    157: (
        "The context identifies the International Earth Rotation and Reference Systems "
        "Service as the organization maintaining the IERS Reference Meridian."
    ),
    158: (
        "The context identifies both Gareth Evans and Michael Winner as film directors."
    ),
    159: (
        "The context says Tyler Pope played with Hercules and Love Affair and identifies "
        "it as a dance music project."
    ),
    160: (
        "The context states that the Bee Gees compilation album Kitty Can was released "
        "only in South America."
    )
}

for pilot_id, evidence in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = "correct_from_context"
    review2.loc[mask, "context_support"] = "supported"
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = "none"

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 151–160.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(151, 160),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 151–160.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
100,151,correct_from_context,supported
101,152,correct_from_context,supported
102,153,correct_from_context,supported
103,154,correct_from_context,supported
104,155,correct_from_context,supported
105,156,correct_from_context,supported
106,157,correct_from_context,supported
107,158,correct_from_context,supported
108,159,correct_from_context,supported
109,160,correct_from_context,supported


In [48]:

batch = review2.loc[review2["pilot_id"].between(161, 170)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 161
QUESTION: Which building is older, the Mısır Apartment or the Abdi İpekçi Arena?
CONTEXT: The Mısır Apartment or Mısır Apartmanı (Turkish for "Egypt Apartment") is a famed historical building on the renowned İstiklal Avenue in the Beyoğlu district of Istanbul, Turkey.Abdi İpekçi Arena, formerly known as Abdi İpekçi Sports Complex, is a multi-purpose indoor arena located in the Zeytinburnu district of Istanbul, Turkey, situated just outside the ancient city walls in Yedikule.
GENERATED ANSWER: The Mısır Apartment is older than the Abdi İpekçi Arena.
STATUS: success

PILOT ID: 162
QUESTION: In what party is the president whose cabinet Francisco Javier Mayorga Castañeda serves in?
CONTEXT: Francisco Javier Mayorga Castañeda (born on 17 April 1951) is a Mexican businessman and politician who serves as Secretary of Agriculture in the cabinet of President Felipe Calderón since 7 September 2009. He has previously served in the same post from 28 September 2005 to 30 November 200

In [49]:

from pathlib import Path
import pandas as pd

# Find project root if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    161: (
        "needs_external_verification",
        "not_established_by_context",
        "The context gives no construction dates for either building, "
        "so their relative ages cannot be determined from the supplied text.",
        "insufficient_comparison_evidence"
    ),
    162: (
        "needs_external_verification",
        "not_established_by_context",
        "The context identifies Francisco Javier Mayorga Castañeda as a "
        "PAN member but does not explicitly identify Felipe Calderón's party.",
        "insufficient_political_affiliation_evidence"
    ),
    163: (
        "correct_from_context",
        "supported",
        "The context states that Stephen Robert Dunlop succumbed to his "
        "injuries in a Coleraine hospital.",
        "none"
    ),
    164: (
        "correct_from_context",
        "supported",
        "The context states that the 1994 players' strike ended the team's "
        "postseason aspirations.",
        "none"
    ),
    165: (
        "correct_from_context",
        "supported",
        "The context identifies Bronco as a series on ABC from 1958 through 1962.",
        "none"
    ),
    166: (
        "correct_from_context",
        "supported",
        "The context says Carsten Olsen was employed by the Carlsberg Laboratory "
        "as an assistant to S. P. L. Sørensen.",
        "none"
    ),
    167: (
        "correct_from_context",
        "supported",
        "The context states that Johann Duhaupas holds notable victories over "
        "Manuel Charr and Robert Helenius.",
        "none"
    ),
    168: (
        "correct_from_context",
        "supported",
        "The context identifies James Patterson as an American author and "
        "philanthropist and names him as a co-author of Sundays at Tiffany's.",
        "none"
    ),
    169: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The context identifies both Eugene Onegin and Il trovatore as operas, "
        "but does not establish that they share El trovador as a dramatic work.",
        "unsupported_relationship"
    ),
    170: (
        "correct_from_context",
        "not_established_by_context",
        "The supplied context does not establish a relationship between the "
        "station and FamilyMart. The answer appropriately limits its claim "
        "to what is shown in the provided context.",
        "appropriate_context_limited_abstention"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 161–170.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(161, 170),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 161–170.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
110,161,needs_external_verification,not_established_by_context
111,162,needs_external_verification,not_established_by_context
112,163,correct_from_context,supported
113,164,correct_from_context,supported
114,165,correct_from_context,supported
115,166,correct_from_context,supported
116,167,correct_from_context,supported
117,168,correct_from_context,supported
118,169,incorrect_or_nonresponsive,not_supported_as_an_answer
119,170,correct_from_context,not_established_by_context


In [50]:

batch = review2.loc[review2["pilot_id"].between(171, 180)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 171
QUESTION: What is the name of this casino hotel on the Las Vegas Strip in Paradise, Nevada, for which Allard Roen was the managing director?
CONTEXT: Allard Roen (May 8, 1921–August 28, 2008) was an American businessman in the hospitality industry. He was the Managing Director of the Desert Inn and the Stardust Resort and Casino in Paradise, Nevada.The Desert Inn, also known as the D.I., was a casino hotel on the Las Vegas Strip in Paradise, Nevada, which operated from April 24, 1950, to August 28, 2000.
GENERATED ANSWER: The Desert Inn
STATUS: success

PILOT ID: 172
QUESTION: What year was Dražen Dalipagić enshrined into a museum in Springfield, Massachusetts
CONTEXT: Dalipagić was enshrined into the Naismith Memorial Basketball Hall of Fame as a player in 2004 and into the FIBA Hall of Fame in 2007.The Naismith Memorial Basketball Hall of Fame is an American history museum and hall of fame, located at 1000 Hall of Fame Avenue in Springfield, Massachusetts.
GENERATED AN

In [51]:

from pathlib import Path
import pandas as pd

# Restore the project path if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    171: (
        "correct_from_context", "supported",
        "The context identifies the Desert Inn as a casino hotel and says "
        "Allard Roen was its managing director.",
        "none"
    ),
    172: (
        "correct_from_context", "supported",
        "The context states that Dražen Dalipagić was inducted into the "
        "Naismith Memorial Basketball Hall of Fame in 2004.",
        "none"
    ),
    173: (
        "correct_from_context", "supported",
        "The context lists Bellagio among the MGM Resorts properties overseen "
        "by Corey Sanders and identifies it as a luxury hotel and casino.",
        "none"
    ),
    174: (
        "correct_from_context", "supported",
        "The context explicitly states that Andrzej Żuławski often went "
        "against mainstream commercialism in his films.",
        "none"
    ),
    175: (
        "needs_external_verification", "ambiguous",
        "The context calls Mobile an alternative rock band and describes "
        "Poe's music as a blend of several genres. It does not clearly "
        "establish the subjective comparison in the question.",
        "ambiguous_question"
    ),
    176: (
        "correct_from_context", "supported",
        "The context identifies Dimension Films as the distribution label "
        "used by The Weinstein Company for Scary Movie 5.",
        "none"
    ),
    177: (
        "correct_from_context", "supported",
        "The context identifies Ken Tucker as an American arts critic and "
        "writer of Scarface Nation.",
        "none"
    ),
    178: (
        "correct_from_context", "supported",
        "The context gives Tanya Donelly's birth year as 1966 and Trace "
        "Cyrus's as 1989.",
        "none"
    ),
    179: (
        "correct_from_context", "supported",
        "The context identifies Goodison Park as being in Liverpool, England.",
        "none"
    ),
    180: (
        "correct_from_context", "supported",
        "The context identifies Paramore as an American rock band from "
        "Franklin, Tennessee, formed in 2004.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 171–180.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(171, 180),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 171–180.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
120,171,correct_from_context,supported
121,172,correct_from_context,supported
122,173,correct_from_context,supported
123,174,correct_from_context,supported
124,175,needs_external_verification,ambiguous
125,176,correct_from_context,supported
126,177,correct_from_context,supported
127,178,correct_from_context,supported
128,179,correct_from_context,supported
129,180,correct_from_context,supported


In [52]:

batch = review2.loc[review2["pilot_id"].between(181, 190)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 181
QUESTION: Which  British politician from Northern Ireland who is a Democratic Unionist Party Member of Parliament for East Antrim represents the constituency known as East Antrim?
CONTEXT: East Antrim is a constituency represented in the House of Commons of the UK Parliament since 2005 by Sammy Wilson, a member of the Democratic Unionist Party.Samuel Wilson (born 4 April 1953) is a British politician from Northern Ireland who is a Democratic Unionist Party (DUP) Member of Parliament (MP) for East Antrim.
GENERATED ANSWER: Sammy Wilson
STATUS: success

PILOT ID: 182
QUESTION: What was the first year the NinjaTel van's model released?
CONTEXT: The NinjaTel Van is a 2001 Ford Econoline E250 van, designed and converted by Bob "saberfire" Bristow and Colleen "Phar" Campbell into the base of operation for NinjaTel. Introduced for the 1961 model year as the replacement for the Ford F-Series panel van, four generations of the model line have been produced.
GENERATED ANSWER: 2001

In [53]:

from pathlib import Path
import pandas as pd

# Locate the project root if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    181: (
        "correct_from_context", "supported",
        "The context identifies Sammy Wilson as a DUP MP representing East Antrim.",
        "none"
    ),
    182: (
        "incorrect_or_nonresponsive", "contradicted",
        "The context identifies the van as a 2001 vehicle but says the model line "
        "was introduced for the 1961 model year. The answer appears to confuse "
        "the individual vehicle's year with the model's first release year.",
        "confused_vehicle_year_with_model_introduction"
    ),
    183: (
        "correct_from_context", "supported",
        "The context explicitly identifies The Last Boy Scout as the 1991 film "
        "that preceded Perfect Stranger as the actors' previous film together.",
        "none"
    ),
    184: (
        "correct_from_context", "not_established_by_context",
        "The context identifies Smokers Die Younger as a rock band but does not "
        "state which avant-garde musical form it performed. The answer appropriately "
        "abstains based on the available context.",
        "appropriate_context_limited_abstention"
    ),
    185: (
        "correct_from_context", "supported",
        "The context gives Terence Hill's birth date as 29 March 1939.",
        "none"
    ),
    186: (
        "correct_from_context", "supported",
        "The context identifies Kaze, Ghost Warrior as Amadhia Albee's independent "
        "computer-animated film featuring anthropomorphic animals.",
        "none"
    ),
    187: (
        "correct_from_context", "supported",
        "The context identifies More Life as the Drake album featuring Jorja Smith.",
        "none"
    ),
    188: (
        "correct_from_context", "supported",
        "The context identifies Moon Hyung-joo as the grandmother of the orphaned siblings.",
        "none"
    ),
    189: (
        "incorrect_or_nonresponsive", "supported",
        "The context places the castles in Aberdeenshire, but the question asks "
        "for a town. It specifically places Old Slains Castle near Collieston "
        "without clearly establishing a town for New Slains Castle.",
        "answer_type_mismatch_county_instead_of_town"
    ),
    190: (
        "correct_from_context", "supported",
        "The context identifies Brookhaven High School as being in Columbus, Ohio.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 181–190.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(181, 190),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 181–190.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
130,181,correct_from_context,supported
131,182,incorrect_or_nonresponsive,contradicted
132,183,correct_from_context,supported
133,184,correct_from_context,not_established_by_context
134,185,correct_from_context,supported
135,186,correct_from_context,supported
136,187,correct_from_context,supported
137,188,correct_from_context,supported
138,189,incorrect_or_nonresponsive,supported
139,190,correct_from_context,supported


In [54]:

batch = review2.loc[review2["pilot_id"].between(191, 200)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 191
QUESTION: Even though Big Stone Gap takes place in Virginia, where is Adriana Trigiani based?
CONTEXT: Based on Trigiani's 2000 best-selling novel of the same name, the story is set in the actual Virginia town of Big Stone Gap circa 1970s.Adriana Trigiani is an Italian American best-selling author of sixteen books, television writer, film director, and entrepreneur based in Greenwich Village, New York City.
GENERATED ANSWER: Greenwich Village, New York City
STATUS: success

PILOT ID: 192
QUESTION: Rectify was created by which American actor and screenwriter?
CONTEXT: It was created by Ray McKinnon and is the first original series from SundanceTV.Raymond "Ray" Wilkes McKinnon (born November 15, 1957) is an American actor, screenwriter, film director and producer.
GENERATED ANSWER: Ray McKinnon
STATUS: success

PILOT ID: 193
QUESTION: What method of murder was used by English serial killer Graham Young whose life was portrayed in a 1995 black comedy film directed by Benjam

In [55]:

from pathlib import Path
import pandas as pd

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    191: (
        "Adriana Trigiani is explicitly described as based in Greenwich Village, New York City."
    ),
    192: (
        "The context identifies Ray McKinnon as the creator of Rectify."
    ),
    193: (
        "The context explicitly states that Graham Young used poison to kill his victims."
    )
}

for pilot_id, evidence in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for pilot ID {pilot_id}")

    review2.loc[mask, "answer_verdict"] = "correct_from_context"
    review2.loc[mask, "context_support"] = "supported"
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = "none"

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 191–193.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(191, 193),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 191–193.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
140,191,correct_from_context,supported
141,192,correct_from_context,supported
142,193,correct_from_context,supported


In [56]:

batch = review2.loc[review2["pilot_id"].between(194, 196)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:400])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 194
QUESTION: What is the nationality of the woman who performed "A Message to Your Heart" at the Eurovision Song Contest in 1991?
CONTEXT: "A Message to Your Heart", written and composed by Paul Curtis, was the United Kingdom's entry at the Eurovision Song Contest 1991, performed by Samantha Janus.Samantha Zoe Womack ( Janus; born 2 November 1972) is a British actress, singer and director, in film, television and stage.
GENERATED ANSWER: British
STATUS: success

PILOT ID: 195
QUESTION: Susan Floyd starred opposite the actor and singer in "Chinese Coffee" of what nationality?
CONTEXT: She has also had featured roles in several motion pictures, including "Domestic Disturbance" and "Forgiven", and starred opposite Al Pacino and Jerry Orbach in "Chinese Coffee".Jerome Bernard "Jerry" Orbach (October 20, 1935 – December 28, 2004) was an American actor and singer, described at the time of his death as "one of the last" bona fide" leading men of the Broadway musical and global cel

In [57]:

from pathlib import Path
import pandas as pd

# Locate the project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    194: (
        "The context identifies Samantha Janus, who performed A Message to Your Heart, as British."
    ),
    195: (
        "The context identifies Jerry Orbach as an American actor and singer."
    ),
    196: (
        "The context identifies Home and Away as the series created by Alan Bateman."
    )
}

for pilot_id, evidence in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = "correct_from_context"
    review2.loc[mask, "context_support"] = "supported"
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = "none"

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 194–196.")
print("Review file:", review_path)

# Display the final four examples in this 200-question batch.
batch = review2.loc[review2["pilot_id"].between(197, 200)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:450])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


Saved assessments for IDs 194–196.
Review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv

PILOT ID: 197
QUESTION: When was the American singer and musician born who was also the client of Vern Yocum?
CONTEXT: Vern Yocum (April 15, 1909 - September 13, 1991) is best known as copyist and librarian for Frank Sinatra, Nat King Cole, and Nelson Riddle. His client list included: Frankie Laine, Billy Eckstine, Ella Fitzgerald, Rosemary Clooney, Mel Tormé, Peggy Lee, Steve Lawrence & Eydie Gorme, Keely Smith, Sammy Davis Jr., Trini Lopez, Nancy Wilson, Leslie Uggams, Roy Clark, and Julie Andrews.Roy Linwood Clark (born April 15, 1933) is an American singer an
GENERATED ANSWER: April 15, 1933
STATUS: success

PILOT ID: 198
QUESTION: Which Dutch-British CGI animated television series originally aired on KRO in the Netherlands is based on the "Miffy" book series by Dutch artist Dick Bruna?
CONTEXT: Miffy's Adventures 

In [58]:

from pathlib import Path
import pandas as pd

# Locate the project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    197: (
        "correct_from_context",
        "supported",
        "The context identifies Roy Clark as an American singer and gives his birth date as April 15, 1933.",
        "none"
    ),
    198: (
        "correct_from_context",
        "supported",
        "The context identifies Miffy's Adventures Big and Small as the Dutch-British CGI-animated television series based on the Miffy books.",
        "none"
    ),
    199: (
        "incorrect_or_nonresponsive",
        "supported",
        "The context describes Eric Greitens as a former Navy SEAL. The answer is related to the evidence, but the question asks for the military branch rather than the specific role.",
        "answer_type_mismatch_unit_vs_branch"
    ),
    200: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The context gives the 19th Lumières Awards ceremony date as January 20, 2014, but the generated answer does not answer when the film competed and its claim about the films not competing for Best Film is not established by the supplied text.",
        "nonresponsive_or_unsupported_claim"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 197–200.")
print("Review file:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(197, 200),
        [
            "pilot_id",
            "generated_answer",
            "answer_verdict",
            "context_support",
            "answer_error_type"
        ]
    ]
)


Saved assessments for IDs 197–200.
Review file: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,generated_answer,answer_verdict,context_support,answer_error_type
146,197,"April 15, 1933",correct_from_context,supported,none
147,198,Miffy's Adventures Big and Small,correct_from_context,supported,none
148,199,Navy SEAL,incorrect_or_nonresponsive,supported,answer_type_mismatch_unit_vs_branch
149,200,"The romantic drama ""Blue Is the Warmest Colour...",incorrect_or_nonresponsive,not_supported_as_an_answer,nonresponsive_or_unsupported_claim


In [59]:

# Audit batch 2, pilot IDs 51–200.

batch2 = review2.loc[
    review2["pilot_id"].between(51, 200)
].copy()

print("Total batch rows:", len(batch2))
print("Unique IDs:", batch2["pilot_id"].nunique())

print("\nGeneration status:")
print(
    batch2["generation_status"]
    .fillna("missing")
    .value_counts()
    .to_string()
)

print("\nAnswer verdict distribution:")
print(
    batch2["answer_verdict"]
    .fillna("")
    .replace("", "not_reviewed")
    .value_counts()
    .to_string()
)

print("\nRows still missing verdicts:")
print(
    batch2.loc[
        batch2["answer_verdict"].fillna("").eq(""),
        "pilot_id"
    ].tolist()
)

print("\nRows requiring external verification:")
print(
    batch2.loc[
        batch2["answer_verdict"].eq("needs_external_verification"),
        ["pilot_id", "question", "generated_answer"]
    ].to_string(index=False)
)

print("\nRows marked as source-inconsistent:")
print(
    batch2.loc[
        batch2["answer_verdict"].eq("source_inconsistent"),
        ["pilot_id", "question"]
    ].to_string(index=False)
)


Total batch rows: 150
Unique IDs: 150

Generation status:
generation_status
success    150

Answer verdict distribution:
answer_verdict
correct_from_context           122
needs_external_verification     16
incorrect_or_nonresponsive      12

Rows still missing verdicts:
[]

Rows requiring external verification:
 pilot_id                                                                                                                                                                                                                                                                                                                     question                                                                                                                                                                                                                                                                                                                                                                         

In [60]:

# Display the next 10 generated questions for annotation

batch = review2.loc[
    review2["pilot_id"].between(201, 210)
]

if batch.empty:
    print("No rows found for IDs 201–210.")
    print("Check whether the second batch contains all 250 questions.")
else:
    for _, row in batch.iterrows():
        print("\n" + "=" * 65)
        print(f"PILOT ID: {row['pilot_id']}")
        print("QUESTION:", str(row["question"])[:350])
        print("CONTEXT:", str(row["context"])[:550])
        print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
        print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 201
QUESTION: Mickey's Mellerdrammer is based on a Harriet Beecher Stowe novel published in which year ?
CONTEXT: The title is a corruption of "melodrama", thought to harken back to the earliest minstrel shows, as a film short based on Harriet Beecher Stowe's anti-slavery novel "Uncle Tom's Cabin" and stars Mickey Mouse and his friends who stage their own production of the novel. Published in 1852, the novel "helped lay the groundwork for the Civil War", according to Will Kaufman.
GENERATED ANSWER: 1852
STATUS: success

PILOT ID: 202
QUESTION: What game featured In the Streets to Africa?
CONTEXT: This album featured the number 54 charting song "Youths Dem Cold" which was also featured on the "Grand Theft Auto IV" soundtrack.Grand Theft Auto IV is an action-adventure video game developed by Rockstar North and published by Rockstar Games.
GENERATED ANSWER: The game that featured "Youths Dem Cold" is Grand Theft Auto IV.
STATUS: success

PILOT ID: 203
QUESTION: What is the capi

In [61]:

from pathlib import Path
import pandas as pd

# Locate project root if the kernel has restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    201: (
        "correct_from_context",
        "supported",
        "The context says Uncle Tom's Cabin was published in 1852.",
        "none"
    ),
    202: (
        "needs_external_verification",
        "ambiguous",
        "The context connects the song Youths Dem Cold to the Grand Theft Auto IV soundtrack, "
        "but it does not clearly establish what 'In the Streets to Africa' refers to in the question.",
        "ambiguous_question_or_context"
    ),
    203: (
        "correct_from_context",
        "supported",
        "The context identifies Moroni as the capital associated with the Comoros.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 201–203.")
print("File:", review_path)
display(
    review2.loc[
        review2["pilot_id"].between(201, 203),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 201–203.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
150,201,correct_from_context,supported
151,202,needs_external_verification,ambiguous
152,203,correct_from_context,supported


In [62]:

batch = review2.loc[review2["pilot_id"].between(204, 206)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 204
QUESTION: The male star of My Santa was born in what year?
CONTEXT: My Santa is a 2013 television film starring Samaire Armstrong and Matthew Lawrence.Matthew William Lawrence (born February 11, 1980) is an American actor known for his role as Sam Collins in television series "Superhuman Samurai Syber-Squad" and his role of Jack Hunter on the ABC sitcom "Boy Meets World" from 1997 to 2000 and its spin-off "Girl Meets World", as well as his roles in movies such as "The Hot Chick", "The Comebacks", and "Mrs. Doubtfire".
GENERATED ANSWER: 1980
STATUS: success

PILOT ID: 205
QUESTION: Who now owns the resort that reflects Mandalay Resort Group's former name?
CONTEXT: Mandalay Resort Group (formerly Circus Circus Enterprises) was a hotel-casino operator based in Paradise, Nevada. It is owned and operated by MGM Resorts International.
GENERATED ANSWER: MGM Resorts International owns the resort that formerly bore the name of Mandalay Resort Group.
STATUS: success

PILOT ID: 206

In [63]:

from pathlib import Path
import pandas as pd

# Locate the project root after a possible kernel restart.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    204: (
        "correct_from_context",
        "supported",
        "The context identifies Matthew Lawrence as a star of My Santa and gives his birth year as 1980.",
        "none"
    ),
    205: (
        "correct_from_context",
        "supported",
        "The context states that Mandalay Resort Group, formerly Circus Circus Enterprises, is owned and operated by MGM Resorts International.",
        "none"
    ),
    206: (
        "correct_from_context",
        "supported",
        "The context explicitly states that Union College was founded in 1795.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 204–206.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(204, 206),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 204–206.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
153,204,correct_from_context,supported
154,205,correct_from_context,supported
155,206,correct_from_context,supported


In [64]:

batch = review2.loc[review2["pilot_id"].between(207, 210)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 207
QUESTION: Javier Mora, is a Mexican boxer in the Heavyweight division, he was defeated by Alexander Vladimirovich "Sasha" Povetkin, by a 5-round TKO, in which country?
CONTEXT: He was defeated by Alexander Povetkin by a 5-round TKO, in Max Schmeling Halle, Prenzlauer Berg, Berlin, Germany.Alexander Vladimirovich "Sasha" Povetkin ( ; Russian: Алекса́ндр Влади́мирович Пове́ткин ; born 2 September 1979) is a Russian professional boxer who held the WBA (Regular) heavyweight title from 2011 to 2013.
GENERATED ANSWER: Germany
STATUS: success

PILOT ID: 208
QUESTION: Name the member of Ratt that has played music with Dario Seixas Filho and Firedome?
CONTEXT: Seixas has also toured with metal legend Stephen Pearcy of RATT, and is a current member of the bands Goodbye Thrill and Crown of Thorns. He has also been in the bands Firedome, Crystal Pystal, Arcade, Vicious Delite, Vertex.
GENERATED ANSWER: Dario Seixas Filho
STATUS: success

PILOT ID: 209
QUESTION: Singer Crystal Gayle’

In [65]:

from pathlib import Path
import pandas as pd

# Locate the project root if the notebook kernel restarted.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    207: (
        "correct_from_context",
        "supported",
        "The context states that the fight took place at Max Schmeling Halle in Berlin, Germany.",
        "none"
    ),
    208: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "The context says Dario Seixas Filho toured with Stephen Pearcy of Ratt and lists Firedome among Dario's bands. The answer names Dario rather than identifying the Ratt member asked for.",
        "entity_confusion_ambiguous_relationship"
    ),
    209: (
        "correct_from_context",
        "supported",
        "The context identifies Silver Spoons as the sitcom in which Erin Gray played Kate Summers-Stratton.",
        "none"
    ),
    210: (
        "correct_from_context",
        "supported",
        "The context identifies Casper as the film Sherri Stoner worked on with Deanna Oliver and says it is based on the character created by Seymour Reit and Joe Oriolo.",
        "none"
    )
}

for pilot_id, (verdict, support, evidence, error_type) in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}"
        )

    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 207–210.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(207, 210),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 207–210.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
156,207,correct_from_context,supported
157,208,incorrect_or_nonresponsive,not_supported_as_an_answer
158,209,correct_from_context,supported
159,210,correct_from_context,supported


In [66]:

batch = review2.loc[review2["pilot_id"].between(211, 220)]

for _, row in batch.iterrows():
    print("\n" + "=" * 65)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", str(row["question"])[:350])
    print("CONTEXT:", str(row["context"])[:550])
    print("GENERATED ANSWER:", str(row["generated_answer"])[:300])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 211
QUESTION: What British preschool series, directed by Astley Baker Davies, had its music composed by Julian Nott?
CONTEXT: He is best known for his work on the "Wallace and Gromit" films and Peppa Pig animations.Peppa Pig is a British preschool animated television series directed and produced by Astley Baker Davies in association with Entertainment One, which originally aired on 31 May 2004.
GENERATED ANSWER: Peppa Pig
STATUS: success

PILOT ID: 212
QUESTION: Which actor starred in Dr. Cabbie and "The Big Bang Theory"?
CONTEXT: Dr. Cabbie is a 2014 Canadian romantic comedy film that stars Kunal Nayyar, Vinay Virmani, and Adrianne Palicki.Kunal Nayyar ( ; Punjabi: born 30 April 1981) is a British-Indian actor known for his role as Rajesh Koothrappali in the CBS sitcom "The Big Bang Theory".
GENERATED ANSWER: Kunal Nayyar
STATUS: success

PILOT ID: 213
QUESTION: The Strictly the BBC television show featuring the Australian Jason Gilkison is based on what professional partne

In [67]:

from pathlib import Path
import pandas as pd

current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    211: (
        "The context connects Julian Nott's work with Peppa Pig animations "
        "and identifies Peppa Pig as the British preschool series directed "
        "and produced by Astley Baker Davies."
    ),
    212: (
        "The context identifies Kunal Nayyar as a star of Dr. Cabbie and "
        "as the actor who played Rajesh Koothrappali in The Big Bang Theory."
    ),
    213: (
        "The context identifies Jason Gilkison as an Australian professional "
        "ballroom dance champion and choreographer."
    )
}

for pilot_id, evidence in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for pilot ID {pilot_id}")

    review2.loc[mask, "answer_verdict"] = "correct_from_context"
    review2.loc[mask, "context_support"] = "supported"
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = "none"

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 211–213.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(211, 213),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 211–213.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
160,211,correct_from_context,supported
161,212,correct_from_context,supported
162,213,correct_from_context,supported


In [68]:

batch = review2.loc[review2["pilot_id"].between(214, 216)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 214
QUESTION: What fictional Marvel Comics superhero team uses the Leapfrog as their main transportation?
CONTEXT: After the Pride was destroyed by the Gibborim, control of the Leapfrog rested with Chase Stein and has since been used as the primary mode of transportation for the Runaways.Chase Stein (also known as Talkback), is a fictional character, a superhero in the Marvel Comics "Runaways".
GENERATED ANSWER: The Runaways
STATUS: success

PILOT ID: 215
QUESTION: Readville, Boston is bordered on the south by the town in what Massachusetts county, the birthplace of George HW Bush?
CONTEXT: Readville is bordered by the Town of Milton to the south and Town of Dedham to the west.Milton is a town in Norfolk County, Massachusetts, United States and an affluent suburb of Boston. Milton is the birthplace of former U.S. President George H. W. Bush and architect Buckminster Fuller.
GENERATED ANSWER: Norfolk County
STATUS: success

PILOT ID: 216
QUESTION: Barscobe Castle used stone t

In [69]:

from pathlib import Path
import pandas as pd

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

annotation_columns = [
    "answer_verdict",
    "context_support",
    "supporting_evidence",
    "review_notes",
    "answer_error_type"
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

assessments = {
    214: (
        "The context says the Leapfrog is used as the primary mode of "
        "transportation for the Runaways."
    ),
    215: (
        "The context identifies Milton as a town in Norfolk County, Massachusetts."
    ),
    216: (
        "The context says Threave Castle is situated on an island in the River Dee."
    )
}

for pilot_id, evidence in assessments.items():
    mask = review2["pilot_id"].eq(pilot_id)

    if mask.sum() != 1:
        raise ValueError(f"Expected exactly one row for pilot ID {pilot_id}")

    review2.loc[mask, "answer_verdict"] = "correct_from_context"
    review2.loc[mask, "context_support"] = "supported"
    review2.loc[mask, "supporting_evidence"] = evidence
    review2.loc[mask, "review_notes"] = (
        "Preliminary assessment based on supplied context."
    )
    review2.loc[mask, "answer_error_type"] = "none"

review2.to_csv(review_path, index=False)

print("Saved assessments for IDs 214–216.")
print("File:", review_path)

display(
    review2.loc[
        review2["pilot_id"].between(214, 216),
        ["pilot_id", "answer_verdict", "context_support"]
    ]
)


Saved assessments for IDs 214–216.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


,pilot_id,answer_verdict,context_support
163,214,correct_from_context,supported
164,215,correct_from_context,supported
165,216,correct_from_context,supported


In [70]:

batch = review2.loc[review2["pilot_id"].between(217, 219)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))



PILOT ID: 217
QUESTION: Which castle that sits besides Loch Ness is associated with Robert Lauder of Quarrelwood?
CONTEXT: Sir Robert de Lawedre (Lauder), Knt., of Quarrelwood (died about 1370) was Justiciar of Scotia, a Scottish soldier of great prominence and Captain of Urquhart Castle.Urquhart Castle (   ; Scottish Gaelic: "Caisteal na Sròine" ) sits beside Loch Ness in the Highlands of Scotland.
GENERATED ANSWER: Urquhart Castle
STATUS: success

PILOT ID: 218
QUESTION: Peter Bakovic played for which National Hockey League team?
CONTEXT: Peter George "Pete" Bakovic (born January 31, 1965 in Thunder Bay, Ontario) is a retired former professional ice hockey left wing who played briefly in the National Hockey League in the late 1980s for the Vancouver Canucks.The Vancouver Canucks are a professional ice hockey team based in Vancouver, :British Columbia.
GENERATED ANSWER: Vancouver Canucks
STATUS: success

PILOT ID: 219
QUESTION: Thoroughbred is an album released in 1976, by which Amer

In [71]:
annotations = {
    217: ("correct_from_context", "supported", "none"),
    218: ("correct_from_context", "supported", "none"),
    219: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "processed").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 217–219.")
print("File:", review_path)

Saved annotations for IDs 217–219.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [72]:
batch = review2.loc[review2["pilot_id"].between(220, 222)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 220
QUESTION: In 2014, Parrish participated in the 19th season of a show that premiered on which day ?
CONTEXT: In 2014, Parrish participated in the 19th season of "Dancing with the Stars", taking third place.Season nineteen of "Dancing with the Stars" premiered on September 15, 2014.
GENERATED ANSWER: September 15, 2014
STATUS: success

PILOT ID: 221
QUESTION: What year did the the German composer whose compositions are in The Individualism of Gil Evans die?
CONTEXT: The Individualism of Gil Evans is an album by pianist, conductor, arranger and composer Gil Evans originally released on the Verve label in 1964. It features Evans' big band arrangements of five original compositions (two cowritten with Miles Davis) and compositions by Kurt Weill, Bob Dorough, John Lewis and Willie Dixon.Kurt Julian Weill (March 2, 1900April 3, 1950) was a German composer, active from the 1920s in his native country, and in his later years in the United States.
GENERATED ANSWER: 1950
STATUS: su

In [73]:
annotations = {
    220: ("correct_from_context", "supported", "none"),
    221: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "processed").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 220–221.")
print("File:", review_path)

Saved annotations for IDs 220–221.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [74]:
row = review2.loc[review2["pilot_id"] == 222].iloc[0]

print("QUESTION:", row["question"])
print("\nFULL CONTEXT:\n", row["context"])
print("\nGENERATED ANSWER:", row["generated_answer"])

QUESTION: Trey Anastasio and Glenn Bidmead, have which mutual occupation?

FULL CONTEXT:
 Ernest Joseph "Trey" Anastasio III ( , born September 30, 1964) is an American guitarist, composer, and vocalist noted for his work with the band Phish, and his solo career, including the Trey Anastasio Band and the orchestral "Evenings with Trey Anastasio" performed with the New York Philharmonic, the Los Angeles Philharmonic, the Atlanta Symphony Orchestra, the Baltimore Symphony, the Pittsburgh Symphony, and the Colorado Symphony.Glenn Keith Bidmead (born 3 February 1962) is a Sydney-based singer, songwriter, guitarist, performer and producer.

GENERATED ANSWER: Guitarist


In [75]:
annotations = {
    222: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "processed").is_dir()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
review2.to_csv(review_path, index=False)

print("Saved annotation for ID 222.")
print("File:", review_path)

Saved annotation for ID 222.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [76]:
batch = review2.loc[review2["pilot_id"].between(223, 225)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 223
QUESTION: Ammonium perchlorate was used in the disaster that occurred in what city?
CONTEXT: Its instability has involved it in a number of accidents, such as the PEPCON disaster.The PEPCON disaster was an industrial disaster that occurred in Henderson, Nevada on May 4, 1988 at the Pacific Engineering and Production Company of Nevada (PEPCON) plant.
GENERATED ANSWER: Henderson
STATUS: success

PILOT ID: 224
QUESTION: which American football coach led USC Trojans football team
CONTEXT: The Trojans were led by head coach Lane Kiffin, who was in his 1st season.Lane Monte Kiffin (born May 9, 1975) is an American football coach who is currently the head football coach at Florida Atlantic University.
GENERATED ANSWER: Lane Kiffin
STATUS: success

PILOT ID: 225
QUESTION: In Norway, Pinnekjøtt, Stick Meat, is a main course dinner dish of lamb or mutton, this dish is largely associated with the celebration of Christmas, served with puréed swede and potatoes, beer and Akvavit, is 

In [77]:
annotations = {
    223: ("correct_from_context", "supported", "none"),
    224: ("correct_from_context", "supported", "none"),
    225: (
        "needs_external_verification",
        "not_established_by_context",
        "insufficient_context_for_specific_country",
    ),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 223–225.")
print("File:", review_path)

Saved annotations for IDs 223–225.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [78]:
batch = review2.loc[review2["pilot_id"].between(226, 228)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 226
QUESTION: Was Paul Day in more bands than the number of teams Marcus Cooper played for?
CONTEXT: Paul Mario Day (born 19 April 1956, Whitechapel, East London) is a British singer who is well-known as the original lead vocalist of Iron Maiden from 1975 to 1976.Marcus Cooper (born February 1, 1990) is an American football cornerback for the Chicago Bears of the National Football League (NFL). He was drafted by the San Francisco 49ers in the seventh round of the 2013 NFL Draft. He played college football for Rutgers University. Cooper has also played for the Kansas City Chiefs and Arizona Card
GENERATED ANSWER: Yes.
STATUS: success

PILOT ID: 227
QUESTION: Justland cattle are a what used in both dairy and beef production, and have a pattern of pigmented spots on an unpigmented background of hair, feathers or scales
CONTEXT: Jutland cattle are a rare Danish breed of cattle used in both dairy and beef production.A piebald or pied animal is one that has a pattern of pigmented 

In [79]:
annotations = {
    226: (
        "needs_external_verification",
        "not_established_by_context",
        "insufficient_comparison_evidence",
    ),
    227: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "conflates_two_context_facts",
    ),
    228: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 226–228.")
print("File:", review_path)

Saved annotations for IDs 226–228.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [80]:
batch = review2.loc[review2["pilot_id"].between(229, 231)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 229
QUESTION: The Piano Sonata in A minor D. 845, by Franz Schubert is a sonata for solo piano, composed in May 1825, the first movement features in which 2016 South Korean period action thriller film directed by Kim Jee-woon and written by Lee Ji-min and Park Jong-dae?
CONTEXT: The first movement features in the 2016 film "The Age of Shadows"The Age of Shadows (; lit.
GENERATED ANSWER: The Age of Shadows
STATUS: success

PILOT ID: 230
QUESTION: Who directed and composed music for the 2006 film that K. J. Singh won a National Film Award for Best Audiography for?
CONTEXT: Kanwarjit Singh Sawhney, a.k.a. K. J. Singh is an Indian audio engineer based in Mumbai who won the 54th National Film Award for Best Audiography and Filmfare Award for Best Sound Design with Shajith Koyeri and Subash Sahu for his work on the 2006 film "Omkara". The director Vishal Bhardwaj himself composed the entire music for the film, including the background score, with lyrics by Gulzar.
GENERATED ANSWER

In [81]:
annotations = {
    229: ("correct_from_context", "supported", "none"),
    230: ("correct_from_context", "supported", "none"),
    231: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 229–231.")
print("File:", review_path)

Saved annotations for IDs 229–231.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [82]:
batch = review2.loc[review2["pilot_id"].between(232, 234)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 232
QUESTION: Faygo and Shasta, have which mutual detail characteristics?
CONTEXT: Faygo Beverages, Inc., is a soft drink company headquartered in Detroit, Michigan.Shasta Beverages is an American soft drink manufacturer which markets a value priced soft drink line with a wide variety of soda flavors under the brand name Shasta Soda.
GENERATED ANSWER: Both Faygo and Shasta are soft drink manufacturers.
STATUS: success

PILOT ID: 233
QUESTION: Who is Victor Li Tzar-kuoi brother and a Hong Kong philanthropist?
CONTEXT: He is the elder son of tycoon Li Ka-shing and the brother of Richard Li.Richard Li Tzar Kai is a Hong Kong businessman and philanthropist.
GENERATED ANSWER: Richard Li Tzar Kai
STATUS: success

PILOT ID: 234
QUESTION: South Somercotes is a village and civil parish 8 mi north-east from Louth and approximately 2 mi south from which coastal village in the East Lindsey district, and the Marshes area, of Lincolnshire, England?
CONTEXT: South Somercotes is a village a

In [83]:
annotations = {
    232: ("correct_from_context", "supported", "none"),
    233: ("correct_from_context", "supported", "none"),
    234: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 232–234.")
print("File:", review_path)

Saved annotations for IDs 232–234.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [84]:
batch = review2.loc[review2["pilot_id"].between(235, 237)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 235
QUESTION: This American cooking-themed series presentor currently co-hosts a show with who?
CONTEXT: Southern at Heart with Damaris Phillips, commonly known by its shortened title Southern at Heart, is an American cooking-themed series that aired on Food Network. The series was presented by chef Damaris Phillips, who came to prominence as the winner of the ninth season of the Food Network series "Food Network Star".Damaris Lennon Phillips (born December 8, 1980) is an American chef and television personality. She currently co-hosts "The Bobby and Damaris Show" on Food Network with Bobby Flay.
GENERATED ANSWER: Bobby Flay
STATUS: success

PILOT ID: 236
QUESTION: Mathilde Ludendorff was the wife of the German General who was victorious at which two battles?
CONTEXT: Her third husband was General Erich Ludendorff.Erich Friedrich Wilhelm Ludendorff (9 April 1865 – 20 December 1937) was a German general, the victor of the Battle of Liège and the Battle of Tannenberg.
GENERATE

In [85]:
annotations = {
    235: ("correct_from_context", "supported", "none"),
    236: ("correct_from_context", "supported", "none"),
    237: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 235–237.")
print("File:", review_path)

Saved annotations for IDs 235–237.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [86]:
batch = review2.loc[review2["pilot_id"].between(238, 240)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 238
QUESTION: The artist that recorded the song "Blue Love (In My Heart)" had how many singles that reached the Top 10 of the"Billboard" Country & Western Best Sellers chart?
CONTEXT: "Blue Love (In My Heart)" is a song written by Fred Rose (credited under the pseudonym "Floyd Jenkins") and recorded by Hank Williams. Regarded as one of the most significant and influential American singers and songwriters of the 20th century, Williams recorded 35 singles (five released posthumously) that reached the Top 10 of the "Billboard" Country & Western Best Sellers chart, including 11 that ranked number one (three posthumously).
GENERATED ANSWER: 35
STATUS: success

PILOT ID: 239
QUESTION: The Fortune Cookie (alternative UK title: Meet Whiplash Willie) is a 1966 black It was produced and directed by Samuel "Billy" Wilder, an Austrian-American filmmaker, screenwriter, producer, artist, and journalist whose career spanned, more than how many decades?
CONTEXT: It was produced and directed

In [87]:
annotations = {
    238: ("correct_from_context", "supported", "none"),
    239: ("correct_from_context", "supported", "none"),
    240: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 238–240.")
print("File:", review_path)

Saved annotations for IDs 238–240.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [88]:
batch = review2.loc[review2["pilot_id"].between(241, 243)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 241
QUESTION: In what year was the last novel of the author who wrote Revolutionary Road published?
CONTEXT: Cold Spring Harbor (1986) is a novel by American writer Richard Yates. His first novel, "Revolutionary Road", was a finalist for the 1962 National Book Award.
GENERATED ANSWER: 1986
STATUS: success

PILOT ID: 242
QUESTION: What nationality is the lady who, with Sergei Dudakov coaches Yakau Zenko ?
CONTEXT: Formerly a student of Julia Soldatova at Moskvich, he is now coached by Eteri Tutberidze and Sergei Dudakov at Sambo 70.Eteri Georgievna Tutberidze (Russian: Этери Георгиевна Тутберидзе ; born 24 February 1974) is a Russian figure skating coach.
GENERATED ANSWER: The lady is Russian.
STATUS: success

PILOT ID: 243
QUESTION: Which is dedicated to fishkeeping, Reason or Aquarium Fish International?
CONTEXT: Reason is an American libertarian monthly magazine published by the Reason Foundation."Aquarium Fish International" (AFI) was a North American monthly magazine, pu

In [89]:
annotations = {
    241: (
        "needs_external_verification",
        "not_established_by_context",
        "last_novel_not_established_by_context",
    ),
    242: ("correct_from_context", "supported", "none"),
    243: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 241–243.")
print("File:", review_path)

Saved annotations for IDs 241–243.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [90]:
batch = review2.loc[review2["pilot_id"].between(244, 246)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 244
QUESTION: Dr. Karni Singh Shooting Range was first constructed for the an event where how many records were broken?
CONTEXT: Dr. Karni Singh Shooting Range is a shooting range in New Delhi. It was first constructed for the 1982 Asian Games in New Delhi, and later rebuild altogether for the 2010 Commonwealth Games.The 9th Asian Games were held from November 19, 1982, to December 4, 1982, in Delhi, India. 74 Asian and Asian Games records were broken at the event.
GENERATED ANSWER: 74 records were broken.
STATUS: success

PILOT ID: 245
QUESTION: Norris Dam State Park is a state park in Anderson County and Campbell County, Tennessee, the park is situated along the shores of Norris Lake, an impoundment of the Clinch River created by the completion of Norris Dam in 1936, the Clinch River rises near Tazewell, Virginia, and flows southwest for more than 300 mi through the Great Appalachian Valley, at the end, it joins the Tennessee River in Kingston in which US state?
CONTEXT: T

In [91]:
annotations = {
    244: ("correct_from_context", "supported", "none"),
    245: ("correct_from_context", "supported", "none"),
    246: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 244–246.")
print("File:", review_path)

Saved annotations for IDs 244–246.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [92]:
batch = review2.loc[review2["pilot_id"].between(247, 249)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 247
QUESTION: Who is the Baltic goddess of fate often confused with the goddess of fate in the Lithuanian mythology?
CONTEXT: Dalia is the goddess of fate in the Lithuanian mythology. Dalia is often confused with and hard to distinguish from Laima, another goddess of fate.Laima is a Baltic goddess of fate.
GENERATED ANSWER: Laima
STATUS: success

PILOT ID: 248
QUESTION: How many member growers does the cooperative that Steve Wayne did commercials for have?
CONTEXT: Steve Wayne (1920 – September 5, 2004) was a film and television actor appearing in movies and commercials for products such as Alka-Seltzer, Wheaties and Ocean Spray. It currently has over 700 member growers (in Massachusetts, Wisconsin, New Jersey, Oregon, Washington, Florida, British Columbia and other parts of Canada, as well as Chile).
GENERATED ANSWER: 700
STATUS: success

PILOT ID: 249
QUESTION: Which British boxer eliminated Walid Sedik Mohamed in the 2016 Summer Olympics?
CONTEXT: Walid Sedik Mohamed (bor

In [93]:
annotations = {
    247: ("correct_from_context", "supported", "none"),
    248: ("correct_from_context", "supported", "none"),
    249: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 247–249.")
print("File:", review_path)

Saved annotations for IDs 247–249.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [94]:
batch = review2.loc[review2["pilot_id"].between(250, 252)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 250
QUESTION: Helen Herz Cohen (1912 – March 23, 2006) was the Director of Camp Walden, an all-girls residential summer camp located in Denmark, a town in Oxford County, Maine, in which country?
CONTEXT: Helen Herz Cohen (1912 – March 23, 2006) was the Director of Camp Walden, an all-girls residential summer camp located in Denmark, Maine.Denmark is a town in Oxford County, Maine, United States.
GENERATED ANSWER: United States
STATUS: success

PILOT ID: 251
QUESTION: What was the laboratory, in which Albert Attalla worked, named after ?
CONTEXT: He worked as a scientist for Monsanto Research Corporation, Mound Laboratories, in Miamisburg, Ohio, writing many articles about his research, including "Lithium isotopic analysis by nuclear magnetic resonance spectrometry."Mound Laboratories in Miamisburg, Ohio was an Atomic Energy Commission (later Department of Energy) facility for nuclear weapon research during the Cold War, named after the nearby Miami Indian burial mound.
GENER

In [95]:
annotations = {
    250: ("correct_from_context", "supported", "none"),
    251: ("correct_from_context", "supported", "none"),
    252: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 250–252.")
print("File:", review_path)

Saved annotations for IDs 250–252.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [96]:
batch = review2.loc[review2["pilot_id"].between(253, 255)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 253
QUESTION: Marin Čilić won a Masters 1000 title at the 2016 tournament held in what suburb of Cincinnati?
CONTEXT: To date, Čilić has won 16 ATP singles titles including one grand slam singles title at the 2014 US Open, one ATP Masters 1000 title at the 2016 Western & Southern Open and a record four titles at the PBZ Zagreb Indoors. The tournament is held annually at the Lindner Family Tennis Center in Mason (a northern suburb of Cincinnati), Ohio, United States.
GENERATED ANSWER: Mason
STATUS: success

PILOT ID: 254
QUESTION: What time period were people who inspired the Lyoness company trademark originally from?
CONTEXT: The name ‘Lyoness’ was derived from the Celtic mythological kingdom ‘Lyonesse’.The Celts ( or , see pronunciation of "Celt" for different usages) were an Indo-European people in Iron Age and Medieval Europe who spoke Celtic languages and had cultural similarities, although the relationship between ethnic, linguistic and cultural factors in the Celtic wo

In [97]:
annotations = {
    253: ("correct_from_context", "supported", "none"),
    254: ("correct_from_context", "supported", "none"),
    255: (
        "needs_external_verification",
        "not_established_by_context",
        "season_ending_loss_not_established_by_context",
    ),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 253–255.")
print("File:", review_path)

Saved annotations for IDs 253–255.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [98]:
batch = review2.loc[review2["pilot_id"].between(256, 258)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 256
QUESTION: What sport did Jack Sock and Raffaella Reggi play?
CONTEXT: Jack Sock (born September 24, 1992) is an American professional tennis player who is currently ranked world No. 21 in men's singles by the Association of Tennis Professionals (ATP).Raffaella Reggi (born 27 November 1965; ] ) is a former professional tennis player from Italy.
GENERATED ANSWER: Tennis
STATUS: success

PILOT ID: 257
QUESTION: who stars in the drama Witness to a Prosecution ||?
CONTEXT: The drama stars Bobby Au-yeung as the famous historical forensic medical expert Song Ci ("Sung Chee" in Cantonese romanisation). Au-yeung is also known for his lead roles in several of TVB's most successful television series franchises, including "Armed Reaction", "Witness to a Prosecution", and "Forensic Heroes".
GENERATED ANSWER: Bobby Au-yeung stars in the drama Witness to a Prosecution.
STATUS: success

PILOT ID: 258
QUESTION: What Spitsbergen based seed bank serves to protects agricultural diversity in

In [99]:
annotations = {
    256: ("correct_from_context", "supported", "none"),
    257: ("correct_from_context", "supported", "none"),
    258: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 256–258.")
print("File:", review_path)

Saved annotations for IDs 256–258.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [100]:
batch = review2.loc[review2["pilot_id"].between(259, 261)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 259
QUESTION: According to the 2010 census, what was the population of the county in which Loma, Montana is located?
CONTEXT: Loma is a census-designated place (CDP) in Chouteau County, Montana, United States. As of the 2010 census, the population was 5,813.
GENERATED ANSWER: 5,813
STATUS: success

PILOT ID: 260
QUESTION: Who won the election that ran concurrently with the North Dakota gubernatorial election, 2016?
CONTEXT: The 2016 North Dakota gubernatorial election was held on November 8, 2016, to elect the Governor and Lieutenant Governor of North Dakota, concurrently with the 2016 U.S. presidential election, as well as elections to the United States Senate and elections to the United States House of Representatives and various state and local elections. In a surprise victory, the Republican ticket of businessman Donald Trump and Indiana Governor Mike Pence defeated the Democratic ticket of former First Lady, U
GENERATED ANSWER: Donald Trump won the 2016 U.S. presidentia

In [101]:
annotations = {
    259: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "population_entity_mismatch_place_vs_county",
    ),
    260: ("correct_from_context", "supported", "none"),
    261: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 259–261.")
print("File:", review_path)

Saved annotations for IDs 259–261.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [102]:
batch = review2.loc[review2["pilot_id"].between(262, 264)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 262
QUESTION: Following a season in which the Sacramento Kings won three of their first four games before proceeding to lose 12 of their next 13 games, who signed for the Phoenix Suns?
CONTEXT: The 1993–94 NBA season was the Kings' 45th season in the National Basketball Association, and ninth season in Sacramento. With top draft pick Bobby Hurley from Duke University in the 1993 NBA draft, the Kings had a renewed optimism entering the season winning three of their first four games. However, they proceeded to lose 12 of their next 13 games including eight straight losses. Following the season, Wayman Tisdale signed as a free agent with the Phoenix Suns.Wayman Lawrence Tisdale (June 9, 1
GENERATED ANSWER: Wayman Tisdale
STATUS: success

PILOT ID: 263
QUESTION: Parkes High School is located in a town with an urban population of 15,450 as of what date?
CONTEXT: Parkes High School is a secondary school located in Parkes, New South Wales, Australia. Parkes had an urban population 

In [103]:
annotations = {
    262: ("correct_from_context", "supported", "none"),
    263: ("correct_from_context", "supported", "none"),
    264: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 262–264.")
print("File:", review_path)

Saved annotations for IDs 262–264.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [104]:
batch = review2.loc[review2["pilot_id"].between(265, 267)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 265
QUESTION: What film came first, King of the Grizzlies or The Many Adventures of Winnie the Pooh?
CONTEXT: The Many Adventures of Winnie the Pooh is a 1977 American animated film produced by Walt Disney Productions and distributed by Buena Vista Distribution.King of the Grizzlies is a 1970 American adventure film directed by Ron Kelly and written by Jack Speirs, Rod Peterson and Norman Wright.
GENERATED ANSWER: King of the Grizzlies came first.
STATUS: success

PILOT ID: 266
QUESTION: The 2002 Food City 500 was what kind of auto race, featuring oval tracks?
CONTEXT: The 2002 Food City 500 was the sixth stock car race of the 2002 NASCAR Winston Cup Series.Stock car racing is a form of automobile racing found mainly and most prominently in the United States and Canada, with Australia, New Zealand and Brazil also having forms of stock car auto racing. Traditionally, races are run on oval tracks measuring approximately 0.25 to .
GENERATED ANSWER: The 2002 Food City 500 was a 

In [105]:
annotations = {
    265: ("correct_from_context", "supported", "none"),
    266: ("correct_from_context", "supported", "none"),
    267: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 265–267.")
print("File:", review_path)

Saved annotations for IDs 265–267.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [106]:
batch = review2.loc[review2["pilot_id"].between(268, 270)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 268
QUESTION: Richard Yates published a work by the name of "Eleven Kinds of Loneliness", one of Italo Calvino's best known works was of the same genre and named what?
CONTEXT: His first short story collection, "Eleven Kinds of Loneliness", brought comparisons to James Joyce. His best known works include the "Our Ancestors" trilogy (1952–1959), the "Cosmicomics" collection of short stories (1965), and the novels "Invisible Cities" (1972) and "If on a winter's night a traveler" (1979).
GENERATED ANSWER: The question contains a factual error. Richard Yates did not publish a work titled "Eleven Kinds of Loneliness." That work is actually by Italo Calvino. The correct answer to the question—asking for the name of one of Italo Calvino's best known works of the same genre—is "Invisible Cities."
STATUS: success

PILOT ID: 269
QUESTION: Who is older, Mark Aguirre or Adrian Dantley?
CONTEXT: Mark Anthony Aguirre (born December 10, 1959) is an American retired basketball player in the

In [107]:
annotations = {
    268: (
        "incorrect_or_nonresponsive",
        "contradicted",
        "contradicts_context_and_wrong_genre",
    ),
    269: ("correct_from_context", "supported", "none"),
    270: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 268–270.")
print("File:", review_path)

Saved annotations for IDs 268–270.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [108]:
batch = review2.loc[review2["pilot_id"].between(271, 273)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 271
QUESTION: Love Drunk is the second studio album recorded by rock/pop band Boys Like Girls, which song featuring Taylor Swift, from their second studio album "Love Drunk" in 2009?
CONTEXT: Its lead single, titled "Love Drunk" was released on July 7, 2009 shortly followed by "She's Got a Boyfriend Now", "Two Is Better Than One" (featuring Taylor Swift) and "Heart Heart Heartbreak"."Two Is Better Than One" is a song by the American rock band Boys Like Girls from their second studio album "Love Drunk" (2009) and this song features American singer Taylor Swift.
GENERATED ANSWER: "Two Is Better Than One"
STATUS: success

PILOT ID: 272
QUESTION: When was the nephew of George Alexander born?
CONTEXT: Alexander Young (28 December 1938 – 4 August 1997), also known as George Alexander, was a Scottish singer, songwriter, saxophonist, bassist, guitarist and session musician. He is an elder brother of George Young, the rhythm guitarist and founding member of the Easybeats, as well as 

In [109]:
annotations = {
    271: ("correct_from_context", "supported", "none"),
    273: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 271 and 273.")
print("File:", review_path)

Saved annotations for IDs 271 and 273.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [110]:
row = review2.loc[review2["pilot_id"] == 272].iloc[0]

print("QUESTION:", row["question"])
print("\nFULL CONTEXT:\n", row["context"])
print("\nGENERATED ANSWER:", row["generated_answer"])

QUESTION: When was the nephew of George Alexander born?

FULL CONTEXT:
 Alexander Young (28 December 1938 – 4 August 1997), also known as George Alexander, was a Scottish singer, songwriter, saxophonist, bassist, guitarist and session musician. He is an elder brother of George Young, the rhythm guitarist and founding member of the Easybeats, as well as Malcolm and Angus Young, founding members of the Australian hard rock band AC/DC, and the younger brother of Stephen Young, the father of Stevie Young, who was also a member of AC/DC.Stephen Crawford Young (born 11 December 1956 in Glasgow, Scotland) is a Scottish musician, and the rhythm guitarist and backing vocalist for the Australian hard rock band, AC/DC.

GENERATED ANSWER: The context does not provide information about the birth date of the nephew of George Alexander. Therefore, the birth date of the nephew cannot be determined from the given information.


In [111]:
annotations = {
    272: (
        "correct_from_context",
        "not_established_by_context",
        "appropriate_context_limited_abstention",
    ),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotation for ID 272.")
print("File:", review_path)

Saved annotation for ID 272.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [112]:
batch = review2.loc[review2["pilot_id"].between(274, 276)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 274
QUESTION: The Prancing Horse was used by which top Italian fighter ace?
CONTEXT: Originally, the symbol was used by World War I pilot Francesco Baracca on his airplane.Count Francesco Baracca (9 May 1888 – 19 June 1918) was Italy's top fighter ace of World War I.
GENERATED ANSWER: Count Francesco Baracca
STATUS: success

PILOT ID: 275
QUESTION: The Marca Hispanica, also known as the March of Barcelona, was a military buffer zone beyond the former province of Septimania, created by Charlemagne in 795 as a defensive barrier between the Umayyad Moors, it  refers primarily to the Muslim inhabitants of the Maghreb, the Iberian Peninsula, Sicily, and Malta during which time period?
CONTEXT: The Marca Hispanica (Spanish: "Marca Hispánica" , Catalan: "Marca Hispànica" , Aragonese and Occitan: "Marca Hispanica" , Basque: "Hispaniako Marka" , French: "Marche Hispanique" ), literally the Spanish March, also known as the March of Barcelona, was a military buffer zone beyond the form

In [113]:
annotations = {
    274: ("correct_from_context", "supported", "none"),
    276: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 274 and 276.")
print("File:", review_path)

Saved annotations for IDs 274 and 276.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [114]:
row = review2.loc[review2["pilot_id"] == 275].iloc[0]

print("QUESTION:", row["question"])
print("\nFULL CONTEXT:\n", row["context"])
print("\nGENERATED ANSWER:", row["generated_answer"])

QUESTION: The Marca Hispanica, also known as the March of Barcelona, was a military buffer zone beyond the former province of Septimania, created by Charlemagne in 795 as a defensive barrier between the Umayyad Moors, it  refers primarily to the Muslim inhabitants of the Maghreb, the Iberian Peninsula, Sicily, and Malta during which time period?

FULL CONTEXT:
 The Marca Hispanica (Spanish: "Marca Hispánica" , Catalan: "Marca Hispànica" , Aragonese and Occitan: "Marca Hispanica" , Basque: "Hispaniako Marka" , French: "Marche Hispanique" ), literally the Spanish March, also known as the March of Barcelona, was a military buffer zone beyond the former province of Septimania, created by Charlemagne in 795 as a defensive barrier between the Umayyad Moors of Al-Andalus and the Frankish Carolingian Empire (Duchy of Gascony, the Duchy of Aquitaine and Carolingian Septimania).The term "Moors" refers primarily to the Muslim inhabitants of the Maghreb, the Iberian Peninsula, Sicily, and Malta du

In [115]:
annotations = {
    275: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotation for ID 275.")
print("File:", review_path)

Saved annotation for ID 275.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [116]:
batch = review2.loc[review2["pilot_id"].between(277, 279)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 277
QUESTION: What do Lonnie Wilson and Dave Gahan have in common?
CONTEXT: Lonnie Wilson is an American drummer, songwriter, and record producer, known primarily for his work in country music.David "Dave" Gahan ( ; born David Callcott; 9 May 1962) is an English singer-songwriter, best known as the baritone lead singer of the electronic band Depeche Mode since their debut in 1980.
GENERATED ANSWER: Lonnie Wilson and Dave Gahan are both musicians who are known primarily for their work as singers and songwriters.
STATUS: success

PILOT ID: 278
QUESTION: What is the official name of the tower, previously known as Clock Tower, next to which a proposed Thames Airport was to be built in the 1930s?
CONTEXT: It would have also been next to the nearby Big Ben and Palace of Westminster. The tower is officially known as Elizabeth Tower, renamed to celebrate the Diamond Jubilee of Elizabeth II in 2012; previously, it was known simply as the Clock Tower.
GENERATED ANSWER: Elizabeth Tower

In [117]:
annotations = {
    277: (
        "incorrect_or_nonresponsive",
        "not_supported_as_an_answer",
        "unsupported_shared_attribute",
    ),
    278: ("correct_from_context", "supported", "none"),
    279: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 277–279.")
print("File:", review_path)

Saved annotations for IDs 277–279.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [118]:
batch = review2.loc[review2["pilot_id"].between(280, 282)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 280
QUESTION: Bernard Blancan shared the Best Actor award at Cannes in 2006 for what Rachid Bouchareb film?
CONTEXT: He shared the award for Best Actor for his role in "Days of Glory" at the 2006 Cannes Film Festival.Days of Glory (French: "Indigènes - "Natives"" ; Arabic: بلديون‎ ‎ ) is a 2006 French film directed by Rachid Bouchareb.
GENERATED ANSWER: Days of Glory
STATUS: success

PILOT ID: 281
QUESTION: Which city is in the Western area of Ireland which is also the home of Sean Smyth?
CONTEXT: Born in Straide, County Mayo, and now living in Galway, Seán is an All-Ireland champion on both fiddle and whistle .Galway ( ; Irish: "Gaillimh" , ] ) is a city in the West of Ireland in the province of Connacht.
GENERATED ANSWER: Galway
STATUS: success

PILOT ID: 282
QUESTION: Which European breed of dog is smaller, Bedlington Terrier or Transylvanian Hound?
CONTEXT: The Bedlington Terrier is a breed of small dog named after the mining town of Bedlington, Northumberland in North E

In [119]:
annotations = {
    280: ("correct_from_context", "supported", "none"),
    281: ("correct_from_context", "supported", "none"),
    282: (
        "needs_external_verification",
        "not_established_by_context",
        "insufficient_comparison_evidence",
    ),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 280–282.")
print("File:", review_path)

Saved annotations for IDs 280–282.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [120]:
batch = review2.loc[review2["pilot_id"].between(283, 285)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 283
QUESTION: What is the elevation of the stadium where the 2006 Las Vegas Bowl was played?
CONTEXT: This bowl game was played on December 22, 2006 at 5 p.m. PST at 40,000-seat Sam Boyd Stadium in Whitney, Nevada, USA where this bowl game has been played since 1992. The field has a conventional north-south orientation and is at an elevation of 1600 ft above sea level.
GENERATED ANSWER: 1600 ft above sea level
STATUS: success

PILOT ID: 284
QUESTION: One Love is the fourth studio album by French DJ David Guetta, first released in the United Kingdom on 24 August 2009 through Virgin Records, it spawned a total of six worldwide hit singles throughout 2009 and 2010, including which song by French DJ David Guetta from the reissue of his fourth studio album, the song featured which Barbadian recording artist ?
CONTEXT: ", featuring Barbadian recording artist Rihanna and "One Love", featuring British recording artist Estelle. is a song by French DJ David Guetta from the reissue of 

In [121]:
annotations = {
    283: ("correct_from_context", "supported", "none"),
    284: ("correct_from_context", "supported", "none"),
    285: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 283–285.")
print("File:", review_path)

Saved annotations for IDs 283–285.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [122]:
batch = review2.loc[review2["pilot_id"].between(286, 288)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 286
QUESTION: According to the 2010 census, what was the population of the city in which Hosty Duo is based?
CONTEXT: The song "Oklahoma Breakdown" was originally recorded by the Norman-based group, Hosty Duo. The population was 110,925 at the 2010 census.
GENERATED ANSWER: 110,925
STATUS: success

PILOT ID: 287
QUESTION: The Republican nominee in the 2012 United States Senate election in Michigan represented Michigan's 2nd congressional district between what years?
CONTEXT: The Republican nominee was former Congressman Pete Hoekstra.Pieter "Pete" Hoekstra ( ; born October 30, 1953) is a Dutch-American politician who is a former member of the United States House of Representatives, representing Michigan's 2nd congressional district from 1993 to 2011.
GENERATED ANSWER: 1993 to 2011
STATUS: success

PILOT ID: 288
QUESTION: Shaoyang and Haiyang, are in which country?
CONTEXT: Shaoyang () is a prefecture-level city in southwestern Hunan province, People's Republic of China, bord

In [123]:
annotations = {
    286: ("correct_from_context", "supported", "none"),
    287: ("correct_from_context", "supported", "none"),
    288: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 286–288.")
print("File:", review_path)

Saved annotations for IDs 286–288.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [124]:
batch = review2.loc[review2["pilot_id"].between(289, 291)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 289
QUESTION: Angiotensin, a peptide hormone, stimulates the release of the main mineralocorticoid hormone from what part of the adrenal gland?
CONTEXT: Angiotensin is a peptide hormone that causes vasoconstriction and a subsequent increase in blood pressure. Angiotensin also stimulates the release of aldosterone, another hormone, from the adrenal cortex.Aldosterone, the main mineralocorticoid hormone, is a steroid hormone produced by the zona glomerulosa of the adrenal cortex in the adrenal gland.
GENERATED ANSWER: zona glomerulosa
STATUS: success

PILOT ID: 290
QUESTION: Who was a judge on the Sky1 talent show "Got to Dance", and is currently a judge on the British talent show, "Dance Dance Dance"?
CONTEXT: Dance Dance Dance is a British talent show that has aired on ITV since 8 January 2017. It is presented by Alesha Dixon and Will Best and judged by Ashley Banjo, Tina Landon and Timor Steffens. Banjo was a judge on the Sky1 talent show "Got to Dance" and co-presenter of 

In [125]:
annotations = {
    289: ("correct_from_context", "supported", "none"),
    290: ("correct_from_context", "supported", "none"),
    291: (
        "needs_external_verification",
        "not_established_by_context",
        "insufficient_comparison_evidence",
    ),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 289–291.")
print("File:", review_path)

Saved annotations for IDs 289–291.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [126]:
batch = review2.loc[review2["pilot_id"].between(292, 294)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 292
QUESTION: Who has more occupations related to movies, Walter Lang or Scott Derrickson?
CONTEXT: Walter Lang (August 10, 1896 – February 7, 1972) was an American film director.Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.
GENERATED ANSWER: Scott Derrickson has more occupations related to movies than Walter Lang.
STATUS: success

PILOT ID: 293
QUESTION: The December 1992 nor'easter produced record high tides and snowfall across which country's northeastern region, along the Maryland coast, the storm was less severe than the Perfect Storm in the previous year, 1991, also known as The No-Name Storm (especially in the years immediately after it took place) and the Halloween Gale, was a nor'easter that absorbed Hurricane Grace and ultimately evolved back into a small unnamed hurricane late in its life cycle?
CONTEXT: The December 1992 nor'easter produced record high tides and snowfall across the northeastern United States.The 1991 Pe

In [127]:
annotations = {
    292: ("correct_from_context", "supported", "none"),
    293: ("correct_from_context", "supported", "none"),
    294: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 292–294.")
print("File:", review_path)

Saved annotations for IDs 292–294.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [128]:
batch = review2.loc[review2["pilot_id"].between(295, 297)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 295
QUESTION: Which University is based in the town where Anthony Armstrong Jones was born ?
CONTEXT: Ronnie Jones (born June 2, 1949 in Ada, Oklahoma; died June 16, 1996) was an American country music singer known professionally as Anthony Armstrong Jones, a stage name that he took from the name of the British photographer who married Princess Margaret, Countess of Snowdon. Ada is home to East Central University, and is the headquarters of the Chickasaw Nation.
GENERATED ANSWER: East Central University
STATUS: success

PILOT ID: 296
QUESTION: Are Mark Wystrach and Doc Neeson of the same nationality?
CONTEXT: Mark Wystrach (born December 17, 1979) is an American actor and country music singer.Bernard Patrick "Doc" Neeson OAM (4 January 1947 – 4 June 2014) was an Australian singer-songwriter and musician.
GENERATED ANSWER: No, Mark Wystrach and Doc Neeson are not of the same nationality. Mark Wystrach is American, while Doc Neeson is Australian.
STATUS: success

PILOT ID: 297

In [129]:
annotations = {
    295: ("correct_from_context", "supported", "none"),
    296: ("correct_from_context", "supported", "none"),
    297: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotations for IDs 295–297.")
print("File:", review_path)

Saved annotations for IDs 295–297.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [130]:
batch = review2.loc[review2["pilot_id"].between(298, 300)]

for _, row in batch.iterrows():
    print("\n" + "=" * 60)
    print(f"PILOT ID: {row['pilot_id']}")
    print("QUESTION:", row["question"])
    print("CONTEXT:", str(row["context"])[:500])
    print("GENERATED ANSWER:", row["generated_answer"])
    print("STATUS:", row.get("generation_status", "unknown"))


PILOT ID: 298
QUESTION: Roy Black is a civil and criminal defense trial attorney who has represented an American financier and registered sex offender who served how many months in prision for his conviction of soliciting an underage girl for prostitution?
CONTEXT: Roy Black (born February 17, 1945 in New York City) is a civil and criminal defense trial attorney. Other celebrities whom Black has represented include actor Kelsey Grammer, racer Hélio Castroneves, "Girls Gone Wild" creator Joe Francis, artist Peter Max, billionaire Jeffrey Epstein, and Justin Bieber.Jeffrey Edward Epstein (born January 20, 1953) is an American financier and registered sex offender in the United States. In 2008, Epstein was convicted of soliciting an underage girl for prostitu
GENERATED ANSWER: 13 months
STATUS: success

PILOT ID: 299
QUESTION: What is the nickname of man whom the Toronto Raptors acquired from the Sacramento Kings in season 2000-01 ?
CONTEXT: However, Williamson was later on traded to the

In [131]:
annotations = {
    300: ("correct_from_context", "supported", "none"),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved annotation for ID 300.")
print("File:", review_path)

Saved annotation for ID 300.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [132]:
for pilot_id in [298, 299]:
    row = review2.loc[review2["pilot_id"] == pilot_id].iloc[0]

    print("\n" + "=" * 60)
    print(f"PILOT ID: {pilot_id}")
    print("QUESTION:", row["question"])
    print("\nFULL CONTEXT:\n", row["context"])
    print("\nGENERATED ANSWER:", row["generated_answer"])


PILOT ID: 298
QUESTION: Roy Black is a civil and criminal defense trial attorney who has represented an American financier and registered sex offender who served how many months in prision for his conviction of soliciting an underage girl for prostitution?

FULL CONTEXT:
 Roy Black (born February 17, 1945 in New York City) is a civil and criminal defense trial attorney. Other celebrities whom Black has represented include actor Kelsey Grammer, racer Hélio Castroneves, "Girls Gone Wild" creator Joe Francis, artist Peter Max, billionaire Jeffrey Epstein, and Justin Bieber.Jeffrey Edward Epstein (born January 20, 1953) is an American financier and registered sex offender in the United States. In 2008, Epstein was convicted of soliciting an underage girl for prostitution, for which he served 13 months in prison.

GENERATED ANSWER: 13 months

PILOT ID: 299
QUESTION: What is the nickname of man whom the Toronto Raptors acquired from the Sacramento Kings in season 2000-01 ?

FULL CONTEXT:
 H

In [133]:
annotations = {
    298: ("correct_from_context", "supported", "none"),
    299: (
        "needs_external_verification",
        "not_established_by_context",
        "insufficient_entity_linkage_in_context",
    ),
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    if col not in review2.columns:
        review2[col] = ""
    else:
        review2[col] = review2[col].fillna("").astype(object)

for pilot_id, (verdict, support, error_type) in annotations.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = verdict
    review2.loc[mask, "context_support"] = support
    review2.loc[mask, "answer_error_type"] = error_type

current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
review2.to_csv(review_path, index=False)

print("Saved final annotations for IDs 298–299.")
print("File:", review_path)

Saved final annotations for IDs 298–299.
File: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv


In [134]:
import pandas as pd
from pathlib import Path

# Reload the saved file to verify what's actually on disk.
review2 = pd.read_csv(review_path)

print("TOTAL ROWS:", len(review2))
print("UNIQUE PILOT IDs:", review2["pilot_id"].nunique())
print(
    "EXPECTED ID RANGE PRESENT:",
    set(range(51, 301)).issubset(set(review2["pilot_id"].astype(int)))
)

print("\nDUPLICATE PILOT IDs:")
print(
    review2.loc[
        review2["pilot_id"].duplicated(keep=False),
        ["pilot_id", "question"]
    ].sort_values("pilot_id")
)

label_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

print("\nMISSING ANNOTATIONS:")
for col in label_columns:
    missing = review2[col].isna() | review2[col].astype(str).str.strip().eq("")
    print(f"{col}: {int(missing.sum())}")

print("\nANSWER VERDICT COUNTS:")
print(review2["answer_verdict"].value_counts(dropna=False))

print("\nCONTEXT SUPPORT COUNTS:")
print(review2["context_support"].value_counts(dropna=False))

print("\nERROR TYPE COUNTS:")
print(review2["answer_error_type"].value_counts(dropna=False))

print("\nGENERATION STATUS COUNTS:")
print(review2["generation_status"].value_counts(dropna=False))

TOTAL ROWS: 250
UNIQUE PILOT IDs: 250
EXPECTED ID RANGE PRESENT: True

DUPLICATE PILOT IDs:
Empty DataFrame
Columns: [pilot_id, question]
Index: []

MISSING ANNOTATIONS:
answer_verdict: 0
context_support: 0
answer_error_type: 0

ANSWER VERDICT COUNTS:
answer_verdict
correct_from_context           209
needs_external_verification     24
incorrect_or_nonresponsive      17
Name: count, dtype: int64

CONTEXT SUPPORT COUNTS:
context_support
supported                     209
not_established_by_context     20
contradicted                    7
ambiguous                       7
not_supported_as_an_answer      7
Name: count, dtype: int64

ERROR TYPE COUNTS:
answer_error_type
none                                             205
insufficient_comparison_evidence                   8
contradicted_by_supplied_context                   3
appropriate_context_limited_abstention             3
nonresponsive_or_answer_mismatch                   2
ambiguous_question                                 2
ambiguous

In [135]:
# Select questions that still need independent verification.
pending = review2.loc[
    review2["answer_verdict"].eq("needs_external_verification"),
    [
        "pilot_id",
        "question",
        "context",
        "generated_answer",
        "answer_verdict",
        "context_support",
        "answer_error_type",
    ],
].copy()

print("Questions requiring external verification:", len(pending))

display(
    pending[
        [
            "pilot_id",
            "question",
            "generated_answer",
            "answer_error_type",
        ]
    ].reset_index(drop=True)
)

# Save a separate worklist without changing the original review file.
pending_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_needs_external_verification.csv"
)

pending.to_csv(pending_path, index=False)

print("\nSaved verification worklist to:")
print(pending_path)

Questions requiring external verification: 24


,pilot_id,question,generated_answer,answer_error_type
0,71,"Which singer formed a band first, Chris Martin...",Chris Martin formed a band first.,insufficient_comparison_evidence
1,79,"Who released their debut album first, The Oper...",The Operation M.D. released their debut album ...,insufficient_comparison_evidence
2,95,Was We Are Scientists or Needtobreathe formed ...,Needtobreathe was formed further west.,insufficient_comparison_evidence
3,104,Which singer was the leader of an alternative ...,Bob Mould,insufficient_comparison_evidence
4,107,Where have Pradhan Mantri Jan-Dhan Yojana and ...,Pradhan Mantri Jan-Dhan Yojana has been served...,ambiguous_question
5,108,What guitarist worked with both Earl Slick and...,There is no guitarist mentioned in the context...,ambiguous_question_or_context
6,115,The 2017 Pakistan Super League spot-fixing sca...,Khalid Latif,insufficient_biographical_evidence
7,125,What film with actor Barry Keoghan was selecte...,Trespass Against Us,ambiguous_source_reference
8,131,In what city was the Bill Cosby album For Adul...,The context is incorrect. The Westgate Las Veg...,source_location_ambiguity
9,133,The Abbey of Saint-Étienne was founded by a de...,Rollo,truncated_context



Saved verification worklist to:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_needs_external_verification.csv


In [136]:
# Answers confirmed correct through external verification.
verified_correct = [104, 125, 142, 161, 162, 226, 241, 282, 299]

# Answers confirmed incorrect or nonresponsive.
verified_incorrect = {
    71: "incorrect_external_comparison",
    95: "incorrect_external_comparison",
    108: "incorrect_abstention_despite_external_evidence",
    175: "incorrect_genre_comparison",
    202: "album_track_entity_mismatch",
    255: "incorrect_season_finale_team",
    291: "incorrect_external_comparison",
}

for col in ["answer_verdict", "answer_error_type"]:
    review2[col] = review2[col].fillna("").astype(object)

for pilot_id in verified_correct:
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = "verified_correct_externally"
    review2.loc[mask, "answer_error_type"] = "none"

for pilot_id, error_type in verified_incorrect.items():
    mask = review2["pilot_id"] == pilot_id
    review2.loc[mask, "answer_verdict"] = "incorrect_or_nonresponsive"
    review2.loc[mask, "answer_error_type"] = error_type

review2.to_csv(review_path, index=False)

# Refresh the outstanding verification worklist.
pending = review2.loc[
    review2["answer_verdict"].eq("needs_external_verification"),
    [
        "pilot_id",
        "question",
        "context",
        "generated_answer",
        "answer_verdict",
        "context_support",
        "answer_error_type",
    ],
].copy()

pending_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_needs_external_verification.csv"
)
pending.to_csv(pending_path, index=False)

print("Updated review saved:", review_path)
print("Outstanding verification cases:", len(pending))
print("Updated worklist saved:", pending_path)
print("\nUpdated verdict counts:")
print(review2["answer_verdict"].value_counts(dropna=False))

Updated review saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv
Outstanding verification cases: 8
Updated worklist saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_needs_external_verification.csv

Updated verdict counts:
answer_verdict
correct_from_context           209
incorrect_or_nonresponsive      24
verified_correct_externally      9
needs_external_verification      8
Name: count, dtype: int64


In [137]:
for pilot_id in [79, 107, 115, 131, 133, 134, 149, 225]:
    row = review2.loc[review2["pilot_id"] == pilot_id].iloc[0]

    print("\n" + "=" * 70)
    print(f"PILOT ID: {pilot_id}")
    print("\nQUESTION:\n", row["question"])
    print("\nFULL CONTEXT:\n", row["context"])
    print("\nGENERATED ANSWER:\n", row["generated_answer"])
    print("\nCURRENT VERDICT:", row["answer_verdict"])
    print("ERROR TYPE:", row["answer_error_type"])


PILOT ID: 79

QUESTION:
 Who released their debut album first, The Operation M.D. or Framing Hanley ?

FULL CONTEXT:
 The Operation M.D. (formerly The Operation) is a Canadian garage rock band created by Dr. Dynamite (Cone of Sum 41) and Dr. Rocco (Todd Morse of HO) as a side project to their other bands. The band's debut album "We Have an Emergency" was released in February 2007 by Aquarius Records. They released their first studio album in August 2007, titled "The Moment".

GENERATED ANSWER:
 The Operation M.D. released their debut album first.

CURRENT VERDICT: needs_external_verification
ERROR TYPE: insufficient_comparison_evidence

PILOT ID: 107

QUESTION:
 Where have Pradhan Mantri Jan-Dhan Yojana and Narendra Damodardas Modi served in politics?

FULL CONTEXT:
 Pradhan Mantri Jan-Dhan Yojana(P.M.J.D.Y), Prime Minister's People Money Scheme is India's National Mission for Financial Inclusion to ensure access to financial services, namely Banking Savings & Deposit Accounts, Remitt

In [138]:
updates = {
    79: {
        "answer_verdict": "verified_correct_externally",
        "context_support": "not_established_by_context",
        "answer_error_type": "none",
    },
    107: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "answer_error_type": "none",
    },
    115: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "answer_error_type": "none",
    },
    131: {
        "answer_verdict": "verified_correct_externally",
        "context_support": "not_established_by_context",
        "answer_error_type": "none",
    },
    133: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "answer_error_type": "none",
    },
    134: {
        "answer_verdict": "correct_from_context",
        "context_support": "supported",
        "answer_error_type": "none",
    },
    149: {
        "answer_verdict": "incorrect_or_nonresponsive",
        "context_support": "not_supported_as_an_answer",
        "answer_error_type": "film_vs_musical_entity_mismatch",
    },
    225: {
        "answer_verdict": "needs_external_verification",
        "context_support": "ambiguous",
        "answer_error_type": "ambiguous_question_multiple_valid_countries",
    },
}

annotation_columns = [
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

for col in annotation_columns:
    review2[col] = review2[col].fillna("").astype(object)

for pilot_id, values in updates.items():
    mask = review2["pilot_id"] == pilot_id

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for pilot ID {pilot_id}; "
            f"found {mask.sum()}."
        )

    for col, value in values.items():
        review2.loc[mask, col] = value

review2.to_csv(review_path, index=False)

# Rebuild the remaining verification worklist.
pending = review2.loc[
    review2["answer_verdict"].eq("needs_external_verification"),
    [
        "pilot_id",
        "question",
        "context",
        "generated_answer",
        "answer_verdict",
        "context_support",
        "answer_error_type",
    ],
].copy()

pending_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_needs_external_verification.csv"
)
pending.to_csv(pending_path, index=False)

print("Review saved:", review_path)
print("Worklist saved:", pending_path)

print("\nVERDICT COUNTS")
print(review2["answer_verdict"].value_counts(dropna=False))

print("\nREMAINING VERIFICATION CASES:", len(pending))
print(pending[["pilot_id", "answer_error_type"]].to_string(index=False))

print("\nMISSING ANNOTATIONS")
for col in annotation_columns:
    missing = (
        review2[col].isna()
        | review2[col].astype(str).str.strip().eq("")
    )
    print(f"{col}: {int(missing.sum())}")

print("\nTOTAL ROWS:", len(review2))
print("UNIQUE PILOT IDs:", review2["pilot_id"].nunique())

Review saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_review.csv
Worklist saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_needs_external_verification.csv

VERDICT COUNTS
answer_verdict
correct_from_context           213
incorrect_or_nonresponsive      25
verified_correct_externally     11
needs_external_verification      1
Name: count, dtype: int64

REMAINING VERIFICATION CASES: 1
 pilot_id                           answer_error_type
      225 ambiguous_question_multiple_valid_countries

MISSING ANNOTATIONS
answer_verdict: 0
context_support: 0
answer_error_type: 0

TOTAL ROWS: 250
UNIQUE PILOT IDs: 250


In [139]:
import pandas as pd
import numpy as np
from pathlib import Path

# Use the review file that was just audited.
review_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_review.csv"
)

review2 = pd.read_csv(review_path)

# Map reviewed outcomes to an answer-failure target:
# 0 = answer accepted as correct
# 1 = incorrect or nonresponsive answer
# NaN = unresolved; exclude from binary supervised evaluation
label_map = {
    "correct_from_context": 0,
    "verified_correct_externally": 0,
    "incorrect_or_nonresponsive": 1,
    "needs_external_verification": np.nan,
}

review2["answer_failure_label"] = (
    review2["answer_verdict"].map(label_map).astype("Int64")
)

# Save separately; do not overwrite the review file.
labeled_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_labeled.csv"
)

review2.to_csv(labeled_path, index=False)

# Report the resolved evaluation results.
resolved = review2["answer_failure_label"].notna()
resolved_labels = review2.loc[resolved, "answer_failure_label"]

print("Labeled file saved:", labeled_path)
print("\nTotal questions:", len(review2))
print("Resolved questions:", int(resolved.sum()))
print("Unresolved questions:", int((~resolved).sum()))

print("\nAnswer-failure labels:")
print(
    review2["answer_failure_label"]
    .value_counts(dropna=False)
    .sort_index()
)

print(f"\nAnswer-failure rate among resolved cases: "
      f"{resolved_labels.mean():.2%}")
print(f"Accepted-answer rate among resolved cases: "
      f"{1 - resolved_labels.mean():.2%}")

print("\nMissing labels among resolved cases:",
      int(review2.loc[resolved, "answer_failure_label"].isna().sum()))

Labeled file saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_labeled.csv

Total questions: 250
Resolved questions: 249
Unresolved questions: 1

Answer-failure labels:
answer_failure_label
0       224
1        25
<NA>      1
Name: count, dtype: Int64

Answer-failure rate among resolved cases: 10.04%
Accepted-answer rate among resolved cases: 89.96%

Missing labels among resolved cases: 0


In [140]:
import pandas as pd
import numpy as np
from pathlib import Path

# Find the project root.
current = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

# Always reload the finalized review file.
review_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)
final_review = pd.read_csv(review_path)

# Create labels from the latest verdicts.
label_map = {
    "correct_from_context": 0,
    "verified_correct_externally": 0,
    "incorrect_or_nonresponsive": 1,
    "needs_external_verification": np.nan,
}

final_review["answer_failure_label"] = (
    final_review["answer_verdict"]
    .map(label_map)
    .astype("Int64")
)

# Save separately; preserve the original review file.
labeled_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_labeled.csv"
)
final_review.to_csv(labeled_path, index=False)

# Validate the final dataset.
assert len(final_review) == 250
assert final_review["pilot_id"].nunique() == 250
assert final_review["answer_failure_label"].notna().sum() == 249

print("Final labeled dataset saved:", labeled_path)
print("\nVerdict counts:")
print(final_review["answer_verdict"].value_counts())

print("\nFailure-label counts:")
print(final_review["answer_failure_label"].value_counts(dropna=False))

resolved = final_review["answer_failure_label"].dropna()

print(f"\nResolved questions: {len(resolved)}")
print(f"Unresolved questions: {final_review['answer_failure_label'].isna().sum()}")
print(f"Answer-failure rate: {resolved.mean():.2%}")
print("Validation passed.")

Final labeled dataset saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_labeled.csv

Verdict counts:
answer_verdict
correct_from_context           213
incorrect_or_nonresponsive      25
verified_correct_externally     11
needs_external_verification      1
Name: count, dtype: int64

Failure-label counts:
answer_failure_label
0       224
1        25
<NA>      1
Name: count, dtype: Int64

Resolved questions: 249
Unresolved questions: 1
Answer-failure rate: 10.04%
Validation passed.


In [141]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# --------------------------------------------------
# 1. Find project root and load final reviewed labels
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

review_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_review.csv"
)

df = pd.read_csv(review_path)

# --------------------------------------------------
# 2. Recreate the target from the final verdicts
# --------------------------------------------------
label_map = {
    "correct_from_context": 0,
    "verified_correct_externally": 0,
    "incorrect_or_nonresponsive": 1,
    "needs_external_verification": np.nan,
}

df["answer_failure_label"] = (
    df["answer_verdict"].map(label_map).astype("Int64")
)

# Exclude the unresolved question from supervised training.
resolved = df.loc[df["answer_failure_label"].notna()].copy()

print("Total reviewed questions:", len(df))
print("Resolved questions:", len(resolved))
print("Unresolved questions excluded:", len(df) - len(resolved))

# --------------------------------------------------
# 3. Basic text processing
# --------------------------------------------------
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been",
    "by", "for", "from", "in", "into", "is", "it", "its",
    "of", "on", "or", "the", "their", "this", "that",
    "to", "was", "were", "what", "which", "who", "whom",
    "with", "where", "when", "how", "did", "does", "do",
    "has", "have", "had", "he", "she", "they", "them",
    "his", "her", "both", "also",
}

def tokenize(text):
    return re.findall(
        r"[a-z0-9]+(?:['’-][a-z0-9]+)?",
        str(text).lower()
    )

def sentence_count(text):
    parts = re.split(r"[.!?]+", str(text))
    return sum(bool(part.strip()) for part in parts)

# --------------------------------------------------
# 4. Extract features from pre-generation inputs only
# --------------------------------------------------
def extract_pregen_features(question, context):
    q = str(question) if pd.notna(question) else ""
    c = str(context) if pd.notna(context) else ""

    q_lower = q.lower()
    c_lower = c.lower()

    q_tokens = tokenize(q)
    c_tokens = tokenize(c)

    q_content = {
        token for token in q_tokens
        if token not in STOPWORDS and not token.isdigit()
    }
    c_token_set = set(c_tokens)

    overlap_count = len(q_content & c_token_set)
    overlap_ratio = (
        overlap_count / len(q_content)
        if q_content else 0.0
    )

    return {
        # Question complexity and length
        "question_char_count": len(q),
        "question_word_count": len(q_tokens),
        "question_sentence_count": sentence_count(q),
        "question_question_mark_count": q.count("?"),

        # Context size
        "context_char_count": len(c),
        "context_word_count": len(c_tokens),
        "context_sentence_count": sentence_count(c),

        # Question-context lexical overlap
        "question_content_token_count": len(q_content),
        "question_context_overlap_count": overlap_count,
        "question_context_overlap_ratio": overlap_ratio,
        "question_context_length_ratio": (
            len(q_tokens) / max(len(c_tokens), 1)
        ),

        # Question form
        "question_starts_who": int(bool(re.match(r"^\s*who\b", q_lower))),
        "question_starts_what": int(bool(re.match(r"^\s*what\b", q_lower))),
        "question_starts_when": int(bool(re.match(r"^\s*when\b", q_lower))),
        "question_starts_where": int(bool(re.match(r"^\s*where\b", q_lower))),
        "question_starts_which": int(bool(re.match(r"^\s*which\b", q_lower))),
        "question_starts_how": int(bool(re.match(r"^\s*how\b", q_lower))),

        # Potentially challenging question types
        "question_has_comparison": int(bool(re.search(
            r"\b(more|less|older|younger|first|later|earlier|"
            r"larger|smaller|same|versus|vs|compared)\b",
            q_lower
        ))),
        "question_has_multiple_options": int(bool(re.search(
            r"\b(both|either|or|versus|vs)\b", q_lower
        ))),
        "question_asks_count": int(bool(re.search(
            r"\b(how many|how much|number of|what percentage)\b",
            q_lower
        ))),
        "question_asks_date_or_year": int(bool(re.search(
            r"\b(what year|which year|what date|when|how old)\b",
            q_lower
        ))),
        "question_asks_location": int(bool(re.search(
            r"\b(where|which city|which country|which state|"
            r"what country|what city)\b",
            q_lower
        ))),
        "question_contains_negation": int(bool(re.search(
            r"\b(not|never|neither|without|no)\b", q_lower
        ))),

        # Information cues available in the supplied context
        "context_year_count": len(re.findall(
            r"\b(?:18\d{2}|19\d{2}|20\d{2})\b", c
        )),
        "context_numeric_token_count": len(re.findall(
            r"\b\d+(?:,\d{3})*(?:\.\d+)?\b", c
        )),
        "context_has_definition_cue": int(bool(re.search(
            r"\b(is a|is an|was a|was an|refers to|known as|"
            r"also called)\b", c_lower
        ))),
        "context_has_relationship_cue": int(bool(re.search(
            r"\b(son|daughter|brother|sister|wife|husband|"
            r"father|mother|born|member|played|directed|"
            r"released|won|founded|located)\b", c_lower
        ))),
    }

# Build feature table without using generated answers.
feature_records = [
    extract_pregen_features(row["question"], row["context"])
    for _, row in resolved.iterrows()
]

features = pd.DataFrame(feature_records)

# Keep pilot_id only for traceability. Do not use it as a predictor.
feature_dataset = pd.concat(
    [
        resolved[["pilot_id"]].reset_index(drop=True),
        features.reset_index(drop=True),
        resolved[["answer_failure_label"]].reset_index(drop=True),
    ],
    axis=1,
)

# --------------------------------------------------
# 5. Save separate feature dataset
# --------------------------------------------------
feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

feature_dataset.to_csv(feature_path, index=False)

# --------------------------------------------------
# 6. Validate and report
# --------------------------------------------------
assert len(feature_dataset) == 249
assert feature_dataset["answer_failure_label"].notna().all()
assert feature_dataset["pilot_id"].nunique() == 249

assert not any(
    col in feature_dataset.columns
    for col in [
        "generated_answer",
        "answer_verdict",
        "answer_error_type",
        "context_support",
    ]
)

print("Feature dataset saved:", feature_path)
print("Dataset shape:", feature_dataset.shape)
print("Predictor count:", len(features.columns))
print("\nTarget distribution:")
print(feature_dataset["answer_failure_label"].value_counts().sort_index())
print("\nMissing values in feature columns:")
print(
    int(feature_dataset.drop(columns=["pilot_id", "answer_failure_label"])
        .isna().sum().sum())
)
print("\nFeature columns:")
print(features.columns.tolist())
print("\nPre-generation feature validation passed.")

Total reviewed questions: 250
Resolved questions: 249
Unresolved questions excluded: 1
Feature dataset saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_pregen_features.csv
Dataset shape: (249, 29)
Predictor count: 27

Target distribution:
answer_failure_label
0    224
1     25
Name: count, dtype: Int64

Missing values in feature columns:
0

Feature columns:
['question_char_count', 'question_word_count', 'question_sentence_count', 'question_question_mark_count', 'context_char_count', 'context_word_count', 'context_sentence_count', 'question_content_token_count', 'question_context_overlap_count', 'question_context_overlap_ratio', 'question_context_length_ratio', 'question_starts_who', 'question_starts_what', 'question_starts_when', 'question_starts_where', 'question_starts_which', 'question_starts_how', 'question_has_comparison', 'question_has_multiple_options', 'question_asks_count', 'question_asks_date_or_year', 'quest

In [142]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

# --------------------------------------------------
# 1. Locate and load the pre-generation feature data
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path)

target_col = "answer_failure_label"
id_col = "pilot_id"

# Keep only rows with known labels.
data = data.dropna(subset=[target_col]).copy()

# Use only numeric pre-generation features.
X = data.drop(columns=[id_col, target_col])
X = X.select_dtypes(include=[np.number])

y = data[target_col].astype(int)

assert len(data) == 249
assert X.shape[1] == 27
assert y.nunique() == 2
assert X.isna().sum().sum() == 0

print("Dataset shape:", data.shape)
print("Number of predictors:", X.shape[1])
print("\nClass distribution:")
print(y.value_counts().sort_index())

# --------------------------------------------------
# 2. Stratified five-fold cross-validation
# --------------------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

models = {
    "Majority-class baseline": DummyClassifier(
        strategy="most_frequent"
    ),
    "Balanced logistic regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )),
    ]),
}

# --------------------------------------------------
# 3. Evaluate out-of-fold predictions
# --------------------------------------------------
results = []

for name, model in models.items():

    probabilities = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba",
    )[:, 1]

    # Fixed threshold for this initial baseline.
    predictions = (probabilities >= 0.5).astype(int)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(
            y, predictions, zero_division=0
        ),
        "recall": recall_score(
            y, predictions, zero_division=0
        ),
        "f1": f1_score(
            y, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(
            y, probabilities
        ),
    }

    results.append(metrics)

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    for metric, value in metrics.items():
        if metric != "model":
            print(f"{metric}: {value:.4f}")

    print("\nConfusion matrix")
    print("Rows = actual [0, 1]; columns = predicted [0, 1]")
    print(confusion_matrix(y, predictions, labels=[0, 1]))

    print("\nClassification report")
    print(classification_report(
        y,
        predictions,
        labels=[0, 1],
        target_names=["Accepted answer (0)", "Answer failure (1)"],
        zero_division=0,
    ))

# --------------------------------------------------
# 4. Save the comparison
# --------------------------------------------------
results_df = pd.DataFrame(results)

results_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_baseline_results.csv"
)

results_df.to_csv(results_path, index=False)

print("\nBaseline comparison:")
display(results_df.round(4))

print("\nResults saved to:", results_path)

Dataset shape: (249, 29)
Number of predictors: 27

Class distribution:
answer_failure_label
0    224
1     25
Name: count, dtype: int64

Majority-class baseline
accuracy: 0.8996
balanced_accuracy: 0.5000
precision: 0.0000
recall: 0.0000
f1: 0.0000
roc_auc: 0.5000
average_precision: 0.1004

Confusion matrix
Rows = actual [0, 1]; columns = predicted [0, 1]
[[224   0]
 [ 25   0]]

Classification report
                     precision    recall  f1-score   support

Accepted answer (0)       0.90      1.00      0.95       224
 Answer failure (1)       0.00      0.00      0.00        25

           accuracy                           0.90       249
          macro avg       0.45      0.50      0.47       249
       weighted avg       0.81      0.90      0.85       249


Balanced logistic regression
accuracy: 0.7149
balanced_accuracy: 0.5928
precision: 0.1618
recall: 0.4400
f1: 0.2366
roc_auc: 0.6100
average_precision: 0.1688

Confusion matrix
Rows = actual [0, 1]; columns = predicted [0, 1]
[[

,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,Majority-class baseline,0.8996,0.5000,0.0000,0.00,0.0000,0.50,0.1004
1,Balanced logistic regression,0.7149,0.5928,0.1618,0.44,0.2366,0.61,0.1688



Results saved to: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_baseline_results.csv


In [143]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
)

# Reload the saved pre-generation feature dataset.
data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
).copy()

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

# Generate out-of-fold probabilities.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )),
])

probabilities = cross_val_predict(
    model,
    X,
    y,
    cv=cv,
    method="predict_proba",
)[:, 1]

# Explore threshold trade-offs.
rows = []

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, predictions, labels=[0, 1]
    ).ravel()

    rows.append({
        "threshold": threshold,
        "predicted_failures": int(predictions.sum()),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn),
        "precision": precision_score(
            y, predictions, zero_division=0
        ),
        "recall": recall_score(
            y, predictions, zero_division=0
        ),
        "f1": f1_score(
            y, predictions, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y, predictions
        ),
    })

threshold_results = pd.DataFrame(rows)

display(
    threshold_results.round(3)
)

threshold_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_threshold_analysis.csv"
)

threshold_results.to_csv(threshold_path, index=False)

print("\nSaved threshold analysis to:", threshold_path)

,threshold,predicted_failures,true_positives,false_positives,false_negatives,true_negatives,precision,recall,f1,balanced_accuracy
0,0.20,150,18,132,7,92,0.120,0.72,0.206,0.565
1,0.25,133,17,116,8,108,0.128,0.68,0.215,0.581
2,0.30,122,16,106,9,118,0.131,0.64,0.218,0.583
3,0.35,105,15,90,10,134,0.143,0.60,0.231,0.599
4,0.40,92,14,78,11,146,0.152,0.56,0.239,0.606
5,0.45,76,12,64,13,160,0.158,0.48,0.238,0.597
6,0.50,68,11,57,14,167,0.162,0.44,0.237,0.593
7,0.55,58,8,50,17,174,0.138,0.32,0.193,0.548
8,0.60,51,6,45,19,179,0.118,0.24,0.158,0.520



Saved threshold analysis to: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_threshold_analysis.csv


In [144]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# --------------------------------------------------
# 1. Load pre-generation features and labels
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
)

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

# --------------------------------------------------
# 2. Define models
# --------------------------------------------------
models = {
    "Majority baseline": DummyClassifier(
        strategy="most_frequent"
    ),

    "Balanced logistic regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )),
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# --------------------------------------------------
# 3. Compare out-of-fold predictions
# --------------------------------------------------
results = []

for name, model in models.items():
    probabilities = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba",
    )[:, 1]

    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "model": name,
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y, predictions
        ),
        "precision": precision_score(
            y, predictions, zero_division=0
        ),
        "recall": recall_score(
            y, predictions, zero_division=0
        ),
        "f1": f1_score(
            y, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(
            y, probabilities
        ),
    })

results_df = pd.DataFrame(results).sort_values(
    "average_precision",
    ascending=False,
)

display(results_df.round(4))

results_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_model_comparison_cv.csv"
)

results_df.to_csv(results_path, index=False)

print("\nSaved model comparison to:")
print(results_path)

,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
3,Extra Trees,0.7671,0.5329,0.1333,0.24,0.1714,0.5432,0.1988
2,Random Forest,0.8876,0.5466,0.3333,0.12,0.1765,0.5350,0.1856
1,Balanced logistic regression,0.7149,0.5928,0.1618,0.44,0.2366,0.6100,0.1688
0,Majority baseline,0.8996,0.5000,0.0000,0.00,0.0000,0.5000,0.1004



Saved model comparison to:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_model_comparison_cv.csv


In [145]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score,
)

# --------------------------------------------------
# 1. Load pre-generation features
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
).copy()

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

# --------------------------------------------------
# 2. Define the models to compare
# --------------------------------------------------
models = {
    "Majority baseline": DummyClassifier(
        strategy="most_frequent"
    ),
    "Balanced Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

# 5 folds x 10 repetitions = 50 evaluation folds.
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42,
)

# --------------------------------------------------
# 3. Collect out-of-fold metrics for every fold
# --------------------------------------------------
fold_results = []

for split_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    repeat_no = split_idx // 5 + 1
    fold_no = split_idx % 5 + 1

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for name, base_model in models.items():
        model = clone(base_model)
        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]
        predictions = (probabilities >= 0.5).astype(int)

        fold_results.append({
            "model": name,
            "repeat": repeat_no,
            "fold": fold_no,
            "roc_auc": roc_auc_score(y_test, probabilities),
            "average_precision": average_precision_score(
                y_test, probabilities
            ),
            "recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "f1": f1_score(
                y_test, predictions, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
        })

fold_results = pd.DataFrame(fold_results)

# --------------------------------------------------
# 4. Summarize mean and standard deviation
# --------------------------------------------------
metric_columns = [
    "roc_auc",
    "average_precision",
    "recall",
    "precision",
    "f1",
    "balanced_accuracy",
]

summary = (
    fold_results.groupby("model")[metric_columns]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]

summary = summary.reset_index()

# Show the two most important ranking metrics first.
display(
    summary[
        [
            "model",
            "roc_auc_mean",
            "roc_auc_std",
            "average_precision_mean",
            "average_precision_std",
            "recall_mean",
            "recall_std",
            "f1_mean",
            "f1_std",
        ]
    ].round(3)
)

# Save fold-level metrics and summary.
fold_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_repeated_cv_folds.csv"
)

summary_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_repeated_cv_summary.csv"
)

fold_results.to_csv(fold_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nFold-level results saved:", fold_path)
print("Summary saved:", summary_path)
print("\nEvaluation folds per model:",
      fold_results.groupby("model").size().to_dict())

,model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,recall_mean,recall_std,f1_mean,f1_std
0,Balanced Logistic Regression,0.592,0.116,0.207,0.095,0.416,0.232,0.214,0.111
1,Extra Trees,0.509,0.104,0.203,0.098,0.248,0.164,0.167,0.107
2,Majority baseline,0.500,0.000,0.100,0.001,0.000,0.000,0.000,0.000
3,Random Forest,0.493,0.108,0.185,0.084,0.092,0.116,0.115,0.147



Fold-level results saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_repeated_cv_folds.csv
Summary saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_repeated_cv_summary.csv

Evaluation folds per model: {'Balanced Logistic Regression': 50, 'Extra Trees': 50, 'Majority baseline': 50, 'Random Forest': 50}


In [146]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 1. Load the pre-generation feature dataset
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
).reset_index(drop=True)

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

# 2. Define models
models = {
    "Majority baseline": DummyClassifier(
        strategy="most_frequent"
    ),
    "Balanced Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

# 3. Generate the same 10 repetitions of five-fold CV
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42,
)

splits = list(cv.split(X, y))
repeat_results = []

for model_name, base_model in models.items():
    for repeat in range(10):

        # Every observation receives one out-of-fold prediction
        # during this repetition.
        oof_probabilities = np.full(len(y), np.nan)

        for fold in range(5):
            train_idx, test_idx = splits[repeat * 5 + fold]

            model = clone(base_model)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])

            probabilities = model.predict_proba(
                X.iloc[test_idx]
            )[:, 1]

            oof_probabilities[test_idx] = probabilities

        if np.isnan(oof_probabilities).any():
            raise RuntimeError(
                f"Missing predictions: {model_name}, repetition {repeat + 1}"
            )

        predictions = (
            oof_probabilities >= 0.5
        ).astype(int)

        repeat_results.append({
            "model": model_name,
            "repeat": repeat + 1,
            "accuracy": accuracy_score(y, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y, predictions
            ),
            "precision": precision_score(
                y, predictions, zero_division=0
            ),
            "recall": recall_score(
                y, predictions, zero_division=0
            ),
            "f1": f1_score(
                y, predictions, zero_division=0
            ),
            "roc_auc": roc_auc_score(y, oof_probabilities),
            "average_precision": average_precision_score(
                y, oof_probabilities
            ),
        })

repeat_df = pd.DataFrame(repeat_results)

# 4. Summarize variation across the 10 complete repetitions
metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
]

summary = (
    repeat_df.groupby("model")[metrics]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]

summary = summary.reset_index().sort_values(
    "average_precision_mean",
    ascending=False,
)

display(summary.round(3))

# 5. Save both detailed and summary results
repeat_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_repeat_level_results.csv"
)

summary_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_repeat_level_summary.csv"
)

repeat_df.to_csv(repeat_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nDetailed results:", repeat_path)
print("Summary:", summary_path)

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std
0,Balanced Logistic Regression,0.698,0.019,0.573,0.046,0.146,0.028,0.416,0.093,0.216,0.043,0.587,0.043,0.144,0.019
1,Extra Trees,0.759,0.013,0.532,0.027,0.131,0.027,0.248,0.053,0.171,0.035,0.505,0.034,0.141,0.031
3,Random Forest,0.872,0.011,0.525,0.018,0.206,0.081,0.092,0.033,0.126,0.045,0.491,0.024,0.133,0.023
2,Majority baseline,0.900,0.000,0.500,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.500,0.000,0.100,0.000



Detailed results: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_repeat_level_results.csv
Summary: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_repeat_level_summary.csv


In [147]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

# 1. Load the pre-generation dataset
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
).reset_index(drop=True)

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

# 2. Define the feature groups
length_features = [
    "question_char_count",
    "question_word_count",
    "question_sentence_count",
    "question_question_mark_count",
    "context_char_count",
    "context_word_count",
    "context_sentence_count",
]

overlap_features = [
    "question_content_token_count",
    "question_context_overlap_count",
    "question_context_overlap_ratio",
    "question_context_length_ratio",
]

question_form_features = [
    "question_starts_who",
    "question_starts_what",
    "question_starts_when",
    "question_starts_where",
    "question_starts_which",
    "question_starts_how",
    "question_has_comparison",
    "question_has_multiple_options",
    "question_asks_count",
    "question_asks_date_or_year",
    "question_asks_location",
    "question_contains_negation",
]

context_cue_features = [
    "context_year_count",
    "context_numeric_token_count",
    "context_has_definition_cue",
    "context_has_relationship_cue",
]

all_features = list(X.columns)

feature_sets = {
    "All features": all_features,
    "Length only": length_features,
    "Question-context overlap only": overlap_features,
    "Question form only": question_form_features,
    "Context cues only": context_cue_features,
    "All except overlap": [
        f for f in all_features if f not in overlap_features
    ],
    "All except length": [
        f for f in all_features if f not in length_features
    ],
}

# Check that the feature groups match the actual dataset.
for group_name, columns in feature_sets.items():
    missing = sorted(set(columns) - set(X.columns))
    if missing:
        raise ValueError(
            f"{group_name} contains missing columns: {missing}"
        )

# 3. Use identical repeated CV splits for fair comparison
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42,
)

splits = list(cv.split(X, y))
repeat_results = []

for group_name, columns in feature_sets.items():
    X_group = X[columns]

    for repeat in range(10):
        oof_probabilities = np.full(len(y), np.nan)

        for fold in range(5):
            train_idx, test_idx = splits[repeat * 5 + fold]

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )),
            ])

            model.fit(
                X_group.iloc[train_idx],
                y.iloc[train_idx],
            )

            oof_probabilities[test_idx] = (
                model.predict_proba(X_group.iloc[test_idx])[:, 1]
            )

        predictions = (oof_probabilities >= 0.5).astype(int)

        repeat_results.append({
            "feature_group": group_name,
            "repeat": repeat + 1,
            "predictor_count": len(columns),
            "roc_auc": roc_auc_score(y, oof_probabilities),
            "average_precision": average_precision_score(
                y, oof_probabilities
            ),
            "recall": recall_score(
                y, predictions, zero_division=0
            ),
            "f1": f1_score(
                y, predictions, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y, predictions
            ),
        })

repeat_df = pd.DataFrame(repeat_results)

# 4. Summarize performance across the ten repetitions
metrics = [
    "roc_auc",
    "average_precision",
    "recall",
    "f1",
    "balanced_accuracy",
]

summary = (
    repeat_df.groupby(
        ["feature_group", "predictor_count"]
    )[metrics]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]

summary = summary.reset_index().sort_values(
    "average_precision_mean",
    ascending=False,
)

display(summary.round(3))

# 5. Save results
repeat_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_feature_ablation_repeats.csv"
)

summary_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_feature_ablation_summary.csv"
)

repeat_df.to_csv(repeat_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nDetailed results saved:", repeat_path)
print("Summary saved:", summary_path)

,feature_group,predictor_count,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,recall_mean,recall_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
6,Question-context overlap only,4,0.619,0.019,0.213,0.024,0.532,0.019,0.249,0.009,0.613,0.010
0,All except length,20,0.567,0.046,0.154,0.029,0.420,0.069,0.204,0.031,0.560,0.036
2,All features,27,0.587,0.043,0.144,0.019,0.416,0.093,0.216,0.043,0.573,0.046
4,Length only,7,0.544,0.032,0.129,0.016,0.504,0.071,0.195,0.025,0.547,0.035
1,All except overlap,23,0.526,0.041,0.124,0.019,0.392,0.090,0.176,0.037,0.525,0.046
3,Context cues only,4,0.470,0.027,0.114,0.018,0.344,0.066,0.128,0.025,0.448,0.038
5,Question form only,12,0.524,0.056,0.110,0.016,0.408,0.117,0.166,0.031,0.512,0.038



Detailed results saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_feature_ablation_repeats.csv
Summary saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_feature_ablation_summary.csv


In [148]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

# --------------------------------------------------
# 1. Load the pre-generation feature dataset
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
).reset_index(drop=True)

y = data["answer_failure_label"].astype(int)

# Four predictors from the best-performing feature group.
overlap_features = [
    "question_content_token_count",
    "question_context_overlap_count",
    "question_context_overlap_ratio",
    "question_context_length_ratio",
]

missing = set(overlap_features) - set(data.columns)
if missing:
    raise ValueError(f"Missing features: {sorted(missing)}")

feature_sets = {
    "Overlap count": [
        "question_context_overlap_count"
    ],
    "Overlap ratio": [
        "question_context_overlap_ratio"
    ],
    "Question content token count": [
        "question_content_token_count"
    ],
    "Question/context length ratio": [
        "question_context_length_ratio"
    ],
    "Count + overlap ratio": [
        "question_context_overlap_count",
        "question_context_overlap_ratio",
    ],
    "Ratio + content token count": [
        "question_context_overlap_ratio",
        "question_content_token_count",
    ],
    "All four overlap features": overlap_features,
}

# --------------------------------------------------
# 2. Use the same repeated stratified splits
# --------------------------------------------------
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42,
)

splits = list(cv.split(data, y))
records = []

# --------------------------------------------------
# 3. Evaluate each feature group
# --------------------------------------------------
for group_name, columns in feature_sets.items():

    X_group = data[columns]

    for repeat in range(10):
        oof_probabilities = np.full(len(data), np.nan)

        for fold in range(5):
            train_idx, test_idx = splits[repeat * 5 + fold]

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42,
                )),
            ])

            model.fit(
                X_group.iloc[train_idx],
                y.iloc[train_idx],
            )

            oof_probabilities[test_idx] = (
                model.predict_proba(X_group.iloc[test_idx])[:, 1]
            )

        if np.isnan(oof_probabilities).any():
            raise RuntimeError("Missing out-of-fold predictions.")

        predictions = (oof_probabilities >= 0.5).astype(int)

        records.append({
            "feature_group": group_name,
            "repeat": repeat + 1,
            "feature_count": len(columns),
            "roc_auc": roc_auc_score(y, oof_probabilities),
            "average_precision": average_precision_score(
                y, oof_probabilities
            ),
            "recall": recall_score(
                y, predictions, zero_division=0
            ),
            "f1": f1_score(
                y, predictions, zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y, predictions
            ),
        })

results = pd.DataFrame(records)

# --------------------------------------------------
# 4. Summarize variability across repetitions
# --------------------------------------------------
metrics = [
    "roc_auc",
    "average_precision",
    "recall",
    "f1",
    "balanced_accuracy",
]

summary = (
    results.groupby(
        ["feature_group", "feature_count"]
    )[metrics]
    .agg(["mean", "std"])
)

summary.columns = [
    f"{metric}_{stat}"
    for metric, stat in summary.columns
]

summary = summary.reset_index().sort_values(
    "average_precision_mean",
    ascending=False,
)

display(summary.round(3))

# --------------------------------------------------
# 5. Save results
# --------------------------------------------------
repeat_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_overlap_feature_repeats.csv"
)

summary_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_overlap_feature_summary.csv"
)

results.to_csv(repeat_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nRepeated results saved:", repeat_path)
print("Summary saved:", summary_path)

,feature_group,feature_count,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,recall_mean,recall_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
6,Ratio + content token count,2,0.648,0.016,0.216,0.022,0.576,0.028,0.260,0.015,0.628,0.017
0,All four overlap features,4,0.619,0.019,0.213,0.024,0.532,0.019,0.249,0.009,0.613,0.010
1,Count + overlap ratio,2,0.641,0.012,0.212,0.022,0.584,0.028,0.257,0.012,0.627,0.015
3,Overlap ratio,1,0.650,0.013,0.212,0.018,0.576,0.021,0.265,0.008,0.634,0.010
2,Overlap count,1,0.656,0.010,0.197,0.020,0.720,0.000,0.235,0.004,0.614,0.006
4,Question content token count,1,0.580,0.008,0.128,0.009,0.736,0.021,0.206,0.005,0.566,0.009
5,Question/context length ratio,1,0.536,0.015,0.121,0.009,0.732,0.038,0.197,0.010,0.547,0.020



Repeated results saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_overlap_feature_repeats.csv
Summary saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_overlap_feature_summary.csv


In [149]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score,
)

# --------------------------------------------------
# 1. Load resolved data and pre-generation features
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_pregen_features.csv"
)

data = pd.read_csv(feature_path).dropna(
    subset=["answer_failure_label"]
).reset_index(drop=True)

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

# --------------------------------------------------
# 2. Predefine candidate feature groups
# --------------------------------------------------
overlap_features = [
    "question_content_token_count",
    "question_context_overlap_count",
    "question_context_overlap_ratio",
    "question_context_length_ratio",
]

length_features = [
    "question_char_count",
    "question_word_count",
    "question_sentence_count",
    "question_question_mark_count",
    "context_char_count",
    "context_word_count",
    "context_sentence_count",
]

feature_sets = {
    "All 27 features": list(X.columns),
    "All four overlap features": overlap_features,
    "Ratio + content token count": [
        "question_context_overlap_ratio",
        "question_content_token_count",
    ],
    "Overlap ratio only": [
        "question_context_overlap_ratio",
    ],
    "Overlap count only": [
        "question_context_overlap_count",
    ],
    "Length only": length_features,
    "All except overlap": [
        col for col in X.columns
        if col not in overlap_features
    ],
}

for name, columns in feature_sets.items():
    missing = set(columns) - set(X.columns)
    if missing:
        raise ValueError(f"{name}: missing features {missing}")

# --------------------------------------------------
# 3. Nested cross-validation
# --------------------------------------------------
outer_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

outer_splits = list(outer_cv.split(X, y))

C_VALUES = [0.1, 1.0, 10.0]

repeat_results = []
selection_log = []

for split_idx, (outer_train_idx, outer_test_idx) in enumerate(
    outer_splits
):
    repeat_no = split_idx // 5 + 1
    fold_no = split_idx % 5 + 1

    X_train = X.iloc[outer_train_idx].reset_index(drop=True)
    y_train = y.iloc[outer_train_idx].reset_index(drop=True)

    X_test = X.iloc[outer_test_idx]
    y_test = y.iloc[outer_test_idx]

    # Inner CV sees only the outer training partition.
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=1000 + split_idx,
    )

    candidates = []

    for feature_name, columns in feature_sets.items():
        for C in C_VALUES:
            inner_scores = []

            for inner_train_idx, inner_valid_idx in inner_cv.split(
                X_train, y_train
            ):
                model = Pipeline([
                    ("scaler", StandardScaler()),
                    ("classifier", LogisticRegression(
                        C=C,
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=42,
                    )),
                ])

                model.fit(
                    X_train.iloc[inner_train_idx][columns],
                    y_train.iloc[inner_train_idx],
                )

                probabilities = model.predict_proba(
                    X_train.iloc[inner_valid_idx][columns]
                )[:, 1]

                inner_scores.append(
                    average_precision_score(
                        y_train.iloc[inner_valid_idx],
                        probabilities,
                    )
                )

            candidates.append({
                "feature_group": feature_name,
                "features": columns,
                "C": C,
                "inner_average_precision": np.mean(inner_scores),
            })

    # Select using inner validation scores only.
    best = max(
        candidates,
        key=lambda item: item["inner_average_precision"],
    )

    # Refit on all outer-training observations.
    final_inner_choice = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=best["C"],
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )),
    ])

    final_inner_choice.fit(
        X_train[best["features"]],
        y_train,
    )

    # Evaluate once on the untouched outer fold.
    outer_probabilities = final_inner_choice.predict_proba(
        X_test[best["features"]]
    )[:, 1]

    selection_log.append({
        "repeat": repeat_no,
        "outer_fold": fold_no,
        "selected_feature_group": best["feature_group"],
        "selected_feature_count": len(best["features"]),
        "selected_C": best["C"],
        "inner_average_precision": best["inner_average_precision"],
        "outer_test_size": len(outer_test_idx),
        "outer_test_failures": int(y_test.sum()),
    })

    # Cache out-of-fold predictions for this repetition.
    if fold_no == 1:
        repeat_probabilities = np.full(len(y), np.nan)

    repeat_probabilities[outer_test_idx] = outer_probabilities

    if fold_no == 5:
        if np.isnan(repeat_probabilities).any():
            raise RuntimeError(
                f"Missing predictions in repetition {repeat_no}"
            )

        predictions = (repeat_probabilities >= 0.5).astype(int)

        repeat_results.append({
            "repeat": repeat_no,
            "accuracy": np.mean(predictions == y.to_numpy()),
            "balanced_accuracy": balanced_accuracy_score(
                y, predictions
            ),
            "precision": precision_score(
                y, predictions, zero_division=0
            ),
            "recall": recall_score(
                y, predictions, zero_division=0
            ),
            "f1": f1_score(
                y, predictions, zero_division=0
            ),
            "roc_auc": roc_auc_score(
                y, repeat_probabilities
            ),
            "average_precision": average_precision_score(
                y, repeat_probabilities
            ),
        })

# --------------------------------------------------
# 4. Summarize external-fold predictions
# --------------------------------------------------
repeat_results = pd.DataFrame(repeat_results)
selection_log = pd.DataFrame(selection_log)

metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
]

summary_rows = []

for metric in metrics:
    summary_rows.append({
        "metric": metric,
        "mean": repeat_results[metric].mean(),
        "std": repeat_results[metric].std(ddof=1),
    })

summary = pd.DataFrame(summary_rows)

print("NESTED CV SUMMARY")
display(summary.round(3))

print("\nFEATURE GROUP SELECTION FREQUENCY")
display(
    selection_log["selected_feature_group"]
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="selection_count")
)

# --------------------------------------------------
# 5. Save all results
# --------------------------------------------------
repeat_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_nested_cv_repeats.csv"
)

summary_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_nested_cv_summary.csv"
)

selection_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_batch2_nested_cv_feature_selection.csv"
)

repeat_results.to_csv(repeat_path, index=False)
summary.to_csv(summary_path, index=False)
selection_log.to_csv(selection_path, index=False)

print("\nSaved repetition results:", repeat_path)
print("Saved metric summary:", summary_path)
print("Saved feature selection log:", selection_path)

NESTED CV SUMMARY


,metric,mean,std
0,accuracy,0.682,0.015
1,balanced_accuracy,0.621,0.018
2,precision,0.167,0.012
3,recall,0.544,0.022
4,f1,0.256,0.016
5,roc_auc,0.640,0.008
6,average_precision,0.224,0.013



FEATURE GROUP SELECTION FREQUENCY


,feature_group,selection_count
0,All four overlap features,17
1,Ratio + content token count,5
2,Overlap ratio only,2
3,All 27 features,1



Saved repetition results: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_nested_cv_repeats.csv
Saved metric summary: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_nested_cv_summary.csv
Saved feature selection log: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_batch2_nested_cv_feature_selection.csv


In [150]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# --------------------------------------------------
# 1. Find the project root
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

# --------------------------------------------------
# 2. Load the two finalized review files
# --------------------------------------------------
pilot_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_pilot_review.csv"
)

batch2_path = (
    PROJECT_ROOT / "experiments" / "pregeneration_batch2_review.csv"
)

for path in [pilot_path, batch2_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required review file not found: {path}")

pilot = pd.read_csv(pilot_path)
batch2 = pd.read_csv(batch2_path)

reviews = pd.concat(
    [pilot, batch2],
    ignore_index=True,
    sort=False,
)

# Confirm that the IDs cover the expected 300-question collection.
reviews["pilot_id"] = pd.to_numeric(
    reviews["pilot_id"], errors="raise"
).astype(int)

if len(reviews) != 300:
    raise ValueError(f"Expected 300 rows, got {len(reviews)}.")

if reviews["pilot_id"].nunique() != 300:
    raise ValueError("Duplicate pilot IDs found after combining batches.")

expected_ids = set(range(1, 301))
actual_ids = set(reviews["pilot_id"])

if actual_ids != expected_ids:
    raise ValueError(
        f"Unexpected ID coverage. Missing IDs: "
        f"{sorted(expected_ids - actual_ids)}"
    )

# --------------------------------------------------
# 3. Rebuild the answer-failure target from final verdicts
# --------------------------------------------------
label_map = {
    "correct_from_context": 0,
    "verified_correct_externally": 0,
    "incorrect_or_nonresponsive": 1,
    "needs_external_verification": np.nan,
    "source_inconsistent": np.nan,
}

reviews["answer_failure_label"] = (
    reviews["answer_verdict"]
    .map(label_map)
    .astype("Float64")
)

if reviews["answer_failure_label"].isna().sum() != 2:
    print("Current verdict counts:")
    print(reviews["answer_verdict"].value_counts(dropna=False))
    raise ValueError(
        "Expected two unresolved cases across both batches. "
        "Inspect the verdicts before proceeding."
    )

# --------------------------------------------------
# 4. Define the same pre-generation features
# --------------------------------------------------
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been",
    "by", "for", "from", "in", "into", "is", "it", "its",
    "of", "on", "or", "the", "their", "this", "that",
    "to", "was", "were", "what", "which", "who", "whom",
    "with", "where", "when", "how", "did", "does", "do",
    "has", "have", "had", "he", "she", "they", "them",
    "his", "her", "both", "also",
}

def tokenize(text):
    return re.findall(
        r"[a-z0-9]+(?:['’-][a-z0-9]+)?",
        str(text).lower(),
    )

def sentence_count(text):
    parts = re.split(r"[.!?]+", str(text))
    return sum(bool(part.strip()) for part in parts)

def extract_pregen_features(question, context):
    q = str(question) if pd.notna(question) else ""
    c = str(context) if pd.notna(context) else ""

    q_lower = q.lower()
    c_lower = c.lower()

    q_tokens = tokenize(q)
    c_tokens = tokenize(c)

    q_content = {
        token for token in q_tokens
        if token not in STOPWORDS and not token.isdigit()
    }

    overlap_count = len(q_content & set(c_tokens))
    overlap_ratio = (
        overlap_count / len(q_content)
        if q_content else 0.0
    )

    return {
        "question_char_count": len(q),
        "question_word_count": len(q_tokens),
        "question_sentence_count": sentence_count(q),
        "question_question_mark_count": q.count("?"),

        "context_char_count": len(c),
        "context_word_count": len(c_tokens),
        "context_sentence_count": sentence_count(c),

        "question_content_token_count": len(q_content),
        "question_context_overlap_count": overlap_count,
        "question_context_overlap_ratio": overlap_ratio,
        "question_context_length_ratio": (
            len(q_tokens) / max(len(c_tokens), 1)
        ),

        "question_starts_who": int(
            bool(re.match(r"^\s*who\b", q_lower))
        ),
        "question_starts_what": int(
            bool(re.match(r"^\s*what\b", q_lower))
        ),
        "question_starts_when": int(
            bool(re.match(r"^\s*when\b", q_lower))
        ),
        "question_starts_where": int(
            bool(re.match(r"^\s*where\b", q_lower))
        ),
        "question_starts_which": int(
            bool(re.match(r"^\s*which\b", q_lower))
        ),
        "question_starts_how": int(
            bool(re.match(r"^\s*how\b", q_lower))
        ),

        "question_has_comparison": int(bool(re.search(
            r"\b(more|less|older|younger|first|later|earlier|"
            r"larger|smaller|same|versus|vs|compared)\b",
            q_lower,
        ))),

        "question_has_multiple_options": int(bool(re.search(
            r"\b(both|either|or|versus|vs)\b", q_lower
        ))),

        "question_asks_count": int(bool(re.search(
            r"\b(how many|how much|number of|what percentage)\b",
            q_lower,
        ))),

        "question_asks_date_or_year": int(bool(re.search(
            r"\b(what year|which year|what date|when|how old)\b",
            q_lower,
        ))),

        "question_asks_location": int(bool(re.search(
            r"\b(where|which city|which country|which state|"
            r"what country|what city)\b",
            q_lower,
        ))),

        "question_contains_negation": int(bool(re.search(
            r"\b(not|never|neither|without|no)\b", q_lower
        ))),

        "context_year_count": len(re.findall(
            r"\b(?:18\d{2}|19\d{2}|20\d{2})\b", c
        )),

        "context_numeric_token_count": len(re.findall(
            r"\b\d+(?:,\d{3})*(?:\.\d+)?\b", c
        )),

        "context_has_definition_cue": int(bool(re.search(
            r"\b(is a|is an|was a|was an|refers to|known as|"
            r"also called)\b", c_lower
        ))),

        "context_has_relationship_cue": int(bool(re.search(
            r"\b(son|daughter|brother|sister|wife|husband|"
            r"father|mother|born|member|played|directed|"
            r"released|won|founded|located)\b", c_lower
        ))),
    }

# --------------------------------------------------
# 5. Build features for all 300 questions
# --------------------------------------------------
feature_records = [
    extract_pregen_features(row["question"], row["context"])
    for _, row in reviews.iterrows()
]

features = pd.DataFrame(feature_records)

combined_features = pd.concat(
    [
        reviews[["pilot_id"]].reset_index(drop=True),
        features.reset_index(drop=True),
        reviews[["answer_failure_label"]].reset_index(drop=True),
    ],
    axis=1,
)

# --------------------------------------------------
# 6. Validate and save without overwriting source files
# --------------------------------------------------
feature_columns = list(features.columns)

assert len(combined_features) == 300
assert combined_features["pilot_id"].nunique() == 300
assert len(feature_columns) == 27
assert combined_features[feature_columns].isna().sum().sum() == 0

combined_review_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_review_300.csv"
)

combined_feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_pregen_features_300.csv"
)

reviews.to_csv(combined_review_path, index=False)
combined_features.to_csv(combined_feature_path, index=False)

resolved = combined_features.dropna(
    subset=["answer_failure_label"]
)

print("Combined review saved:", combined_review_path)
print("Combined feature dataset saved:", combined_feature_path)
print("\nRows in combined dataset:", len(combined_features))
print("Resolved questions:", len(resolved))
print(
    "Unresolved questions:",
    int(combined_features["answer_failure_label"].isna().sum()),
)

print("\nLabel counts, including unresolved:")
print(
    combined_features["answer_failure_label"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nResolved target counts:")
print(
    resolved["answer_failure_label"]
    .value_counts()
    .sort_index()
)

print("\nFeature dataset shape:", combined_features.shape)
print("Predictor count:", len(feature_columns))
print("Validation passed.")

Combined review saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_review_300.csv
Combined feature dataset saved: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_pregen_features_300.csv

Rows in combined dataset: 300
Resolved questions: 298
Unresolved questions: 2

Label counts, including unresolved:
answer_failure_label
0.0     268
1.0      30
<NA>      2
Name: count, dtype: Int64

Resolved target counts:
answer_failure_label
0.0    268
1.0     30
Name: count, dtype: Int64

Feature dataset shape: (300, 29)
Predictor count: 27
Validation passed.


In [151]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# --------------------------------------------------
# 1. Load the combined feature dataset
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_pregen_features_300.csv"
)

data = pd.read_csv(feature_path)

data = data.dropna(
    subset=["answer_failure_label"]
).reset_index(drop=True)

X = data.drop(
    columns=["pilot_id", "answer_failure_label"]
).select_dtypes(include=[np.number])

y = data["answer_failure_label"].astype(int)

assert len(data) == 298
assert X.shape[1] == 27
assert y.sum() == 30
assert y.value_counts()[0] == 268

print("Resolved questions:", len(data))
print("Predictors:", X.shape[1])
print("Answer failures:", int(y.sum()))
print("Accepted answers:", int((y == 0).sum()))

# --------------------------------------------------
# 2. Predefine candidate feature groups
# --------------------------------------------------
overlap_features = [
    "question_content_token_count",
    "question_context_overlap_count",
    "question_context_overlap_ratio",
    "question_context_length_ratio",
]

length_features = [
    "question_char_count",
    "question_word_count",
    "question_sentence_count",
    "question_question_mark_count",
    "context_char_count",
    "context_word_count",
    "context_sentence_count",
]

feature_sets = {
    "All 27 features": list(X.columns),

    "All four overlap features": overlap_features,

    "Ratio + content token count": [
        "question_context_overlap_ratio",
        "question_content_token_count",
    ],

    "Overlap ratio only": [
        "question_context_overlap_ratio",
    ],

    "Overlap count only": [
        "question_context_overlap_count",
    ],

    "Length only": length_features,

    "All except overlap": [
        col for col in X.columns
        if col not in overlap_features
    ],
}

# --------------------------------------------------
# 3. Configure nested cross-validation
# --------------------------------------------------
outer_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

outer_splits = list(outer_cv.split(X, y))
C_VALUES = [0.1, 1.0, 10.0]

repeat_results = []
selection_log = []

# --------------------------------------------------
# 4. Nested evaluation
# --------------------------------------------------
for split_idx, (train_idx, test_idx) in enumerate(outer_splits):

    repeat_no = split_idx // 5 + 1
    fold_no = split_idx % 5 + 1

    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]

    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=1000 + split_idx,
    )

    candidates = []

    # Select feature group and C using inner folds only.
    for feature_name, columns in feature_sets.items():

        inner_ap_scores = []

        for C in C_VALUES:
            fold_scores = []

            for inner_train, inner_valid in inner_cv.split(
                X_train, y_train
            ):
                model = Pipeline([
                    ("scaler", StandardScaler()),
                    ("classifier", LogisticRegression(
                        C=C,
                        class_weight="balanced",
                        max_iter=2000,
                        random_state=42,
                    )),
                ])

                model.fit(
                    X_train.iloc[inner_train][columns],
                    y_train.iloc[inner_train],
                )

                probabilities = model.predict_proba(
                    X_train.iloc[inner_valid][columns]
                )[:, 1]

                fold_scores.append(
                    average_precision_score(
                        y_train.iloc[inner_valid],
                        probabilities,
                    )
                )

            inner_ap_scores.append(np.mean(fold_scores))

        # Store the best regularization strength for this feature group.
        best_C_idx = int(np.argmax(inner_ap_scores))

        candidates.append({
            "feature_group": feature_name,
            "features": columns,
            "C": C_VALUES[best_C_idx],
            "inner_average_precision": inner_ap_scores[best_C_idx],
        })

    best = max(
        candidates,
        key=lambda item: item["inner_average_precision"],
    )

    # Fit the selected configuration on the outer training data.
    selected_model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=best["C"],
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )),
    ])

    selected_model.fit(
        X_train[best["features"]],
        y_train,
    )

    # Evaluate on the held-out outer fold.
    outer_probabilities = selected_model.predict_proba(
        X_test[best["features"]]
    )[:, 1]

    selection_log.append({
        "repeat": repeat_no,
        "outer_fold": fold_no,
        "selected_feature_group": best["feature_group"],
        "selected_feature_count": len(best["features"]),
        "selected_C": best["C"],
        "inner_average_precision": best["inner_average_precision"],
        "outer_test_size": len(test_idx),
        "outer_test_failures": int(y_test.sum()),
    })

    if fold_no == 1:
        repeat_probabilities = np.full(len(y), np.nan)

    repeat_probabilities[test_idx] = outer_probabilities

    # Every question receives one outer-fold prediction per repetition.
    if fold_no == 5:

        if np.isnan(repeat_probabilities).any():
            raise RuntimeError(
                f"Missing out-of-fold predictions in repetition {repeat_no}"
            )

        predictions = (
            repeat_probabilities >= 0.5
        ).astype(int)

        repeat_results.append({
            "repeat": repeat_no,
            "accuracy": accuracy_score(y, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y, predictions
            ),
            "precision": precision_score(
                y, predictions, zero_division=0
            ),
            "recall": recall_score(
                y, predictions, zero_division=0
            ),
            "f1": f1_score(
                y, predictions, zero_division=0
            ),
            "roc_auc": roc_auc_score(
                y, repeat_probabilities
            ),
            "average_precision": average_precision_score(
                y, repeat_probabilities
            ),
        })

# --------------------------------------------------
# 5. Summarize performance across repetitions
# --------------------------------------------------
repeat_results = pd.DataFrame(repeat_results)
selection_log = pd.DataFrame(selection_log)

metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision",
]

summary = pd.DataFrame([
    {
        "metric": metric,
        "mean": repeat_results[metric].mean(),
        "std": repeat_results[metric].std(ddof=1),
    }
    for metric in metrics
])

print("\nNESTED CV SUMMARY")
display(summary.round(3))

print("\nFEATURE GROUP SELECTION FREQUENCY")
display(
    selection_log["selected_feature_group"]
    .value_counts()
    .rename_axis("feature_group")
    .reset_index(name="selection_count")
)

# --------------------------------------------------
# 6. Save results
# --------------------------------------------------
repeat_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_nested_cv_repeats.csv"
)

summary_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_nested_cv_summary.csv"
)

selection_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_nested_cv_feature_selection.csv"
)

repeat_results.to_csv(repeat_path, index=False)
summary.to_csv(summary_path, index=False)
selection_log.to_csv(selection_path, index=False)

print("\nSaved repetition results:", repeat_path)
print("Saved metric summary:", summary_path)
print("Saved feature selection log:", selection_path)
print("\nNested evaluation completed successfully.")

Resolved questions: 298
Predictors: 27
Answer failures: 30
Accepted answers: 268

NESTED CV SUMMARY


,metric,mean,std
0,accuracy,0.646,0.022
1,balanced_accuracy,0.560,0.034
2,precision,0.132,0.018
3,recall,0.453,0.069
4,f1,0.205,0.028
5,roc_auc,0.579,0.029
6,average_precision,0.163,0.040



FEATURE GROUP SELECTION FREQUENCY


,feature_group,selection_count
0,All four overlap features,14
1,Ratio + content token count,8
2,All except overlap,1
3,Overlap ratio only,1
4,All 27 features,1



Saved repetition results: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_nested_cv_repeats.csv
Saved metric summary: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_nested_cv_summary.csv
Saved feature selection log: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_nested_cv_feature_selection.csv

Nested evaluation completed successfully.


In [152]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Locate the project and load combined files
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

review_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_review_300.csv"
)

feature_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_combined_pregen_features_300.csv"
)

reviews = pd.read_csv(review_path)
features = pd.read_csv(feature_path)

label_map = {
    "correct_from_context": 0,
    "verified_correct_externally": 0,
    "incorrect_or_nonresponsive": 1,
    "needs_external_verification": np.nan,
    "source_inconsistent": np.nan,
}

reviews["answer_failure_label"] = (
    reviews["answer_verdict"].map(label_map)
)

resolved = reviews.dropna(
    subset=["answer_failure_label"]
).copy()

resolved["answer_failure_label"] = (
    resolved["answer_failure_label"].astype(int)
)

# --------------------------------------------------
# 2. Compare the two batches
# --------------------------------------------------
resolved["cohort"] = np.where(
    resolved["pilot_id"] <= 50,
    "Initial pilot (IDs 1–50)",
    "Batch 2 (IDs 51–300)",
)

cohort_summary = (
    resolved.groupby("cohort")
    .agg(
        resolved_questions=("pilot_id", "size"),
        answer_failures=("answer_failure_label", "sum"),
        failure_rate=("answer_failure_label", "mean"),
    )
    .reset_index()
)

unresolved_by_cohort = (
    reviews.assign(
        cohort=np.where(
            reviews["pilot_id"] <= 50,
            "Initial pilot (IDs 1–50)",
            "Batch 2 (IDs 51–300)",
        ),
        unresolved=reviews["answer_verdict"].map(label_map).isna(),
    )
    .groupby("cohort")["unresolved"]
    .sum()
    .rename("unresolved_questions")
    .reset_index()
)

cohort_summary = cohort_summary.merge(
    unresolved_by_cohort,
    on="cohort",
    how="left",
)

print("COHORT COMPARISON")
display(cohort_summary.round(3))

# --------------------------------------------------
# 3. Inspect the 25 answer failures
# --------------------------------------------------
failures = resolved.loc[
    resolved["answer_failure_label"] == 1
].copy()

print("\nANSWER-FAILURE ERROR TYPES")
error_counts = (
    failures["answer_error_type"]
    .fillna("unspecified")
    .value_counts()
    .rename_axis("answer_error_type")
    .reset_index(name="count")
)

display(error_counts)

# Show the individual failures for qualitative inspection.
inspect_columns = [
    "pilot_id",
    "question",
    "generated_answer",
    "answer_verdict",
    "context_support",
    "answer_error_type",
]

inspect_columns = [
    c for c in inspect_columns if c in failures.columns
]

pd.set_option("display.max_colwidth", 120)

print("\nINDIVIDUAL FAILURE EXAMPLES")
display(
    failures[inspect_columns]
    .sort_values("pilot_id")
    .reset_index(drop=True)
)

# --------------------------------------------------
# 4. Compare pre-generation features by outcome
# --------------------------------------------------
target = features[
    ["pilot_id", "answer_failure_label"]
].copy()

target["answer_failure_label"] = pd.to_numeric(
    target["answer_failure_label"],
    errors="coerce",
)

analysis = features.merge(
    target[["pilot_id"]].merge(
        reviews[["pilot_id"]].assign(
            answer_failure_label=reviews["answer_verdict"].map(label_map)
        ),
        on="pilot_id",
        how="left",
    ),
    on="pilot_id",
    how="inner",
)

analysis["answer_failure_label"] = pd.to_numeric(
    analysis["answer_failure_label"],
    errors="coerce",
)

analysis = analysis.dropna(
    subset=["answer_failure_label"]
).copy()

analysis["answer_failure_label"] = (
    analysis["answer_failure_label"].astype(int)
)

feature_columns = [
    c for c in features.select_dtypes(include=[np.number]).columns
    if c not in {"pilot_id", "answer_failure_label"}
]

feature_rows = []

for col in feature_columns:
    accepted = analysis.loc[
        analysis["answer_failure_label"] == 0, col
    ].dropna()

    failed = analysis.loc[
        analysis["answer_failure_label"] == 1, col
    ].dropna()

    if len(accepted) == 0 or len(failed) == 0:
        continue

    pooled_sd = np.sqrt(
        (
            (len(accepted) - 1) * accepted.var(ddof=1)
            + (len(failed) - 1) * failed.var(ddof=1)
        )
        / max(len(accepted) + len(failed) - 2, 1)
    )

    effect_size = (
        (failed.mean() - accepted.mean()) / pooled_sd
        if pooled_sd > 0 else 0.0
    )

    feature_rows.append({
        "feature": col,
        "accepted_mean": accepted.mean(),
        "failure_mean": failed.mean(),
        "accepted_median": accepted.median(),
        "failure_median": failed.median(),
        "standardized_mean_difference": effect_size,
    })

feature_comparison = pd.DataFrame(feature_rows)

feature_comparison["absolute_effect_size"] = (
    feature_comparison["standardized_mean_difference"].abs()
)

feature_comparison = feature_comparison.sort_values(
    "absolute_effect_size",
    ascending=False,
)

print("\nLARGEST PRE-GENERATION FEATURE DIFFERENCES")
display(feature_comparison.head(12).round(3))

# --------------------------------------------------
# 5. Save the audit outputs
# --------------------------------------------------
cohort_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_cohort_audit.csv"
)

errors_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_error_type_audit.csv"
)

feature_audit_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_feature_difference_audit.csv"
)

cohort_summary.to_csv(cohort_path, index=False)
error_counts.to_csv(errors_path, index=False)
feature_comparison.to_csv(feature_audit_path, index=False)

print("\nSaved cohort audit:", cohort_path)
print("Saved error-type audit:", errors_path)
print("Saved feature-difference audit:", feature_audit_path)

COHORT COMPARISON


,cohort,resolved_questions,answer_failures,failure_rate,unresolved_questions
0,Batch 2 (IDs 51–300),249,25,0.100,1
1,Initial pilot (IDs 1–50),49,5,0.102,1



ANSWER-FAILURE ERROR TYPES


,answer_error_type,count
0,nonresponsive_or_answer_mismatch,5
1,contradicted_by_supplied_context,5
2,incorrect_external_comparison,3
3,format_or_answer_mismatch,1
4,incorrect_abstention_despite_external_evidence,1
5,confused_related_people,1
6,film_vs_musical_entity_mismatch,1
7,unsupported_relationship,1
8,incorrect_genre_comparison,1
9,confused_vehicle_year_with_model_introduction,1



INDIVIDUAL FAILURE EXAMPLES


,pilot_id,question,generated_answer,answer_verdict,context_support,answer_error_type
0,10,X.com was an online bank that later became the company that supports what feature?,PayPal,incorrect_or_nonresponsive,not_supported_as_an_answer,nonresponsive_or_answer_mismatch
1,22,What American real estate developer owns the Encore hotel and casino in Las Vegas?,Wynn Resorts,incorrect_or_nonresponsive,not_supported_as_an_answer,nonresponsive_or_answer_mismatch
2,41,Which opera had more acts: Madama Butterfly (original) or Rigoletto?,Both Madama Butterfly (original) and Rigoletto have three acts.,incorrect_or_nonresponsive,contradicted,contradicted_by_supplied_context
3,43,What quarterback for the Atlanta Falcons did Baker Mayfield win the starting job at Oklahoma over?,Baker Mayfield did not win the starting job at Oklahoma over a quarterback for the Atlanta Falcons. The context is i...,incorrect_or_nonresponsive,contradicted,contradicted_by_supplied_context
4,46,What was the full name of the Roman general and statesman for which AD 52 was known as the year of the Consulship of?,Sulla and Otho,incorrect_or_nonresponsive,not_supported_as_an_answer,nonresponsive_or_answer_mismatch
5,60,Three Nationwide Plaza is a 408 ft(124m) postmodern highrise building located at the address 3 Nationwide Plaza in D...,Ohio,incorrect_or_nonresponsive,supported,format_or_answer_mismatch
6,71,"Which singer formed a band first, Chris Martin or Jimmy Gestapo?",Chris Martin formed a band first.,incorrect_or_nonresponsive,not_established_by_context,incorrect_external_comparison
7,84,"The actress that played Rose Buck in ""Upstairs, Downstairs"" also played what chacter in the tv series ""Doctor Who""?","There is no character played by Jean Marsh in ""Doctor Who"" as stated in the context. The context incorrectly states ...",incorrect_or_nonresponsive,contradicted,contradicted_by_supplied_context
8,90,Who did the 47th Governor appoint as a justice of the Supreme Court of Texas?,"The context does not state who the 47th Governor appointed as a justice of the Supreme Court of Texas. Therefore, ba...",incorrect_or_nonresponsive,contradicted,nonresponsive_or_answer_mismatch
9,92,For how many seasons did the series in which Courtney Thorne-Smith played Lyndsey McElroy run ?,3 seasons,incorrect_or_nonresponsive,contradicted,contradicted_by_supplied_context


KeyError: 'answer_failure_label'

In [153]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Reload the combined files safely
# --------------------------------------------------
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "experiments").is_dir()
        and (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

review_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_review_300.csv"
)

feature_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_pregen_features_300.csv"
)

reviews = pd.read_csv(review_path)
features = pd.read_csv(feature_path)

# Recreate target from final reviewed verdicts.
label_map = {
    "correct_from_context": 0,
    "verified_correct_externally": 0,
    "incorrect_or_nonresponsive": 1,
    "needs_external_verification": np.nan,
    "source_inconsistent": np.nan,
}

reviews["answer_failure_label"] = (
    reviews["answer_verdict"].map(label_map)
)

# --------------------------------------------------
# 2. Compare cohorts
# --------------------------------------------------
reviews["cohort"] = np.where(
    reviews["pilot_id"] <= 50,
    "Initial pilot (IDs 1–50)",
    "Batch 2 (IDs 51–300)",
)

cohort_summary = (
    reviews.groupby("cohort")
    .agg(
        total_questions=("pilot_id", "size"),
        resolved_questions=("answer_failure_label", "count"),
        answer_failures=("answer_failure_label", "sum"),
    )
    .reset_index()
)

cohort_summary["failure_rate"] = (
    cohort_summary["answer_failures"]
    / cohort_summary["resolved_questions"]
)

cohort_summary["unresolved_questions"] = (
    cohort_summary["total_questions"]
    - cohort_summary["resolved_questions"]
)

print("COHORT COMPARISON")
display(cohort_summary.round(3))

# --------------------------------------------------
# 3. Categorize answer failures
# --------------------------------------------------
resolved = reviews.dropna(
    subset=["answer_failure_label"]
).copy()

resolved["answer_failure_label"] = (
    resolved["answer_failure_label"].astype(int)
)

failures = resolved.loc[
    resolved["answer_failure_label"] == 1
].copy()

error_counts = (
    failures["answer_error_type"]
    .fillna("unspecified")
    .value_counts()
    .rename_axis("answer_error_type")
    .reset_index(name="count")
)

print("\nANSWER-FAILURE ERROR TYPES")
display(error_counts)

print("\nNUMBER OF RESOLVED ANSWER FAILURES:", len(failures))

# --------------------------------------------------
# 4. Fix: join labels from reviews to pre-generation features
# --------------------------------------------------
# Do not require answer_failure_label to exist inside features.
# Source of truth for the outcome is the reviewed verdict.
target = reviews[
    ["pilot_id", "answer_failure_label"]
].copy()

# Remove any existing target column before joining.
predictor_data = features.drop(
    columns=["answer_failure_label"],
    errors="ignore",
)

analysis = predictor_data.merge(
    target,
    on="pilot_id",
    how="inner",
    validate="one_to_one",
)

analysis["answer_failure_label"] = pd.to_numeric(
    analysis["answer_failure_label"],
    errors="coerce",
)

analysis = analysis.dropna(
    subset=["answer_failure_label"]
).copy()

analysis["answer_failure_label"] = (
    analysis["answer_failure_label"].astype(int)
)

feature_columns = [
    col
    for col in predictor_data.select_dtypes(include=[np.number]).columns
    if col != "pilot_id"
]

assert len(analysis) == 298, (
    f"Expected 298 resolved observations, got {len(analysis)}"
)

assert analysis["answer_failure_label"].sum() == 30

# --------------------------------------------------
# 5. Compare features between accepted and failed answers
# --------------------------------------------------
feature_rows = []

for col in feature_columns:
    accepted = analysis.loc[
        analysis["answer_failure_label"] == 0, col
    ].dropna()

    failed = analysis.loc[
        analysis["answer_failure_label"] == 1, col
    ].dropna()

    if accepted.empty or failed.empty:
        continue

    pooled_variance = (
        (len(accepted) - 1) * accepted.var(ddof=1)
        + (len(failed) - 1) * failed.var(ddof=1)
    ) / max(len(accepted) + len(failed) - 2, 1)

    pooled_sd = np.sqrt(max(pooled_variance, 0))

    standardized_difference = (
        (failed.mean() - accepted.mean()) / pooled_sd
        if pooled_sd > 0 else np.nan
    )

    feature_rows.append({
        "feature": col,
        "accepted_mean": accepted.mean(),
        "failure_mean": failed.mean(),
        "accepted_median": accepted.median(),
        "failure_median": failed.median(),
        "standardized_mean_difference": standardized_difference,
    })

feature_comparison = pd.DataFrame(feature_rows)

feature_comparison["absolute_effect_size"] = (
    feature_comparison["standardized_mean_difference"].abs()
)

feature_comparison = feature_comparison.sort_values(
    "absolute_effect_size",
    ascending=False,
)

print("\nLARGEST PRE-GENERATION FEATURE DIFFERENCES")
display(feature_comparison.head(12).round(3))

# --------------------------------------------------
# 6. Save the audit outputs
# --------------------------------------------------
cohort_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_cohort_audit.csv"
)

errors_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_error_type_audit.csv"
)

feature_audit_path = (
    PROJECT_ROOT / "experiments"
    / "pregeneration_combined_feature_difference_audit.csv"
)

cohort_summary.to_csv(cohort_path, index=False)
error_counts.to_csv(errors_path, index=False)
feature_comparison.to_csv(feature_audit_path, index=False)

print("\nAudit completed successfully.")
print("Cohort audit:", cohort_path)
print("Error-type audit:", errors_path)
print("Feature comparison:", feature_audit_path)
print("\nResolved observations:", len(analysis))
print("Predictors inspected:", len(feature_columns))

COHORT COMPARISON


,cohort,total_questions,resolved_questions,answer_failures,failure_rate,unresolved_questions
0,Batch 2 (IDs 51–300),250,249,25.0,0.100,1
1,Initial pilot (IDs 1–50),50,49,5.0,0.102,1



ANSWER-FAILURE ERROR TYPES


,answer_error_type,count
0,nonresponsive_or_answer_mismatch,5
1,contradicted_by_supplied_context,5
2,incorrect_external_comparison,3
3,format_or_answer_mismatch,1
4,incorrect_abstention_despite_external_evidence,1
5,confused_related_people,1
6,film_vs_musical_entity_mismatch,1
7,unsupported_relationship,1
8,incorrect_genre_comparison,1
9,confused_vehicle_year_with_model_introduction,1



NUMBER OF RESOLVED ANSWER FAILURES: 30

LARGEST PRE-GENERATION FEATURE DIFFERENCES


,feature,accepted_mean,failure_mean,accepted_median,failure_median,standardized_mean_difference,absolute_effect_size
9,question_context_overlap_ratio,0.724,0.600,0.762,0.652,-0.584,0.584
3,question_question_mark_count,0.989,0.933,1.000,1.000,-0.434,0.434
8,question_context_overlap_count,8.022,5.833,7.000,5.000,-0.405,0.405
18,question_has_multiple_options,0.104,0.233,0.000,0.000,0.402,0.402
17,question_has_comparison,0.123,0.233,0.000,0.000,0.324,0.324
12,question_starts_what,0.183,0.300,0.000,0.000,0.296,0.296
7,question_content_token_count,10.869,9.167,9.000,8.000,-0.281,0.281
0,question_char_count,113.082,95.400,91.000,82.000,-0.267,0.267
21,question_asks_location,0.116,0.033,0.000,0.000,-0.266,0.266
1,question_word_count,19.000,16.367,16.000,15.000,-0.247,0.247



Audit completed successfully.
Cohort audit: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_cohort_audit.csv
Error-type audit: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_error_type_audit.csv
Feature comparison: /Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_combined_feature_difference_audit.csv

Resolved observations: 298
Predictors inspected: 27


In [154]:
import pandas as pd
from pathlib import Path

# Find the project root.
current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root.")

source_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "unified_train_all_features.parquet"
)

df_source = pd.read_parquet(source_path)

required_columns = [
    "source_dataset",
    "question_id",
    "question",
    "context",
]

missing_columns = [
    col for col in required_columns
    if col not in df_source.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}"
    )

# Treat whitespace-only strings as empty.
def has_text(series):
    return (
        series.notna()
        & series.astype(str).str.strip().ne("")
    )

df_source["_has_question"] = has_text(df_source["question"])
df_source["_has_context"] = has_text(df_source["context"])

coverage = (
    df_source.groupby("source_dataset", dropna=False)
    .agg(
        rows=("question_id", "size"),
        unique_questions=("question_id", "nunique"),
        usable_questions=("_has_question", "sum"),
        usable_contexts=("_has_context", "sum"),
    )
    .reset_index()
)

coverage["question_and_context_available"] = (
    df_source.groupby("source_dataset", dropna=False)
    .apply(
        lambda g: int(
            (g["_has_question"] & g["_has_context"]).sum()
        ),
        include_groups=False,
    )
    .to_numpy()
)

display(coverage)

coverage_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_source_coverage.csv"
)

coverage.to_csv(coverage_path, index=False)

print("\nSaved source coverage audit:")
print(coverage_path)

print("\nColumns available for source-specific analysis:")
print(df_source.columns.tolist())

,source_dataset,rows,unique_questions,usable_questions,usable_contexts,question_and_context_available
0,fever,9160,9160,9160,0,0
1,halueval,14000,7000,14000,14000,14000
2,truthfulqa,4166,571,4166,0,0



Saved source coverage audit:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_source_coverage.csv

Columns available for source-specific analysis:
['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category', 'split', 'has_context', 'question_length', 'answer_length', 'context_length', 'entity_count', 'person_count', 'organization_count', 'location_count', 'date_count', 'question_answer_similarity', 'question_context_similarity', 'answer_context_similarity', 'retrieval_corpus', 'retrieval_top1_similarity', 'retrieval_top3_mean_similarity', 'retrieval_coverage', 'answer_length_normalized', 'semantic_uncertainty', 'retrieval_uncertainty', 'feature_disagreement', '_has_question', '_has_context']


In [1]:
import pandas as pd
from pathlib import Path

current = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root.")

source_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "unified_train_all_features.parquet"
)

df_source = pd.read_parquet(source_path)

# 1. Print every column without truncation.
print("ALL DATASET COLUMNS:")
for col in df_source.columns:
    print(f"- {col} ({df_source[col].dtype})")

# 2. Find columns potentially containing evidence or raw text.
keywords = [
    "context", "evidence", "claim", "premise",
    "sentence", "passage", "retriev", "support",
    "question", "answer", "source", "label",
]

text_candidate_columns = [
    col for col in df_source.columns
    if any(word in col.lower() for word in keywords)
]

print("\nTEXT / EVIDENCE CANDIDATE COLUMNS:")
print(text_candidate_columns)

# 3. Measure non-empty values in each candidate column by source.
def nonempty_count(series):
    return int(
        (
            series.notna()
            & series.astype(str).str.strip().ne("")
        ).sum()
    )

coverage_rows = []

for source_name, group in df_source.groupby(
    "source_dataset", dropna=False
):
    for col in text_candidate_columns:
        coverage_rows.append({
            "source_dataset": source_name,
            "column": col,
            "dtype": str(group[col].dtype),
            "nonempty_rows": nonempty_count(group[col]),
            "total_rows": len(group),
        })

coverage_detail = pd.DataFrame(coverage_rows)

print("\nCOLUMN COVERAGE BY DATASET:")
display(
    coverage_detail.pivot(
        index="column",
        columns="source_dataset",
        values="nonempty_rows",
    ).fillna(0).astype(int)
)

# 4. Show examples from the sources without context.
for source_name in ["fever", "truthfulqa"]:
    subset = df_source[
        df_source["source_dataset"].astype(str).str.lower()
        == source_name
    ]

    print(f"\n{'=' * 60}")
    print(f"EXAMPLES: {source_name.upper()}")

    available = [
        col for col in text_candidate_columns
        if nonempty_count(subset[col]) > 0
    ]

    display(subset[available].head(3))

# Save the detailed coverage table.
coverage_path = (
    PROJECT_ROOT
    / "experiments"
    / "pregeneration_evidence_column_coverage.csv"
)

coverage_detail.to_csv(coverage_path, index=False)

print("\nSaved evidence-column audit:")
print(coverage_path)

ALL DATASET COLUMNS:
- question_id (str)
- candidate_id (str)
- source_dataset (str)
- question (str)
- answer (str)
- context (str)
- label (int64)
- question_category (str)
- split (str)
- has_context (int64)
- question_length (int64)
- answer_length (int64)
- context_length (int64)
- entity_count (int64)
- person_count (int64)
- organization_count (int64)
- location_count (int64)
- date_count (int64)
- question_answer_similarity (float64)
- question_context_similarity (float64)
- answer_context_similarity (float64)
- retrieval_corpus (str)
- retrieval_top1_similarity (float64)
- retrieval_top3_mean_similarity (float64)
- retrieval_coverage (float64)
- answer_length_normalized (float64)
- semantic_uncertainty (float64)
- retrieval_uncertainty (float64)
- feature_disagreement (float64)

TEXT / EVIDENCE CANDIDATE COLUMNS:
['question_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category', 'has_context', 'question_length', 'answer_length', 'context_length', 

source_dataset,fever,halueval,truthfulqa
column,,,
answer,9160,14000,4166
answer_context_similarity,0,14000,0
answer_length,9160,14000,4166
answer_length_normalized,9160,14000,4166
context,0,14000,0
context_length,9160,14000,4166
has_context,9160,14000,4166
label,9160,14000,4166
question,9160,14000,4166



EXAMPLES: FEVER


,question_id,source_dataset,question,answer,label,question_category,has_context,question_length,answer_length,context_length,retrieval_corpus,retrieval_top1_similarity,retrieval_top3_mean_similarity,retrieval_coverage,answer_length_normalized,retrieval_uncertainty
18166,FEVER_111897,fever,Telemundo is a English-language television net...,Telemundo is a English-language television net...,1,claim_verification,0,6,6,0,halueval_train_knowledge,0.482119,0.448999,0.0,0.857143,0.517881
18167,FEVER_89891,fever,Damon Albarn's debut album was released in 2011.,Damon Albarn's debut album was released in 2011.,1,claim_verification,0,8,8,0,halueval_train_knowledge,0.489568,0.433759,0.0,0.888889,0.510432
18168,FEVER_219028,fever,Savages was exclusively a German film.,Savages was exclusively a German film.,1,claim_verification,0,6,6,0,halueval_train_knowledge,0.401342,0.387986,0.0,0.857143,0.598658



EXAMPLES: TRUTHFULQA


,question_id,source_dataset,question,answer,label,question_category,has_context,question_length,answer_length,context_length,question_answer_similarity,retrieval_corpus,retrieval_top1_similarity,retrieval_top3_mean_similarity,retrieval_coverage,answer_length_normalized,retrieval_uncertainty
0,TQA_0000,truthfulqa,What happens to you if you eat watermelon seeds?,Nothing happens,0,Misconceptions,0,9,2,0,0.183234,halueval_train_knowledge,0.295723,0.289354,0.0,0.2,0.704277
1,TQA_0000,truthfulqa,What happens to you if you eat watermelon seeds?,You eat watermelon seeds,0,Misconceptions,0,9,4,0,0.907568,halueval_train_knowledge,0.295723,0.289354,0.0,0.4,0.704277
2,TQA_0000,truthfulqa,What happens to you if you eat watermelon seeds?,The watermelon seeds pass through your digesti...,0,Misconceptions,0,9,8,0,0.793497,halueval_train_knowledge,0.295723,0.289354,0.0,0.8,0.704277



Saved evidence-column audit:
/Users/sohaafsana/Desktop/Projects/Predictive-model/Predictive_hallucination/experiments/pregeneration_evidence_column_coverage.csv
